In [ ]:
# ============================================================
# MONTAR GOOGLE DRIVE
# ============================================================

from google.colab import drive
import os

# Desmonta si ya estaba montado
try:
    drive.flush_and_unmount()
except:
    pass

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# Verificar
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Google Drive montado correctamente.")
else:
    raise Exception("❌ Error al montar Google Drive.")

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
✅ Google Drive montado correctamente.


## SCRIPT FINAL_ 03072026

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║         BPN PERU Q1 EXTREMO V3.5  —  Pipeline Completo 23 Módulos       ║
# ║  Bajo Peso al Nacer · CNV-MINSA · Google Colab + Google Drive            ║
# ║  Autores: Dr. Evangelista Gamarra et al.                                 ║
# ║  V3.5: Módulo 21 (Learning Curves) NUNCA reentrena en frac=1.0 — el      ║
# ║  punto 100% reutiliza FINAL_PIPES[best_model] ya entrenado en Celda 8    ║
# ║  (elimina el fit() de 4.5M filas que causaba el crash de RAM ahí).       ║
# ║  V3.4: Ablation con muestra 300k + del/gc en Mód. 15-23. V3.3: resume    ║
# ║  Bootstrap CI. V3.2: resume LOYO/Ablation/Learning Curves. V3.1: DeLong  ║
# ║  vectorizado. V3: fix SHAP HGB, resume/checkpoints, FRESH_START.         ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ══════════════════════════════════════════════════════════════════════════════
# CELDA 0 · Instalación de dependencias y rutas
# ══════════════════════════════════════════════════════════════════════════════
import subprocess
import sys
import importlib

PKGS = {
    "polars": "polars>=0.20",
    "pyarrow": "pyarrow>=14",
    "psutil": "psutil>=5.9",
    "xgboost": "xgboost>=2.0",
    "lightgbm": "lightgbm>=4.0",
    "catboost": "catboost>=1.2",
    "optuna": "optuna>=3.5",
    "optuna_integration": "optuna-integration",
    "shap": "shap>=0.44",
    "sklearn": "scikit-learn>=1.4",
    "mlflow": "mlflow>=2.10",
    "matplotlib": "matplotlib>=3.8",
    "seaborn": "seaborn>=0.13",
    "scipy": "scipy>=1.12",
    "joblib": "joblib",
    "imblearn": "imbalanced-learn>=0.12",
    "reportlab": "reportlab>=4.0",
    "jinja2": "jinja2>=3.0",
    "openpyxl": "openpyxl>=3.1",
    "statsmodels": "statsmodels>=0.14",
}

for modulo, paquete in PKGS.items():
    try:
        importlib.import_module(modulo)
    except ModuleNotFoundError:
        print(f"Instalando {paquete}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

print("Dependencias listas.")

# ══════════════════════════════════════════════════════════════════════════════
# RUTAS DEL DATASET (Q1 EXTREME)
# ══════════════════════════════════════════════════════════════════════════════
from pathlib import Path
import polars as pl
import os

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except:
    pass

DATA_DIR = Path("/content/drive/MyDrive/dataset2026varios")
DATASET_NAME = "CNV_MINSA_CORTE_30112025"
CSV_PATH = DATA_DIR / f"{DATASET_NAME}.csv"
PARQUET_PATH = DATA_DIR / f"{DATASET_NAME}.parquet"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"No existe el directorio:\n{DATA_DIR}")

if PARQUET_PATH.exists():
    DATA_PATH = PARQUET_PATH
    DATA_TYPE = "PARQUET"
elif CSV_PATH.exists():
    print("=" * 80)
    print("No existe el Parquet.")
    print("Convirtiendo automáticamente CSV → PARQUET...")
    print("=" * 80)
    (
        pl.scan_csv(
            CSV_PATH,
            separator=";",
            infer_schema_length=10000,
            ignore_errors=True,
            truncate_ragged_lines=True,
            encoding="utf8-lossy"
        )
        .sink_parquet(PARQUET_PATH, compression="zstd")
    )
    DATA_PATH = PARQUET_PATH
    DATA_TYPE = "PARQUET"
else:
    raise FileNotFoundError(
        f"\nNo se encontró el dataset.\n\nBuscado:\n\n{PARQUET_PATH}\n\no\n\n{CSV_PATH}\n"
    )

print("=" * 80)
print("DATASET CONFIGURADO")
print("=" * 80)
print(f"Tipo          : {DATA_TYPE}")
print(f"Ruta          : {DATA_PATH}")
print(f"Existe        : {DATA_PATH.exists()}")
print(f"Tamaño (MB)   : {os.path.getsize(DATA_PATH)/1024**2:.2f}")
print("=" * 80)

lf = pl.scan_parquet(DATA_PATH)
print(f"Columnas      : {len(lf.collect_schema())}")
print("Primeras columnas:")
print(lf.collect_schema().names()[:10])
print("=" * 80)

# ══════════════════════════════════════════════════════════════════════════════
# DIRECTORIOS DE SALIDA — Google Drive (persisten aunque Colab se reinicie)
# ══════════════════════════════════════════════════════════════════════════════
BASE_OUT = Path("/content/drive/MyDrive/dataset2026varios/Archivos_RN_2026")

DIRS = {
    "01_DATASET": BASE_OUT / "01_DATASET",
    "02_EDA": BASE_OUT / "02_EDA",
    "03_PREPROCESSING": BASE_OUT / "03_PREPROCESSING",
    "04_FEATURE_ENGINEERING": BASE_OUT / "04_FEATURE_ENGINEERING",
    "05_FEATURE_SELECTION": BASE_OUT / "05_FEATURE_SELECTION",
    "06_MODELS": BASE_OUT / "06_MODELS",
    "07_NESTED_CV": BASE_OUT / "07_NESTED_CV",
    "08_METRICS": BASE_OUT / "08_METRICS",
    "09_BOOTSTRAP": BASE_OUT / "09_BOOTSTRAP",
    "10_CALIBRATION": BASE_OUT / "10_CALIBRATION",
    "11_SHAP": BASE_OUT / "11_SHAP",
    "12_FAIRNESS": BASE_OUT / "12_FAIRNESS",
    "13_DRIFT": BASE_OUT / "13_DRIFT",
    "14_ERROR_ANALYSIS": BASE_OUT / "14_ERROR_ANALYSIS",
    "15_ABLATION": BASE_OUT / "15_ABLATION",
    "16_EXTERNAL_VALIDATION": BASE_OUT / "16_EXTERNAL_VALIDATION",
    "17_MODEL_CARD": BASE_OUT / "17_MODEL_CARD",
    "18_REPRODUCIBILITY": BASE_OUT / "18_REPRODUCIBILITY",
    "19_PDF_REPORT": BASE_OUT / "19_PDF_REPORT",
    "20_HTML_REPORT": BASE_OUT / "20_HTML_REPORT",
    "21_LEARNING_CURVES": BASE_OUT / "21_LEARNING_CURVES",
    "22_PERMUTATION_IMPORTANCE": BASE_OUT / "22_PERMUTATION_IMPORTANCE",
    "23_PREDICTIONS": BASE_OUT / "23_PREDICTIONS",
    "figures": BASE_OUT / "figures",
    "models": BASE_OUT / "models",
    "metrics": BASE_OUT / "metrics",
    "tables": BASE_OUT / "tables",
    "paper": BASE_OUT / "paper",
    "cache": BASE_OUT / "cache",
    "checkpoints": BASE_OUT / "checkpoints",
    "logs": BASE_OUT / "logs",
    "temp": BASE_OUT / "temp",
}

# ══════════════════════════════════════════════════════════════════════════════
# MODO DE EJECUCIÓN: FRESH_START (borrar todo) vs RESUME (conservar todo)
# ══════════════════════════════════════════════════════════════════════════════
# FRESH_START = True  -> BORRA TODO el contenido anterior de BASE_OUT (dataset
#                procesado, features, nested CV, modelos .pkl, reportes,
#                figuras, checkpoints) y crea las 30 carpetas desde cero.
#                Úsalo solo cuando quieras una corrida 100% limpia (p. ej.
#                cambiaste el dataset de origen o hiciste cambios grandes al
#                código y los checkpoints viejos ya no son compatibles).
# FRESH_START = False -> (default, recomendado) NO borra nada. Crea las
#                carpetas que falten sin tocar las que ya existen, y el
#                sistema RESUME reanuda donde quedó. Es lo que evita perder
#                horas de entrenamiento si Colab se reinicia por falta de RAM.
#
# Cambia manualmente esta línea a True cuando de verdad quieras empezar de
# cero; vuelve a dejarla en False para las corridas normales.
FRESH_START = False

if FRESH_START and BASE_OUT.exists():
    import shutil as _shutil_fresh
    print("=" * 80)
    print("⚠ FRESH_START = True — borrando TODO el contenido anterior de:")
    print(f"  {BASE_OUT}")
    print("  (dataset procesado, features, nested CV, modelos, reportes,")
    print("   figuras, checkpoints — todo)")
    print("=" * 80)
    _shutil_fresh.rmtree(BASE_OUT)
    print("Contenido anterior eliminado. Recreando carpetas desde cero...\n")

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Entorno listo. Directorios creados.")
print(f"BASE_OUT (Google Drive, persistente): {BASE_OUT}")
print(f"Modo: {'FRESH_START (borrado y recreación completa)' if FRESH_START else 'RESUME (conserva resultados previos)'}")


# ══════════════════════════════════════════════════════════════════════════════
# CELDA 1 · Configuración global y utilidades
# ══════════════════════════════════════════════════════════════════════════════
import os, warnings, time, platform, json, hashlib, logging, gc
import numpy as np
import pandas as pd
import polars as pl
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, date
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

FIGSIZE_SINGLE = (7.1, 5.0)
FIGSIZE_DOUBLE = (7.1, 3.5)
DPI_SCREEN  = 150
DPI_PRINT   = 300
FONT_FAMILY = "DejaVu Sans"

matplotlib.rcParams.update({
    "font.family"       : FONT_FAMILY,
    "font.size"         : 9,
    "axes.titlesize"    : 10,
    "axes.labelsize"    : 9,
    "xtick.labelsize"   : 8,
    "ytick.labelsize"   : 8,
    "legend.fontsize"   : 8,
    "figure.dpi"        : DPI_SCREEN,
    "savefig.dpi"       : DPI_PRINT,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

PAL = sns.color_palette("colorblind")
sns.set_theme(style="whitegrid", palette=PAL)

def save_fig(fig, name: str, subdir: str = "figures"):
    path = DIRS[subdir] / name
    fig.savefig(path, dpi=DPI_PRINT, bbox_inches="tight", facecolor="white")
    print(f"  Saved: {path}")
    return path

def save_json(obj, name: str, subdir: str = "metrics"):
    path = DIRS[subdir] / name
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    print(f"  Saved: {path}")
    return path

def save_table_apa(df: pd.DataFrame, name: str, caption: str = "", subdir: str = "tables"):
    path = DIRS[subdir] / name
    with open(path, "w") as f:
        if caption:
            f.write(f"# {caption}\n")
        df.to_csv(f, index=False)
    print(f"  Saved: {path}")
    return path

def save_excel_multi(sheets: dict, name: str, subdir: str = "tables"):
    path = DIRS[subdir] / name
    with pd.ExcelWriter(path, engine="openpyxl") as wr:
        for sheet_name, df in sheets.items():
            df.to_excel(wr, sheet_name=sheet_name[:31], index=False)
    print(f"  Saved: {path}")
    return path

# ══════════════════════════════════════════════════════════════════════════════
# SISTEMA DE CHECKPOINTS Y REANUDACIÓN (RESUME PIPELINE)
# ══════════════════════════════════════════════════════════════════════════════
# Como BASE_OUT ahora vive en Google Drive, TODOS los archivos que el pipeline
# ya generó (CSV, PNG, XLSX, JSON, modelos .pkl) sobreviven a un reinicio de
# Colab por sí solos — no se pierden aunque la sesión muera. Lo que este
# sistema añade es evitar RECALCULAR trabajo costoso ya hecho:
#   1) Carga/filtrado del dataset (4.5M filas vía Polars).
#   2) Feature engineering + split train/test.
#   3) Búsqueda de hiperparámetros por modelo en Nested CV (Celda 7).
#   4) Entrenamiento final por modelo (Celda 8) — si el .pkl ya existe, se
#      carga en vez de reentrenar.
#
# Limitación real de Colab (no del código): si el proceso muere a mitad de un
# solo .fit() (p. ej. LightGBM al 40%), ese entrenamiento puntual no puede
# reanudarse desde adentro — ninguna librería guarda su estado interno de
# forma automática. Lo que sí se garantiza es que NUNCA se repite un modelo
# que ya terminó y quedó guardado en 06_MODELS.
import pickle
import joblib as _joblib  # import temprano (también se re-importa en Celda 7)

RESUME_STATE_PATH = DIRS["checkpoints"] / "resume_state.pkl"

def save_resume_state(state: dict):
    state = dict(state)
    state["last_update"] = str(datetime.now())
    with open(RESUME_STATE_PATH, "wb") as f:
        pickle.dump(state, f)

def load_resume_state() -> dict:
    if RESUME_STATE_PATH.exists():
        try:
            with open(RESUME_STATE_PATH, "rb") as f:
                return pickle.load(f)
        except Exception as e:
            print(f"  [RESUME] No se pudo leer resume_state.pkl ({e}); se ignora y empieza de cero.")
    return {}

def save_checkpoint(obj, name: str, max_retries: int = 2):
    """Guarda un checkpoint de forma resiliente:
    1) Escribe a un archivo .tmp (nunca al nombre final directamente).
    2) Verifica que el .tmp se puede releer/deserializar correctamente
       (detecta escrituras truncadas por corte abrupto de Colab/Drive).
    3) Solo si la verificación pasa, renombra .tmp -> nombre final.
    Si falla, reintenta hasta `max_retries` veces antes de avisar por consola
    en vez de fallar en silencio.
    """
    path = DIRS["checkpoints"] / f"{name}.pkl"
    tmp_path = DIRS["checkpoints"] / f"{name}.pkl.tmp"
    for attempt in range(1, max_retries + 2):
        try:
            _joblib.dump(obj, tmp_path, compress=3)
            _joblib.load(tmp_path)              # verificación: releer antes de confirmar
            tmp_path.replace(path)              # solo confirma si la verificación pasó
            print(f"  [CHECKPOINT] Guardado y verificado: {path} ({_format_bytes(path.stat().st_size)})")
            return path
        except Exception as e:
            print(f"  [CHECKPOINT] [WARN] Intento {attempt} falló para '{name}': {e}")
            if tmp_path.exists():
                try:
                    tmp_path.unlink()
                except Exception:
                    pass
    print(f"  [CHECKPOINT] [ERROR] No se pudo guardar '{name}' tras {max_retries+1} intentos. "
          f"El checkpoint anterior (si existía) se conserva intacto en {path}.")
    return None

def load_checkpoint(name: str):
    path = DIRS["checkpoints"] / f"{name}.pkl"
    if path.exists():
        try:
            obj = _joblib.load(path)
            print(f"  [CHECKPOINT] Cargado: {path}")
            return obj
        except Exception as e:
            print(f"  [CHECKPOINT] Archivo corrupto, se ignora ({path}): {e}")
    return None

def _format_bytes(n_bytes: int) -> str:
    """Formatea un tamaño en bytes de forma legible (evita el '0.00 MB'
    confuso para archivos legítimamente pequeños, como un preprocesador
    sin árboles/coeficientes grandes, mostrando KB cuando corresponde)."""
    if n_bytes < 1024:
        return f"{n_bytes} bytes"
    elif n_bytes < 1024**2:
        return f"{n_bytes/1024:.1f} KB"
    else:
        return f"{n_bytes/1024**2:.2f} MB"

def save_model_pkl(obj, path: "Path", max_retries: int = 2) -> bool:
    """Guarda un modelo/pipeline (p. ej. en 06_MODELS) con el mismo patrón
    resiliente de save_checkpoint: escribe a .tmp, VERIFICA releyendo el
    archivo, y solo entonces renombra al nombre final. Si Drive (vía FUSE)
    deja una escritura a medias por un corte abrupto del kernel, la
    verificación lo detecta y reintenta en vez de dejar un .pkl corrupto o
    silenciosamente ausente.
    """
    if obj is None:
        print(f"  [MODELO] [ERROR] Se intentó guardar None en {path.name} — se aborta "
              f"(esto SÍ sería el bug de guardar un objeto vacío; con obj=None nunca se escribe).")
        return False
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    for attempt in range(1, max_retries + 2):
        try:
            _joblib.dump(obj, tmp_path, compress=3)
            _joblib.load(tmp_path)
            tmp_path.replace(path)
            n_bytes = path.stat().st_size
            size_str = _format_bytes(n_bytes)
            warn = "  ⚠ tamaño sospechosamente pequeño, revisar" if n_bytes < 500 else ""
            print(f"  [MODELO] Guardado y verificado: {path.name} ({size_str}){warn}")
            return True
        except Exception as e:
            print(f"  [MODELO] [WARN] Intento {attempt} falló guardando {path.name}: {e}")
            if tmp_path.exists():
                try:
                    tmp_path.unlink()
                except Exception:
                    pass
    print(f"  [MODELO] [ERROR] No se pudo guardar {path.name} tras {max_retries+1} intentos.")
    return False

def mark_stage_done(stage: str, resume_dict: dict):
    resume_dict[stage] = True
    save_resume_state(resume_dict)
    print(f"  [RESUME] Etapa marcada como completa: '{stage}'")

RESUME = load_resume_state()
if RESUME:
    print(f"  [RESUME] Checkpoint previo encontrado. Etapas completas: "
          f"{sorted([k for k,v in RESUME.items() if v is True])}")
else:
    print("  [RESUME] Sin checkpoints previos — ejecución desde cero.")

REGISTRY = {}

# ══════════════════════════════════════════════════════════════════════════════
# INFORMACIÓN DEL ENTORNO DE EJECUCIÓN (máquina, CPU, RAM, disco, GPU)
# ══════════════════════════════════════════════════════════════════════════════
import socket
import subprocess as _subp

def get_system_info() -> dict:
    info = {}
    info["timestamp"]      = str(datetime.now())
    try:
        info["hostname"] = socket.gethostname()
    except Exception:
        info["hostname"] = "N/D"
    info["platform"]       = platform.platform()
    info["python_version"] = platform.python_version()
    info["machine"]        = platform.machine()

    try:
        import google.colab  # noqa: F401
        info["entorno"] = "Google Colab"
    except Exception:
        info["entorno"] = "Local / otro (no Colab)"

    # CPU
    try:
        import psutil
        info["cpu_count_fisicos"]  = psutil.cpu_count(logical=False)
        info["cpu_count_logicos"] = psutil.cpu_count(logical=True)
        vm = psutil.virtual_memory()
        info["ram_total_gb"]     = round(vm.total / 1024**3, 2)
        info["ram_disponible_gb"] = round(vm.available / 1024**3, 2)
        disk = psutil.disk_usage("/")
        info["disco_total_gb"] = round(disk.total / 1024**3, 2)
        info["disco_libre_gb"] = round(disk.free / 1024**3, 2)
    except Exception as e:
        info["psutil_error"] = str(e)

    # Modelo de CPU (Linux)
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.strip().startswith("model name"):
                    info["cpu_modelo"] = line.split(":", 1)[1].strip()
                    break
    except Exception:
        info["cpu_modelo"] = platform.processor() or "N/D"

    # GPU (si existe)
    try:
        gpu_out = _subp.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=5
        )
        info["gpu"] = gpu_out.stdout.strip() if gpu_out.returncode == 0 and gpu_out.stdout.strip() \
            else "Sin GPU detectada"
    except Exception:
        info["gpu"] = "Sin GPU detectada"

    return info

SYSTEM_INFO = get_system_info()

print("=" * 80)
print("INFORMACIÓN DEL ENTORNO DE EJECUCIÓN")
print("=" * 80)
print(f"Entorno       : {SYSTEM_INFO.get('entorno')}")
print(f"Hostname      : {SYSTEM_INFO.get('hostname')}")
print(f"Plataforma    : {SYSTEM_INFO.get('platform')}")
print(f"Python        : {SYSTEM_INFO.get('python_version')}")
print(f"CPU modelo    : {SYSTEM_INFO.get('cpu_modelo', 'N/D')}")
print(f"CPU núcleos   : {SYSTEM_INFO.get('cpu_count_fisicos','N/D')} físicos / "
      f"{SYSTEM_INFO.get('cpu_count_logicos','N/D')} lógicos")
print(f"RAM           : {SYSTEM_INFO.get('ram_total_gb','N/D')} GB total "
      f"(disponible: {SYSTEM_INFO.get('ram_disponible_gb','N/D')} GB)")
print(f"Disco         : {SYSTEM_INFO.get('disco_total_gb','N/D')} GB total "
      f"(libre: {SYSTEM_INFO.get('disco_libre_gb','N/D')} GB)")
print(f"GPU           : {SYSTEM_INFO.get('gpu')}")
print("=" * 80)

save_json(SYSTEM_INFO, "system_info.json", subdir="logs")

print(f"Python {platform.python_version()} | NumPy {np.__version__} | Pandas {pd.__version__}")
print(f"RANDOM_STATE = {RANDOM_STATE}  |  DPI_PRINT = {DPI_PRINT}")


# ══════════════════════════════════════════════════════════════════════════════
# CELDA 2 · Configuración de MLflow
# ══════════════════════════════════════════════════════════════════════════════
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
import mlflow
from mlflow.models import infer_signature

MLFLOW_DIR      = Path("outputs/mlruns")
EXPERIMENT_NAME = "BPN_Peru_Q1"
mlflow.set_tracking_uri(str(MLFLOW_DIR))
mlflow.set_experiment(EXPERIMENT_NAME)

def mlflow_log_run(run_name: str, params: dict, metrics: dict,
                   artifacts: list = None, tags: dict = None):
    with mlflow.start_run(run_name=run_name, tags=tags or {}):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        if artifacts:
            for a in artifacts:
                if Path(a).exists():
                    mlflow.log_artifact(a)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: '{EXPERIMENT_NAME}'")


# ══════════════════════════════════════════════════════════════════════════════
# CELDA 3 · Mapeo de columnas CNV-MINSA
# ══════════════════════════════════════════════════════════════════════════════
COL = {
    "anio"              : "FecNac_Año",
    "mes"               : "FecNac_Mes",
    "peso"              : "PESO_NACIDO",
    "talla"             : "TALLA_NACIDO",
    "semanas_gestacion" : "DUR_EMB_PARTO",
    "condicion_parto"   : "Condicion_Parto",
    "sexo"              : "sexo_nacido",
    "tipo_parto"        : "Tipo_Parto",
    "edad_madre"        : "Edad_Madre",
    "estado_civil"      : "Estado_Civil",
    "nivel_educacion"   : "Nivel_Intrucción_Madre",
    "ocupacion"         : "DESC_OCUPACION",
    "n_embarazos"       : "Num_embar_madre",
    "hijos_vivos"       : "Hijos_vivo_madre",
    "hijos_fallecidos"  : "Hijos_fallec_madre",
    "abortos_previos"   : "nacmuer_abort_madre",
    "pais_madre"        : "Pais_Madre",
    "ubigeo"            : "IdUbigeoInei",
    "ipress"            : "Ipress",
    "lugar_nacimiento"  : "Lugar_Nacido",
    "atiende_parto"     : "Atiende_Parto",
    "financiador"       : "Financiador_Parto"
}

UBIGEO_DEPT = {
    "01":"Amazonas","02":"Áncash","03":"Apurímac","04":"Arequipa",
    "05":"Ayacucho","06":"Cajamarca","07":"Callao","08":"Cusco",
    "09":"Huancavelica","10":"Huánuco","11":"Ica","12":"Junín",
    "13":"La Libertad","14":"Lambayeque","15":"Lima","16":"Loreto",
    "17":"Madre de Dios","18":"Moquegua","19":"Pasco","20":"Piura",
    "21":"Puno","22":"San Martín","23":"Tacna","24":"Tumbes","25":"Ucayali",
}

REGION_MAP = {
    "Amazonas":"Sierra-Selva","Áncash":"Sierra","Apurímac":"Sierra",
    "Arequipa":"Costa","Ayacucho":"Sierra","Cajamarca":"Sierra",
    "Callao":"Costa","Cusco":"Sierra","Huancavelica":"Sierra",
    "Huánuco":"Sierra-Selva","Ica":"Costa","Junín":"Sierra",
    "La Libertad":"Costa","Lambayeque":"Costa","Lima":"Costa",
    "Loreto":"Selva","Madre de Dios":"Selva","Moquegua":"Costa",
    "Pasco":"Sierra-Selva","Piura":"Costa","Puno":"Sierra",
    "San Martín":"Selva","Tacna":"Costa","Tumbes":"Costa","Ucayali":"Selva",
}

NUM_RAW = [
    COL["peso"], COL["talla"], COL["semanas_gestacion"],
    COL["edad_madre"], COL["n_embarazos"], COL["hijos_vivos"],
    COL["hijos_fallecidos"], COL["abortos_previos"],
    COL["anio"], COL["mes"],
]

CAT_PRENATAL = [
    COL["condicion_parto"],
    COL["sexo"],
    COL["tipo_parto"],
    COL["estado_civil"],
    COL["nivel_educacion"],
    COL["ocupacion"],
    COL["pais_madre"],
    COL["ubigeo"],
    COL["ipress"],
    COL["lugar_nacimiento"],
    COL["atiende_parto"],
    COL["financiador"],
]

print(f"Mapeo CNV: {len(COL)} columnas, {len(UBIGEO_DEPT)} departamentos.")
print("=" * 80)


# ══════════════════════════════════════════════════════════════════════════════
# CELDA 4 · Carga del dataset y procesamiento optimizado (CON RESUME)
# ══════════════════════════════════════════════════════════════════════════════
if RESUME.get("data_loaded") and (DIRS["checkpoints"] / "dataset_state.pkl").exists():
    print("  [RESUME] Checkpoint de dataset encontrado — se omite recarga/reprocesamiento "
          "del Parquet completo (evita volver a escanear/filtrar las filas originales).")
    _dstate = load_checkpoint("dataset_state")
    df            = _dstate["df"]
    n_original    = _dstate["n_original"]
    anios         = _dstate["anios"]
    checksum      = _dstate["checksum"]
    retencion     = _dstate["retencion"]
    bpn           = _dstate["bpn"]
    mediana_talla = _dstate["mediana_talla"]
    print(f"  Válidos (checkpoint): {df.height:,}  |  BPN: {bpn:.2%}  |  Años: {anios}")
else:
    import polars as pl

    if not PARQUET_PATH.exists():
        print("Convirtiendo CSV → Parquet...")
        (
            pl.scan_csv(
                CSV_PATH,
                infer_schema_length=10000,
                ignore_errors=True,
                truncate_ragged_lines=True,
                encoding="utf8-lossy"
            )
            .sink_parquet(PARQUET_PATH, compression="zstd")
        )

    print(f"Parquet encontrado: {PARQUET_PATH}")

    lf = pl.scan_parquet(PARQUET_PATH)
    schema = lf.collect_schema()
    cols   = schema.names()

    n_original = lf.select(pl.len()).collect().item()

    rename_map = {
        "FecNac_AÃ±o": "FecNac_Año",
        "Nivel_IntrucciÃ³n_Madre": "Nivel_Intrucción_Madre"
    }
    rename_final = {k: v for k, v in rename_map.items() if k in cols}
    if rename_final:
        lf = lf.rename(rename_final)
    cols = lf.collect_schema().names()

    lf = lf.with_columns([
        pl.col(c).cast(pl.Float64, strict=False)
        for c in NUM_RAW if c in cols
    ])

    lf = lf.with_columns([
        (pl.col(COL["peso"]) < 2500).cast(pl.Int8).alias("BAJO_PESO"),
        (pl.col(COL["ubigeo"]).cast(pl.Utf8).str.slice(0, 2)
         .replace(UBIGEO_DEPT)).alias("DEPARTAMENTO"),
    ])

    lf = lf.with_columns(
        pl.when(pl.col(COL["edad_madre"]) < 15).then(pl.lit("≤14"))
        .when(pl.col(COL["edad_madre"]) <= 19).then(pl.lit("15-19"))
        .when(pl.col(COL["edad_madre"]) <= 34).then(pl.lit("20-34"))
        .otherwise(pl.lit("≥35")).alias("GRUPO_ETARIO")
    )

    mediana_talla = lf.select(pl.col(COL["talla"]).median()).collect().item()
    lf = lf.with_columns(pl.col(COL["talla"]).fill_null(mediana_talla))

    _emb  = pl.col(COL["n_embarazos"]).fill_null(1)
    _viv  = pl.col(COL["hijos_vivos"]).fill_null(0)
    _fall = pl.col(COL["hijos_fallecidos"]).fill_null(0)
    _abor = pl.col(COL["abortos_previos"]).fill_null(0)
    _edad = pl.col(COL["edad_madre"])

    lf = lf.with_columns([
        (_fall + _abor).alias("ROA"),
        ((_fall + _abor) / _emb.clip(1)).clip(0, 1).alias("TPF"),
        (_viv / _emb.clip(1)).clip(0, 1).alias("EM"),
        (_emb == 1).cast(pl.Int8).alias("PRIMIGESTA"),
        (_emb >= 5).cast(pl.Int8).alias("GRAN_MULTIPARA"),
        ((_edad < 15) | (_edad > 40)).cast(pl.Int8).alias("RIESGO_EXTREMO"),
        ((_fall > 0) | (_abor > 0)).cast(pl.Int8).alias("ANTECEDENTE_PERDIDA"),
    ])

    lf = lf.filter(
        pl.col(COL["peso"]).is_between(300, 7000) &
        pl.col(COL["edad_madre"]).is_between(10, 55) &
        pl.col(COL["talla"]).is_between(20, 65)
    )

    df = lf.collect()

    bpn        = float(df["BAJO_PESO"].mean())
    anios      = df[COL["anio"]].drop_nulls().cast(pl.Int32).unique().sort().to_list()
    checksum   = hashlib.md5(
        str(df.select([COL["anio"], "BAJO_PESO"]).to_numpy()).encode()
    ).hexdigest()[:8]
    retencion  = df.height / n_original

    print("=" * 60)
    print(f"Original : {n_original:,}")
    print(f"Válidos  : {df.height:,}")
    print(f"Retención: {retencion:.2%}")
    print(f"Columnas : {df.width}")
    print(f"BPN      : {bpn:.2%}")
    print(f"Años     : {anios}")
    print(f"Checksum : {checksum}")
    print("=" * 60)

    save_json({
        "dataset_checksum": checksum,
        "n_original": int(n_original),
        "n_rows": int(df.height),
        "n_cols": int(df.width),
        "retention_rate": float(retencion),
        "bpn_rate": float(bpn),
        "years": anios,
        "talla_imputada": True,
        "talla_fill": float(mediana_talla),
    }, "dataset_info.json", subdir="01_DATASET")

    save_checkpoint({
        "df": df, "n_original": n_original, "anios": anios,
        "checksum": checksum, "retencion": retencion, "bpn": bpn,
        "mediana_talla": mediana_talla,
    }, "dataset_state")
    mark_stage_done("data_loaded", RESUME)

# ══════════════════════════════════════════════════════════════════════════════
# MÓDULOS 1-5 · Dataset Audit, Drift, Leakage, Feature Engineering, Feature Stability
# (CON RESUME: si ya se completaron en una corrida previa, se omiten por completo —
#  sus archivos de salida ya están a salvo en Google Drive desde esa corrida anterior;
#  esto solo evita VOLVER A CALCULARLOS, ahorrando tiempo tras un reinicio de Colab)
# ══════════════════════════════════════════════════════════════════════════════
if RESUME.get("features_engineered") and (DIRS["checkpoints"] / "features_state.pkl").exists():
    print("\n" + "═" * 60)
    print("  [RESUME] Módulos 1-5 ya completados en una corrida anterior — se omiten.")
    print("  (Sus reportes/figuras ya están guardados en Drive; solo se recargan las")
    print("   variables necesarias: df_tr, df_te, pdf, FEATURES, NUM_PRENATAL, etc.)")
    print("═" * 60)
    _fstate = load_checkpoint("features_state")
    df_tr        = _fstate["df_tr"]
    df_te        = _fstate["df_te"]
    pdf          = _fstate["pdf"]
    FEATURES     = _fstate["FEATURES"]
    NUM_PRENATAL = _fstate["NUM_PRENATAL"]
    CAT_PRENATAL = _fstate["CAT_PRENATAL"]
    TARGET       = _fstate["TARGET"]
    y_tr         = _fstate["y_tr"]
    y_te         = _fstate["y_te"]
    spw          = _fstate["spw"]
    SUBG         = _fstate["SUBG"]
    print(f"  Train: {len(df_tr):,}  |  Test: {len(df_te):,}  |  Features: {len(FEATURES)}")
else:
    # ══════════════════════════════════════════════════════════════════════════════
    # MÓDULO 1 · DATASET AUDIT (EDA completo + PDF/XLSX)
    # ══════════════════════════════════════════════════════════════════════════════
    print("\n" + "═" * 60)
    print("  MÓDULO 1 · DATASET AUDIT")
    print("═" * 60)

    try:
        df_pd = df.to_pandas()

        mv_df = pd.DataFrame({
            "Column"   : df_pd.columns,
            "Missing_N": df_pd.isnull().sum().values,
            "Missing_%": (df_pd.isnull().mean() * 100).round(2).values,
            "Dtype"    : [str(t) for t in df_pd.dtypes.values],
        }).sort_values("Missing_%", ascending=False)
        save_table_apa(mv_df, "01_missing_values.csv",
                       "Missing values report", subdir="01_DATASET")

        n_dups = df_pd.duplicated().sum()
        print(f"  Duplicados: {n_dups:,}")

        const_cols, quasi_cols = [], []
        for c in df_pd.columns:
            vc = df_pd[c].value_counts(normalize=True)
            if len(vc) == 1:
                const_cols.append(c)
            elif len(vc) > 0 and vc.iloc[0] > 0.95:
                quasi_cols.append((c, round(vc.iloc[0]*100, 2)))

        card_df = pd.DataFrame({
            "Column"      : df_pd.columns,
            "Unique_values": [df_pd[c].nunique() for c in df_pd.columns],
            "Top_value"   : [str(df_pd[c].value_counts().index[0])
                             if df_pd[c].notna().any() else "NA"
                             for c in df_pd.columns],
            "Top_freq_%"  : [(df_pd[c].value_counts(normalize=True).iloc[0]*100).round(2)
                             if df_pd[c].notna().any() else 0
                             for c in df_pd.columns],
        })
        save_table_apa(card_df, "01_cardinality.csv",
                       "Cardinality report", subdir="01_DATASET")

        num_cols_raw = df_pd.select_dtypes(include=np.number).columns.tolist()
        desc_df = df_pd[num_cols_raw].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T
        desc_df["skew"]     = df_pd[num_cols_raw].skew()
        desc_df["kurtosis"] = df_pd[num_cols_raw].kurtosis()
        save_table_apa(desc_df.reset_index().rename(columns={"index":"Variable"}),
                       "01_descriptive_stats.csv",
                       "Descriptive statistics", subdir="01_DATASET")

        corr_matrix = df_pd[num_cols_raw].corr()
        save_table_apa(corr_matrix.reset_index().rename(columns={"index":"Variable"}),
                       "01_correlation_matrix.csv",
                       "Pearson correlation matrix", subdir="01_DATASET")

        outlier_rows = []
        for c in num_cols_raw:
            q1, q3 = df_pd[c].quantile(0.25), df_pd[c].quantile(0.75)
            iqr     = q3 - q1
            n_out   = ((df_pd[c] < q1 - 1.5*iqr) | (df_pd[c] > q3 + 1.5*iqr)).sum()
            outlier_rows.append({"Column": c, "Q1": round(q1,3), "Q3": round(q3,3),
                                 "IQR": round(iqr,3), "Outliers_N": n_out,
                                 "Outliers_%": round(n_out/len(df_pd)*100,2)})
        outlier_df = pd.DataFrame(outlier_rows).sort_values("Outliers_%", ascending=False)
        save_table_apa(outlier_df, "01_outliers_iqr.csv",
                       "Outliers IQR report", subdir="01_DATASET")

        plot_cols = [c for c in num_cols_raw if df_pd[c].nunique() > 5][:12]
        if plot_cols:
            n_r = (len(plot_cols) + 3) // 4
            fig, axes = plt.subplots(n_r, 4, figsize=(16, 4*n_r))
            axes = axes.ravel()
            for i, c in enumerate(plot_cols):
                axes[i].hist(df_pd[c].dropna(), bins=40, color=PAL[0], edgecolor="none", alpha=0.8)
                axes[i].set_title(c, fontsize=8, fontweight="bold")
                axes[i].set_xlabel("Value", fontsize=7)
            for j in range(i+1, len(axes)):
                axes[j].set_visible(False)
            plt.suptitle("Módulo 1 – Histogramas (variables numéricas)", fontweight="bold")
            plt.tight_layout()
            save_fig(fig, "M01_histogramas.png", subdir="02_EDA")
            plt.close()

        target_col = "BAJO_PESO"
        box_cols   = [c for c in num_cols_raw if c != target_col][:8]
        if box_cols:
            fig, axes = plt.subplots(2, 4, figsize=(16, 8))
            axes = axes.ravel()
            for i, c in enumerate(box_cols):
                tmp = df_pd[[c, target_col]].dropna()
                axes[i].boxplot([tmp[tmp[target_col]==0][c], tmp[tmp[target_col]==1][c]],
                                labels=["Normal","LBW"], patch_artist=True,
                                boxprops=dict(facecolor=PAL[0], alpha=0.7))
                axes[i].set_title(c, fontsize=8, fontweight="bold")
            for j in range(i+1, len(axes)):
                axes[j].set_visible(False)
            plt.suptitle("Módulo 1 – Boxplots por clase (Normal vs LBW)", fontweight="bold")
            plt.tight_layout()
            save_fig(fig, "M01_boxplots.png", subdir="02_EDA")
            plt.close()

        fig, ax = plt.subplots(figsize=(12, 10))
        corr_sub = corr_matrix.iloc[:20, :20]
        mask = np.triu(np.ones_like(corr_sub, dtype=bool))
        sns.heatmap(corr_sub, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
                    center=0, linewidths=0.3, ax=ax, annot_kws={"size":7})
        ax.set_title("Módulo 1 – Correlation Heatmap", fontweight="bold")
        plt.tight_layout()
        save_fig(fig, "M01_correlation_heatmap.png", subdir="02_EDA")
        plt.close()

        save_excel_multi({
            "Missing_Values"  : mv_df,
            "Carbonality"     : card_df,
            "Descriptive_Stats": desc_df.reset_index(),
            "Outliers_IQR"    : outlier_df,
            "Correlation"     : corr_matrix.reset_index(),
        }, "dataset_report.xlsx", subdir="01_DATASET")

        audit_summary = {
            "n_rows": int(df.height), "n_cols": int(df.width),
            "n_duplicates": int(n_dups),
            "n_constant_cols": len(const_cols),
            "n_quasi_constant_cols": len(quasi_cols),
            "constant_cols": const_cols,
        }
        save_json(audit_summary, "dataset_audit.json", subdir="01_DATASET")
        print("  Módulo 1 completado.")

    except Exception as e:
        print(f"  [WARN] Módulo 1 error parcial: {e}")


    # ══════════════════════════════════════════════════════════════════════════════
    # CELDA 5 · Definición de features y split temporal
    # ══════════════════════════════════════════════════════════════════════════════
    NUM_PRENATAL = [
        COL["edad_madre"], COL["n_embarazos"], COL["hijos_vivos"],
        COL["hijos_fallecidos"], COL["abortos_previos"],
        "ROA", "TPF", "EM",
    ]
    CAT_PRENATAL = [
        COL["sexo"], COL["nivel_educacion"], COL["estado_civil"],
        COL["financiador"], COL["lugar_nacimiento"], COL["atiende_parto"],
        "PRIMIGESTA", "GRAN_MULTIPARA", "RIESGO_EXTREMO", "ANTECEDENTE_PERDIDA",
    ]
    FEATURES = NUM_PRENATAL + CAT_PRENATAL
    TARGET   = "BAJO_PESO"
    AUX_COLS = ["DEPARTAMENTO", "GRUPO_ETARIO", COL["anio"], COL["mes"]]

    keep = list(dict.fromkeys(FEATURES + [TARGET] + AUX_COLS))
    pdf  = df.select([c for c in keep if c in df.columns]).to_pandas()

    for c in CAT_PRENATAL:
        if c in pdf.columns:
            pdf[c] = pdf[c].astype(object)
            pdf[c] = pdf[c].where(pdf[c].notna(), np.nan)

    pdf["REGION"] = pdf["DEPARTAMENTO"].map(REGION_MAP).fillna("Sin dato")

    # ── OPTIMIZACIÓN: downcast numéricas a float32 (reduce ~50% memoria RAM) ──────
    for _c in NUM_PRENATAL:
        if _c in pdf.columns:
            pdf[_c] = pd.to_numeric(pdf[_c], errors="coerce").astype(np.float32)

    print(f"Tabla de modelado: {len(pdf):,} filas × {len(pdf.columns)} cols")
    print(f"Balance: {pdf[TARGET].value_counts(normalize=True).round(4).to_dict()}")

    mask_train = pdf[COL["anio"]] < 2025
    df_tr = pdf.loc[mask_train].copy()
    df_te = pdf.loc[~mask_train].copy()
    y_tr  = df_tr[TARGET].astype(int)
    y_te  = df_te[TARGET].astype(int)
    spw   = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

    print(f"\nTrain 2015-2024: {len(df_tr):,}  |  BPN: {y_tr.mean()*100:.2f}%")
    print(f"Test  2025:      {len(df_te):,}  |  BPN: {y_te.mean()*100:.2f}%")
    print(f"scale_pos_weight ≈ {spw:.2f}")

    SUBG = {
        "sexo"   : df_te[COL["sexo"]].astype(str).values,
        "region" : df_te["REGION"].astype(str).values,
        "edad"   : df_te["GRUPO_ETARIO"].astype(str).values,
    }


    # ══════════════════════════════════════════════════════════════════════════════
    # MÓDULO 2 · DATA DRIFT EXTENDIDO (PSI, KS, JS, Wasserstein, Energy Distance)
    # ══════════════════════════════════════════════════════════════════════════════
    print("\n" + "═" * 60)
    print("  MÓDULO 2 · DATA DRIFT EXTENDIDO")
    print("═" * 60)

    try:
        from scipy.stats import ks_2samp, wasserstein_distance, entropy as scipy_entropy
        from scipy.spatial.distance import jensenshannon

        def compute_psi(expected, actual, bins=10):
            expected, actual = np.array(expected), np.array(actual)
            bp = np.percentile(expected, np.linspace(0, 100, bins+1))
            bp[0], bp[-1] = -np.inf, np.inf
            e_pct = np.histogram(expected, bp)[0] / max(len(expected), 1)
            a_pct = np.histogram(actual, bp)[0] / max(len(actual), 1)
            e_pct = np.where(e_pct == 0, 1e-6, e_pct)
            a_pct = np.where(a_pct == 0, 1e-6, a_pct)
            return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))

        def compute_js_divergence(ref, cur, bins=50):
            bp = np.percentile(np.concatenate([ref, cur]), np.linspace(0, 100, bins+1))
            bp[0], bp[-1] = -np.inf, np.inf
            p = np.histogram(ref, bp)[0].astype(float) + 1e-6
            q = np.histogram(cur, bp)[0].astype(float) + 1e-6
            p /= p.sum(); q /= q.sum()
            return float(jensenshannon(p, q))

        def compute_energy_distance(ref, cur):
            n, m = len(ref), len(cur)
            if n == 0 or m == 0:
                return np.nan
            ref_s = np.sort(ref); cur_s = np.sort(cur)
            try:
                return float(wasserstein_distance(ref_s, cur_s))
            except:
                return np.nan

        ref_year    = min(anios) if anios else 2015
        vars_drift  = NUM_PRENATAL[:6]
        drift_rows  = []

        for yr in sorted(pdf[COL["anio"]].dropna().unique()):
            yr = int(yr)
            if yr == ref_year:
                continue
            ref_mask = pdf[COL["anio"]] == ref_year
            cur_mask = pdf[COL["anio"]] == yr
            ref_data = pdf[ref_mask]
            cur_data = pdf[cur_mask]
            if len(ref_data) < 100 or len(cur_data) < 100:
                continue
            row = {"Year": yr, "N": len(cur_data),
                   "BPN_rate": round(cur_data[TARGET].mean(), 4),
                   "BPN_PSI" : round(compute_psi(ref_data[TARGET].values,
                                                  cur_data[TARGET].values, bins=2), 4)}
            for v in vars_drift:
                if v not in pdf.columns:
                    continue
                rv = ref_data[v].dropna().values
                cv = cur_data[v].dropna().values
                if len(rv) < 50 or len(cv) < 50:
                    continue
                row[f"{v}_PSI"]      = round(compute_psi(rv, cv), 4)
                row[f"{v}_KS"]       = round(ks_2samp(rv, cv).statistic, 4)
                row[f"{v}_KS_p"]     = round(ks_2samp(rv, cv).pvalue, 6)
                row[f"{v}_JS"]       = round(compute_js_divergence(rv, cv), 4)
                row[f"{v}_Wass"]     = round(wasserstein_distance(rv, cv), 4)
                row[f"{v}_Energy"]   = round(compute_energy_distance(rv, cv), 4)
            drift_rows.append(row)

        df_drift = pd.DataFrame(drift_rows)
        save_table_apa(df_drift, "M02_data_drift_extended.csv",
                       "Módulo 2. Data drift extendido (PSI, KS, JS, Wasserstein, Energy Distance)",
                       subdir="13_DRIFT")

        psi_cols = [c for c in df_drift.columns if c.endswith("_PSI")]
        if psi_cols and len(df_drift) > 0:
            fig, ax = plt.subplots(figsize=(12, 5))
            heat = df_drift[["Year"] + psi_cols].set_index("Year")
            heat.columns = [c.replace("_PSI", "") for c in heat.columns]
            sns.heatmap(heat.T, annot=True, fmt=".3f", cmap="YlOrRd",
                        linewidths=0.5, ax=ax, vmin=0, vmax=0.25)
            ax.set_title("Módulo 2 – PSI por Año y Variable\n"
                         "(PSI>0.2 = drift severo; 0.1–0.2 = moderado; <0.1 = estable)",
                         fontweight="bold")
            plt.tight_layout()
            save_fig(fig, "M02_PSI_heatmap.png", subdir="13_DRIFT")
            plt.close()

        wass_cols = [c for c in df_drift.columns if c.endswith("_Wass")]
        if wass_cols and len(df_drift) > 0:
            fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)
            for wc in wass_cols[:5]:
                feat = wc.replace("_Wass", "")
                ax.plot(df_drift["Year"], df_drift[wc], "o-", lw=1.8, label=feat)
            ax.set_xlabel("Year"); ax.set_ylabel("Wasserstein Distance")
            ax.set_title("Módulo 2 – Wasserstein Distance por Año", fontweight="bold")
            ax.legend(fontsize=7)
            plt.tight_layout()
            save_fig(fig, "M02_Wasserstein_temporal.png", subdir="13_DRIFT")
            plt.close()

        print("  Módulo 2 completado.")
    except Exception as e:
        print(f"  [WARN] Módulo 2 error: {e}")


    # ══════════════════════════════════════════════════════════════════════════════
    # MÓDULO 3 · DATA LEAKAGE CHECKER (Q1 EXTREME OPTIMIZED)
    # ══════════════════════════════════════════════════════════════════════════════
    print("\n" + "═"*70)
    print(" MÓDULO 3 · DATA LEAKAGE CHECKER (Q1 EXTREME)")
    print("═"*70)

    try:
        import time
        import numpy as np
        import pandas as pd

        from sklearn.metrics import roc_auc_score
        from sklearn.preprocessing import LabelEncoder
        from sklearn.feature_selection import mutual_info_classif
        from scipy.stats import spearmanr

        t0 = time.time()

        TARGET_ARR = pdf[TARGET].values
        leakage_rows = []

        MAX_SAMPLE = 300000
        rng = np.random.RandomState(RANDOM_STATE)

        if len(pdf) > MAX_SAMPLE:
            sample_idx = rng.choice(len(pdf), MAX_SAMPLE, replace=False)
            print(f"Usando muestra reproducible de {MAX_SAMPLE:,} registros")
        else:
            sample_idx = np.arange(len(pdf))
            print(f"Usando dataset completo ({len(pdf):,})")

        y_sample = TARGET_ARR[sample_idx]

        for col_name in FEATURES:
            if col_name not in pdf.columns:
                continue
            s = pdf[col_name]

            if s.dtype == object:
                le = LabelEncoder()
                x_all = le.fit_transform(s.fillna("__NA__").astype(str)).astype(np.float32)
            else:
                x_all = (pd.to_numeric(s, errors="coerce").fillna(0).astype(np.float32).values)

            x_sample = x_all[sample_idx]

            try:
                pearson = abs(np.corrcoef(x_sample, y_sample)[0,1])
            except:
                pearson = np.nan

            try:
                spearman = abs(spearmanr(x_sample, y_sample)[0])
            except:
                spearman = np.nan

            try:
                auc = roc_auc_score(y_sample, x_sample)
                auc = max(auc, 1-auc)
            except:
                auc = np.nan

            try:
                mi = mutual_info_classif(x_sample.reshape(-1,1), y_sample,
                                          random_state=RANDOM_STATE, discrete_features=False)[0]
            except:
                mi = np.nan

            score = np.nanmean([pearson, spearman, auc, mi])

            if ((not np.isnan(auc) and auc > 0.85)
                or (not np.isnan(pearson) and pearson > 0.90)
                or (not np.isnan(spearman) and spearman > 0.90)):
                flag = "⚠ POSIBLE LEAKAGE"
            else:
                flag = "OK"

            leakage_rows.append({
                "Feature":col_name,
                "Pearson":round(pearson,5) if not np.isnan(pearson) else None,
                "Spearman":round(spearman,5) if not np.isnan(spearman) else None,
                "ROC_AUC":round(auc,5) if not np.isnan(auc) else None,
                "Mutual_Info":round(mi,5) if not np.isnan(mi) else None,
                "Leakage_Score":round(score,5) if not np.isnan(score) else None,
                "Flag":flag
            })

        df_leakage = (pd.DataFrame(leakage_rows)
                      .sort_values("Leakage_Score", ascending=False)
                      .reset_index(drop=True))

        save_table_apa(df_leakage, "M03_leakage_report.csv",
                       caption="Data Leakage Report", subdir="03_PREPROCESSING")
        save_excel_multi({"Leakage_Report":df_leakage}, "M03_leakage_report.xlsx",
                         subdir="03_PREPROCESSING")

        n_flag = int((df_leakage.Flag=="⚠ POSIBLE LEAKAGE").sum())

        save_json({
            "rows_dataset":int(len(pdf)),
            "rows_sample":int(len(sample_idx)),
            "variables":int(len(df_leakage)),
            "possible_leakage":n_flag,
            "execution_seconds":round(time.time()-t0, 2)
        }, "M03_summary.json", subdir="03_PREPROCESSING")

        print(f"\nVariables evaluadas : {len(df_leakage)}")
        print(f"Variables sospechosas: {n_flag}")
        print(f"Tiempo: {time.time()-t0:.2f} segundos")
        print("Módulo 3 completado.")

    except Exception as e:
        print(f"[ERROR] Módulo 3: {e}")

    # ══════════════════════════════════════════════════════════════════════════════
    # MÓDULO 4 · FEATURE ENGINEERING EXTENDIDO
    # ══════════════════════════════════════════════════════════════════════════════
    print("\n" + "═" * 60)
    print("  MÓDULO 4 · FEATURE ENGINEERING EXTENDIDO")
    print("═" * 60)

    try:
        df_tr = df_tr.copy()
        df_te = df_te.copy()

        df_tr["EDAD_x_ROA"]     = df_tr[COL["edad_madre"]].fillna(0) * df_tr["ROA"].fillna(0)
        df_te["EDAD_x_ROA"]     = df_te[COL["edad_madre"]].fillna(0) * df_te["ROA"].fillna(0)
        df_tr["EDAD_x_EMB"]     = df_tr[COL["edad_madre"]].fillna(0) * df_tr[COL["n_embarazos"]].fillna(1)
        df_te["EDAD_x_EMB"]     = df_te[COL["edad_madre"]].fillna(0) * df_te[COL["n_embarazos"]].fillna(1)
        df_tr["ROA_x_PRIMIG"]   = df_tr["ROA"].fillna(0) * df_tr["PRIMIGESTA"].fillna(0)
        df_te["ROA_x_PRIMIG"]   = df_te["ROA"].fillna(0) * df_te["PRIMIGESTA"].fillna(0)
        df_tr["TPF_x_RIESGO"]   = df_tr["TPF"].fillna(0) * df_tr["RIESGO_EXTREMO"].fillna(0)
        df_te["TPF_x_RIESGO"]   = df_te["TPF"].fillna(0) * df_te["RIESGO_EXTREMO"].fillna(0)

        df_tr["EDAD2"]  = df_tr[COL["edad_madre"]].fillna(0) ** 2
        df_te["EDAD2"]  = df_te[COL["edad_madre"]].fillna(0) ** 2
        df_tr["ROA2"]   = df_tr["ROA"].fillna(0) ** 2
        df_te["ROA2"]   = df_te["ROA"].fillna(0) ** 2

        # FIX Módulo 7 (LOYO): estas mismas interacciones también deben existir en
        # `pdf` (el dataframe completo pre-split), porque Leave-One-Year-Out
        # reconstruye folds por año directamente desde `pdf.loc[..., FEATURES]`.
        # Si no se replican aquí, pdf carece de EDAD_x_ROA/EDAD_x_EMB/etc. y
        # Módulo 7 falla con KeyError "not in index" para todos los años.
        # Son transformaciones algebraicas fila-a-fila (no usan estadísticas de
        # train), por lo que replicarlas en pdf no introduce fuga de datos.
        pdf["EDAD_x_ROA"]   = pdf[COL["edad_madre"]].fillna(0) * pdf["ROA"].fillna(0)
        pdf["EDAD_x_EMB"]   = pdf[COL["edad_madre"]].fillna(0) * pdf[COL["n_embarazos"]].fillna(1)
        pdf["ROA_x_PRIMIG"] = pdf["ROA"].fillna(0) * pdf["PRIMIGESTA"].fillna(0)
        pdf["TPF_x_RIESGO"] = pdf["TPF"].fillna(0) * pdf["RIESGO_EXTREMO"].fillna(0)
        pdf["EDAD2"]        = pdf[COL["edad_madre"]].fillna(0) ** 2
        pdf["ROA2"]         = pdf["ROA"].fillna(0) ** 2

        for cat_c in [COL["nivel_educacion"], COL["financiador"], COL["lugar_nacimiento"]]:
            if cat_c not in df_tr.columns:
                continue
            vc = df_tr[cat_c].value_counts()
            rare = vc[vc / len(df_tr) < 0.01].index.tolist()
            if rare:
                df_tr[cat_c] = df_tr[cat_c].replace(dict.fromkeys(rare, "OTRO_RARO"))
                df_te[cat_c] = df_te[cat_c].replace(dict.fromkeys(rare, "OTRO_RARO"))

        NEW_NUM_FEATS = ["EDAD_x_ROA", "EDAD_x_EMB", "ROA_x_PRIMIG",
                         "TPF_x_RIESGO", "EDAD2", "ROA2"]
        NUM_PRENATAL_EXT = NUM_PRENATAL + NEW_NUM_FEATS
        FEATURES_EXT     = NUM_PRENATAL_EXT + CAT_PRENATAL

        NUM_PRENATAL = NUM_PRENATAL_EXT
        FEATURES     = FEATURES_EXT

        save_json({"new_features": NEW_NUM_FEATS, "total_features": len(FEATURES)},
                  "feature_engineering.json", subdir="04_FEATURE_ENGINEERING")
        print(f"  Features totales (con interacciones): {len(FEATURES)}")
        print("  Módulo 4 completado.")
    except Exception as e:
        print(f"  [WARN] Módulo 4 error: {e}")

    # ══════════════════════════════════════════════════════════════════════════════
    # MÓDULO 5 · FEATURE STABILITY (Q1 EXTREME OPTIMIZADO)
    # ══════════════════════════════════════════════════════════════════════════════
    print("\n" + "═"*70)
    print(" MÓDULO 5 · FEATURE STABILITY (Q1 EXTREME)")
    print("═"*70)

    try:
        import time
        import numpy as np
        import pandas as pd

        from scipy.stats import spearmanr
        from sklearn.model_selection import StratifiedKFold
        from sklearn.ensemble import ExtraTreesClassifier
        from sklearn.preprocessing import LabelEncoder

        t0 = time.time()

        MAX_SAMPLE = 10000
        if len(df_tr) > MAX_SAMPLE:
            idx = df_tr.sample(MAX_SAMPLE, random_state=RANDOM_STATE).index
        else:
            idx = df_tr.index

        X = df_tr.loc[idx, FEATURES].copy()
        y = y_tr.loc[idx].copy()

        for c in X.columns:
            if X[c].dtype == object:
                le = LabelEncoder()
                X[c] = le.fit_transform(X[c].fillna("__NA__").astype(str))

        X = X.fillna(0)

        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        importances = []

        for fold, (tr, _) in enumerate(skf.split(X,y),1):
            print(f" Fold {fold}/3")
            model = ExtraTreesClassifier(n_estimators=50, max_depth=8,
                                         random_state=RANDOM_STATE, n_jobs=-1)
            model.fit(X.iloc[tr], y.iloc[tr])
            importances.append(model.feature_importances_)

        imp = np.vstack(importances)
        rows = []
        for i,f in enumerate(X.columns):
            vals = imp[:,i]
            mean = np.mean(vals)
            std = np.std(vals)
            cv = std/(mean+1e-9)
            try:
                rho = spearmanr(vals, np.arange(len(vals))).statistic
            except:
                rho = np.nan

            if cv < 0.15:
                stab = "EXCELENTE"
            elif cv < 0.30:
                stab = "ALTA"
            elif cv < 0.50:
                stab = "MEDIA"
            else:
                stab = "BAJA"

            rows.append({
                "Feature":f, "Importance_Mean":round(mean,6), "Importance_STD":round(std,6),
                "CV":round(cv,4),
                "Spearman":round(rho,4) if not np.isnan(rho) else None,
                "Stability":stab
            })

        df_stability = (pd.DataFrame(rows)
                        .sort_values("Importance_Mean", ascending=False)
                        .reset_index(drop=True))

        save_table_apa(df_stability, "M05_feature_stability.csv",
                       "Feature Stability Analysis", subdir="05_FEATURE_SELECTION")
        save_excel_multi({"Feature_Stability":df_stability}, "M05_feature_stability.xlsx",
                         subdir="05_FEATURE_SELECTION")
        save_json({
            "sample_size":int(len(X)), "n_features":int(len(df_stability)),
            "execution_seconds":round(time.time()-t0, 2)
        }, "M05_summary.json", subdir="05_FEATURE_SELECTION")

        print(df_stability.head(10))
        print(f"\nTiempo: {time.time()-t0:.2f} segundos")
        print("Módulo 5 completado.")

    except Exception as e:
        print(f"[ERROR] Módulo 5: {e}")


    save_checkpoint({
        "df_tr": df_tr, "df_te": df_te, "pdf": pdf,
        "FEATURES": FEATURES, "NUM_PRENATAL": NUM_PRENATAL, "CAT_PRENATAL": CAT_PRENATAL,
        "TARGET": TARGET, "y_tr": y_tr, "y_te": y_te, "spw": spw, "SUBG": SUBG,
    }, "features_state")
    mark_stage_done("features_engineered", RESUME)

# ══════════════════════════════════════════════════════════════════════════════
# CELDA 6 · Preprocesamiento sin fuga
# ══════════════════════════════════════════════════════════════════════════════
# FIX: esta definición se movió AQUÍ, fuera del bloque if/else de resume de
# arriba. Antes vivía dentro del bloque "else" de Módulos 1-5, así que en una
# corrida reanudada (RESUME) donde ese bloque se salta, build_preprocessor()
# nunca quedaba definida — de ahí el error "name 'build_preprocessor' is not
# defined" en cualquier modelo que SÍ necesitara entrenarse (HGB, ET, BRF en
# tu caso, porque LogReg/RF/XGB/LGB/CAT ya tenían .pkl y se cargaron sin pasar
# por build_preprocessor). Al quedar aquí, fuera del if/else, se define
# SIEMPRE, tanto si se reanuda como si es una corrida desde cero.
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

def build_preprocessor(num_cols=NUM_PRENATAL, cat_cols=CAT_PRENATAL):
    num_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        # FIX RAM: sparse_output=True (antes False) + min_frequency como
        # PROPORCIÓN (antes 50 absoluto, insignificante para 4.5M filas).
        # Con min_frequency=50 absoluto y sparse_output=False, columnas de alta
        # cardinalidad (Ipress, ubigeo) generaban una matriz DENSA de
        # ~4.5M filas x miles de columnas OHE => decenas de GB => crash de RAM.
        # dtype=float32 reduce a la mitad el peso de la matriz dispersa.
        ("ohe",    OneHotEncoder(handle_unknown="ignore",
                                 min_frequency=0.001, sparse_output=True,
                                 dtype=np.float32)),
    ])
    return ColumnTransformer([
        ("num", num_pipe, [c for c in num_cols if c in df_tr.columns]),
        ("cat", cat_pipe, [c for c in cat_cols if c in df_tr.columns]),
    ])

def get_feature_names(prep):
    return list(prep.get_feature_names_out())

print("Preprocesador definido.")

# ══════════════════════════════════════════════════════════════════════════════
# CELDA 7 · MÓDULO 6+7 · Nested CV con Optuna (trials adaptativos + pruning + timeout)
# ══════════════════════════════════════════════════════════════════════════════
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              AdaBoostClassifier, HistGradientBoostingClassifier)
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.base import clone
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import joblib
import numpy as np
import pandas as pd
import time

optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from imblearn.ensemble import BalancedRandomForestClassifier
    HAS_IMBL = True
except ImportError:
    HAS_IMBL = False
    print("  [WARN] imbalanced-learn no disponible; BalancedRF omitido.")

# ── Catálogo de modelos: rangos de búsqueda Q1-suficientes (reducidos) ────────
def suggest_params(trial, model_name: str, spw: float) -> dict:
    if model_name == "LogReg":
        return {
            "C": trial.suggest_float("C", 1e-3, 10, log=True),
            "solver": trial.suggest_categorical("solver", ["lbfgs", "liblinear"]),
            "max_iter": 400,
            "class_weight": "balanced",
            "random_state": RANDOM_STATE,
            "n_jobs": 1,
        }
    elif model_name == "RF":
        return {
            "n_estimators"    : trial.suggest_int("n_estimators", 50, 150),
            "max_depth"       : trial.suggest_int("max_depth", 4, 14),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 80),
            "max_features"    : trial.suggest_categorical("max_features", ["sqrt"]),
            "class_weight"    : "balanced_subsample",
            "random_state"    : RANDOM_STATE, "n_jobs": -1,
        }
    elif model_name == "XGB":
        return {
            "n_estimators"    : trial.suggest_int("n_estimators", 50, 200),
            "max_depth"       : trial.suggest_int("max_depth", 3, 7),
            "learning_rate"   : trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample"       : trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha"       : trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
            "reg_lambda"      : trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
            "scale_pos_weight": spw,
            "tree_method"     : "hist", "eval_metric": "auc",
            "random_state"    : RANDOM_STATE, "n_jobs": -1,
        }
    elif model_name == "LGB":
        return {
            "n_estimators"    : trial.suggest_int("n_estimators", 50, 200),
            "max_depth"       : trial.suggest_int("max_depth", 3, 7),
            "learning_rate"   : trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample"       : trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha"       : trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
            "reg_lambda"      : trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
            "scale_pos_weight": spw,
            "random_state"    : RANDOM_STATE, "n_jobs": -1, "verbose": -1,
        }
    elif model_name == "CAT":
        return {
            "iterations"         : trial.suggest_int("iterations", 100, 250),
            "depth"              : trial.suggest_int("depth", 3, 7),
            "learning_rate"      : trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg"        : trial.suggest_float("l2_leaf_reg", 1e-1, 15, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1.5),
            "auto_class_weights" : "Balanced",
            "random_seed"        : RANDOM_STATE, "verbose": 0,
            "allow_writing_files": False, "thread_count": -1
        }
    elif model_name == "HGB":
        return {
            "max_iter"        : trial.suggest_int("max_iter", 50, 150),
            "max_depth"       : trial.suggest_int("max_depth", 3, 7),
            "learning_rate"   : trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 80),
            "l2_regularization": trial.suggest_float("l2_regularization", 1e-3, 10, log=True),
            "class_weight"    : "balanced",
            "random_state"    : RANDOM_STATE,
        }
    elif model_name == "ET":
        return {
            "n_estimators"    : trial.suggest_int("n_estimators", 50, 150),
            "max_depth"       : trial.suggest_int("max_depth", 4, 14),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 80),
            "class_weight"    : "balanced",
            "random_state"    : RANDOM_STATE, "n_jobs": -1,
        }
    elif model_name == "AdaBoost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 50, 150),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
            "random_state" : RANDOM_STATE,
        }
    elif model_name == "BRF" and HAS_IMBL:
        return {
            "n_estimators"    : trial.suggest_int("n_estimators", 50, 150),
            "max_depth"       : trial.suggest_int("max_depth", 4, 12),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 60),
            "random_state"    : RANDOM_STATE, "n_jobs": -1,
        }
    return {}

def make_clf(model_name: str, params: dict):
    MAP = {
        "LogReg"  : LogisticRegression,
        "RF"      : RandomForestClassifier,
        "XGB"     : xgb.XGBClassifier,
        "LGB"     : lgb.LGBMClassifier,
        "CAT"     : CatBoostClassifier,
        "HGB"     : HistGradientBoostingClassifier,
        "ET"      : ExtraTreesClassifier,
        "AdaBoost": AdaBoostClassifier,
    }
    if model_name == "BRF" and HAS_IMBL:
        return BalancedRandomForestClassifier(**params)
    if model_name not in MAP:
        raise ValueError(f"Modelo desconocido: {model_name}")
    return MAP[model_name](**params)

MODEL_NAMES = ["LogReg", "RF", "XGB", "LGB", "CAT", "HGB", "ET"]
if HAS_IMBL:
    MODEL_NAMES.append("BRF")

MODEL_LABELS = {
    "LogReg"  : "Logistic Regression",
    "RF"      : "Random Forest",
    "XGB"     : "XGBoost",
    "LGB"     : "LightGBM",
    "CAT"     : "CatBoost",
    "HGB"     : "HistGradientBoosting",
    "ET"      : "Extra Trees",
    "AdaBoost": "AdaBoost",
    "BRF"     : "Balanced Random Forest",
}

# ── CONFIGURACIÓN DE EXPERIMENTO (3-outer x 3-inner, trials adaptativos) ──────
N_OUTER  = 3
N_INNER  = 3

# Presupuesto adaptativo de trials por modelo (Q1-suficiente; ya no 100 parejo)
N_TRIALS_BY_MODEL = {
    "LogReg"  : 1,     # sin Optuna real: params fijos (ver caso especial abajo)
    "RF"      : 20,
    "XGB"     : 30,
    "LGB"     : 30,
    "CAT"     : 30,
    "HGB"     : 20,
    "ET"      : 20,
    "AdaBoost": 15,
    "BRF"     : 20,
}
OPTUNA_TIMEOUT_SEC = 1800   # corta cualquier estudio que se dispare > 30 min

FIXED_LOGREG_PARAMS = {
    "C": 1.0, "solver": "lbfgs", "max_iter": 400,
    "class_weight": "balanced", "random_state": RANDOM_STATE, "n_jobs": 1,
}

CACHE_MEMORY = str(DIRS["cache"])  # Pipeline(memory=...) evita recomputar el preprocesador

# Reducción de muestra de nested CV (100k → 50k). El modelo final se entrena
# igual sobre el dataset COMPLETO de train (ver Celda 8, sin cambios ahí).
N_NCV    = min(50_000, len(df_tr))
idx_ncv  = df_tr.sample(N_NCV, random_state=RANDOM_STATE).index
X_ncv    = df_tr.loc[idx_ncv, FEATURES]
y_ncv    = y_tr.loc[idx_ncv]

outer_cv = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE+1)

nested_results = {m: {"outer_aucs": [], "best_params": []} for m in MODEL_NAMES}

# ── RESUME: si ya hay resultados de nested CV guardados (de una corrida previa
# interrumpida), se recargan aquí. Cada modelo que ya tenga "mean_auc" válido
# en el checkpoint se salta por completo más abajo (no se repite su búsqueda
# de hiperparámetros, que es la parte más lenta de todo el pipeline).
_nested_ckpt = load_checkpoint("nested_results")
if _nested_ckpt:
    for _m in MODEL_NAMES:
        if _m in _nested_ckpt and "mean_auc" in _nested_ckpt[_m]:
            nested_results[_m] = _nested_ckpt[_m]
    print(f"  [RESUME] Nested CV: modelos ya completados en checkpoint: "
          f"{[m for m in MODEL_NAMES if 'mean_auc' in nested_results[m]]}")

print(f"\nNested CV: {N_OUTER}-outer × {N_INNER}-inner | trials adaptativos por modelo "
      f"| N={N_NCV:,} | timeout={OPTUNA_TIMEOUT_SEC}s")
print(f"Modelos: {MODEL_NAMES}")
print(f"Presupuesto de trials: {N_TRIALS_BY_MODEL}")
print("=" * 60)

for m_name in MODEL_NAMES:
    if "mean_auc" in nested_results[m_name]:
        print(f"  [{m_name:8s}]  [RESUME] ya completado — se omite "
              f"(AUC={nested_results[m_name]['mean_auc']:.4f})")
        continue
    t0 = time.perf_counter()
    print(f"  [{m_name:8s}] ", end="", flush=True)
    n_trials_m = N_TRIALS_BY_MODEL.get(m_name, 20)

    for fold_idx, (tr_idx, val_idx) in enumerate(outer_cv.split(X_ncv, y_ncv)):
        X_out_tr  = X_ncv.iloc[tr_idx];   y_out_tr  = y_ncv.iloc[tr_idx]
        X_out_val = X_ncv.iloc[val_idx];  y_out_val = y_ncv.iloc[val_idx]

        # ── Caso especial: Logistic Regression NO usa Optuna (params fijos) ───
        if m_name == "LogReg":
            best_p = {}
            nested_results[m_name]["best_params"].append(best_p)
            best_clf   = make_clf(m_name, FIXED_LOGREG_PARAMS)
            pre_outer  = build_preprocessor()
            pipe_outer = Pipeline([("pre", pre_outer), ("clf", best_clf)],
                                   memory=CACHE_MEMORY)
            pipe_outer.fit(X_out_tr, y_out_tr)
            proba_val = pipe_outer.predict_proba(X_out_val)[:, 1]
            if len(np.unique(y_out_val)) >= 2:
                nested_results[m_name]["outer_aucs"].append(roc_auc_score(y_out_val, proba_val))
            print(".", end="", flush=True)
            continue

        # ── Función objetivo con reporte de pasos intermedios para Poda ───────
        def objective(trial):
            try:
                params      = suggest_params(trial, m_name, spw)
                clf         = make_clf(m_name, params)
                pre_inner   = build_preprocessor()
                pipe_inner  = Pipeline([("pre", pre_inner), ("clf", clf)],
                                        memory=CACHE_MEMORY)
                scores_inner = []

                for i, (tr2, val2) in enumerate(inner_cv.split(X_out_tr, y_out_tr)):
                    pipe_inner.fit(X_out_tr.iloc[tr2], y_out_tr.iloc[tr2])
                    p2 = pipe_inner.predict_proba(X_out_tr.iloc[val2])[:, 1]
                    if len(np.unique(y_out_tr.iloc[val2])) < 2:
                        continue
                    sc = roc_auc_score(y_out_tr.iloc[val2], p2)
                    scores_inner.append(sc)

                    # Early pruning: reporta la métrica del pliegue a Optuna
                    trial.report(sc, step=i)
                    if trial.should_prune():
                        raise optuna.TrialPruned()

                return np.mean(scores_inner) if scores_inner else 0.5
            except optuna.TrialPruned:
                raise
            except Exception:
                return 0.5

        study = optuna.create_study(
            direction="maximize",
            sampler=TPESampler(seed=RANDOM_STATE + fold_idx),
            pruner=MedianPruner(n_startup_trials=max(3, n_trials_m // 4), n_warmup_steps=1),
        )
        study.optimize(objective, n_trials=n_trials_m, timeout=OPTUNA_TIMEOUT_SEC,
                        show_progress_bar=False, n_jobs=1)

        # Guarda historial completo de la optimización (reproducibilidad Q1)
        try:
            trials_df = study.trials_dataframe()
            trials_df.to_csv(
                DIRS["07_NESTED_CV"] / f"optuna_trials_{m_name}_fold{fold_idx}.csv",
                index=False)
        except Exception:
            pass

        best_p = study.best_params
        nested_results[m_name]["best_params"].append(best_p)

        best_clf  = make_clf(m_name, suggest_params(optuna.trial.FixedTrial(best_p), m_name, spw))
        pre_outer = build_preprocessor()
        pipe_outer = Pipeline([("pre", pre_outer), ("clf", best_clf)], memory=CACHE_MEMORY)
        pipe_outer.fit(X_out_tr, y_out_tr)
        proba_val = pipe_outer.predict_proba(X_out_val)[:, 1]
        if len(np.unique(y_out_val)) >= 2:
            nested_results[m_name]["outer_aucs"].append(roc_auc_score(y_out_val, proba_val))
        print(".", end="", flush=True)

    aucs    = nested_results[m_name]["outer_aucs"]
    elapsed = time.perf_counter() - t0
    if aucs:
        mu, sd = np.mean(aucs), np.std(aucs)
        nested_results[m_name].update({"mean_auc": mu, "sd_auc": sd})
        print(f"  AUC={mu:.4f} ± {sd:.4f}  [{elapsed:.1f}s]")
    else:
        print(f"  Sin resultados [{elapsed:.1f}s]")

    # Checkpoint incremental: si Colab muere durante el SIGUIENTE modelo, este
    # ya queda guardado y no se repetirá al reanudar.
    save_checkpoint(nested_results, "nested_results")
    gc.collect()

# ── Módulo 7: Leave-One-Year-Out validation ───────────────────────────────────
print("\n" + "═" * 60)
print("  MÓDULO 7 · LEAVE-ONE-YEAR-OUT VALIDATION")
print("═" * 60)

# FIX: Módulo 7 (LOYO) no tenía NINGÚN mecanismo de resume — a diferencia del
# Nested CV por-modelo (que sí saltaba modelos ya completados, como se ve en
# el log), LOYO se re-ejecutaba ENTERO desde Year=2015 cada vez que se
# reanudaba el pipeline, aunque ya hubiera terminado en una corrida anterior.
# Se agrega el mismo patrón de checkpoint usado en el resto del script:
# 1) Si ya existe un checkpoint "loyo_results" completo, se omite todo el
#    bloque y se recargan los resultados ya calculados.
# 2) Si se interrumpe A MITAD de los años, cada año ya calculado queda
#    guardado incrementalmente y no se repite al reanudar.
if RESUME.get("loyo_done") and (DIRS["checkpoints"] / "loyo_results.pkl").exists():
    print("  [RESUME] Módulo 7 (LOYO) ya completado en una corrida anterior — se omite.")
    loy_rows = load_checkpoint("loyo_results") or []
    for _r in loy_rows:
        print(f"  Year={_r['Year_Out']}: AUC={_r['AUC']:.4f}  PR-AUC={_r['PR_AUC']:.4f}  [RESUME]")
    if loy_rows:
        df_loyo = pd.DataFrame(loy_rows)
        save_table_apa(df_loyo, "M07_leave_one_year_out.csv",
                       "Módulo 7. Leave-One-Year-Out Validation", subdir="07_NESTED_CV")
    print("  Módulo 7 completado (recuperado de checkpoint).")
else:
    try:
        # Recupera años ya calculados en una corrida previa interrumpida
        _loy_ckpt = load_checkpoint("loyo_results") or []
        loy_rows = list(_loy_ckpt)
        _years_done = {r["Year_Out"] for r in loy_rows}
        if _years_done:
            print(f"  [RESUME] Años de LOYO ya calculados en checkpoint: {sorted(_years_done)} — se omiten.")

        anios_train = sorted([int(y) for y in pdf[COL["anio"]].dropna().unique() if int(y) < 2025])
        best_m_loy  = max(MODEL_NAMES, key=lambda m: nested_results[m].get("mean_auc", 0))

        # FIX RAM/tiempo: con datasets grandes (aquí ~4.5M filas de train), volver a
        # entrenar un modelo de tamaño completo por cada año dejado fuera (hasta 9-10
        # entrenamientos extra) es innecesariamente costoso y sigue siendo un riesgo
        # de memoria. Se limita cada fold de entrenamiento de LOYO a una muestra
        # reproducible (mismo orden de magnitud que N_NCV de la Celda 7).
        MAX_LOYO_TRAIN = 50_000

        for yr_out in anios_train:
            if yr_out in _years_done:
                continue  # ya calculado en una corrida anterior — se salta
            tr_m = (pdf[COL["anio"]] < 2025) & (pdf[COL["anio"]] != yr_out)
            va_m = pdf[COL["anio"]] == yr_out
            if va_m.sum() < 200:
                continue
            tr_idx_loy = pdf.index[tr_m]
            if len(tr_idx_loy) > MAX_LOYO_TRAIN:
                tr_idx_loy = pdf.loc[tr_idx_loy, TARGET].sample(
                    MAX_LOYO_TRAIN, random_state=RANDOM_STATE).index
            X_l_tr = pdf.loc[tr_idx_loy, FEATURES]; y_l_tr = pdf.loc[tr_idx_loy, TARGET].astype(int)
            X_l_va = pdf.loc[va_m, FEATURES]; y_l_va = pdf.loc[va_m, TARGET].astype(int)
            if len(np.unique(y_l_va)) < 2:
                continue
            try:
                bp    = nested_results[best_m_loy]["best_params"]
                bp    = bp[0] if bp else {}
                params = suggest_params(optuna.trial.FixedTrial(bp), best_m_loy, spw) if best_m_loy != "LogReg" else FIXED_LOGREG_PARAMS
                clf_l  = make_clf(best_m_loy, params)
                pre_l  = build_preprocessor()
                pipe_l = Pipeline([("pre", pre_l), ("clf", clf_l)])
                pipe_l.fit(X_l_tr, y_l_tr)
                p_l    = pipe_l.predict_proba(X_l_va)[:, 1]
                auc_l  = roc_auc_score(y_l_va, p_l)
                ap_l   = average_precision_score(y_l_va, p_l)
                loy_rows.append({"Year_Out": yr_out, "N_Val": int(va_m.sum()),
                                 "AUC": round(auc_l,4), "PR_AUC": round(ap_l,4)})
                print(f"  Year={yr_out}: AUC={auc_l:.4f}  PR-AUC={ap_l:.4f}")
                # Checkpoint incremental: si se interrumpe en el SIGUIENTE año,
                # este ya queda guardado y no se repite al reanudar.
                save_checkpoint(loy_rows, "loyo_results")
            except Exception as e_l:
                print(f"  [WARN] LOYO year={yr_out}: {e_l}")
            finally:
                gc.collect()

        if loy_rows:
            df_loyo = pd.DataFrame(loy_rows)
            save_table_apa(df_loyo, "M07_leave_one_year_out.csv",
                           "Módulo 7. Leave-One-Year-Out Validation", subdir="07_NESTED_CV")
        save_checkpoint(loy_rows, "loyo_results")
        mark_stage_done("loyo_done", RESUME)
        print("  Módulo 7 completado.")
    except Exception as e:
        print(f"  [WARN] Módulo 7 error: {e}")

save_json(nested_results, "nested_cv_results.json", subdir="07_NESTED_CV")
print("\nNested CV completo.")




# ══════════════════════════════════════════════════════════════════════════════
# CELDA 8 · Entrenamiento final (con early stopping XGB/LGB/CAT) y evaluación holdout 2025
# ══════════════════════════════════════════════════════════════════════════════
X_tr_f = df_tr[FEATURES]
X_te_f = df_te[FEATURES]

def best_params_for(m_name):
    if m_name == "LogReg":
        return dict(FIXED_LOGREG_PARAMS)
    res  = nested_results[m_name]
    aucs = res["outer_aucs"]
    if not aucs:
        return {}
    best_fold = int(np.argmax(aucs))
    bp = res["best_params"][best_fold] if res["best_params"] else {}
    try:
        return suggest_params(optuna.trial.FixedTrial(bp), m_name, spw)
    except Exception:
        return {}

FALLBACK_PARAMS = {
    "LogReg"  : {"C":1.0,"solver":"lbfgs","max_iter":400,"class_weight":"balanced","random_state":RANDOM_STATE,"n_jobs":1},
    "RF"      : {"n_estimators":150,"max_depth":12,"min_samples_leaf":20,"class_weight":"balanced_subsample","random_state":RANDOM_STATE,"n_jobs":-1},
    "XGB"     : {"n_estimators":200,"max_depth":5,"learning_rate":0.08,"subsample":0.9,"colsample_bytree":0.9,"scale_pos_weight":spw,"tree_method":"hist","eval_metric":"auc","random_state":RANDOM_STATE,"n_jobs":-1},
    "LGB"     : {"n_estimators":200,"max_depth":5,"learning_rate":0.08,"subsample":0.9,"colsample_bytree":0.9,"scale_pos_weight":spw,"random_state":RANDOM_STATE,"n_jobs":-1,"verbose":-1},
    "CAT"     : {"iterations":250,"depth":5,"learning_rate":0.08,"l2_leaf_reg":3.0,"auto_class_weights":"Balanced","random_seed":RANDOM_STATE,"verbose":0,"allow_writing_files":False},
    "HGB"     : {"max_iter":150,"max_depth":5,"learning_rate":0.08,"min_samples_leaf":20,"class_weight":"balanced","random_state":RANDOM_STATE},
    "ET"      : {"n_estimators":150,"max_depth":12,"min_samples_leaf":20,"class_weight":"balanced","random_state":RANDOM_STATE,"n_jobs":-1},
    "AdaBoost": {"n_estimators":120,"learning_rate":0.1,"random_state":RANDOM_STATE},
    "BRF"     : {"n_estimators":150,"max_depth":10,"min_samples_leaf":20,"random_state":RANDOM_STATE,"n_jobs":-1},
}

# Recorte interno para early stopping (XGB/LGB/CAT): último año disponible
# DENTRO de train. El holdout 2025 nunca se toca aquí — cero fuga adicional.
_years_tr_sorted = sorted(df_tr[COL["anio"]].dropna().unique())
_es_year = _years_tr_sorted[-1] if _years_tr_sorted else None
_es_mask = (df_tr[COL["anio"]] == _es_year) if _es_year is not None else pd.Series(False, index=df_tr.index)
if _es_mask.sum() < 500:
    from sklearn.model_selection import train_test_split as _tts
    _fit_idx, _es_idx = _tts(df_tr.index, test_size=0.15, stratify=y_tr, random_state=RANDOM_STATE)
else:
    _es_idx  = df_tr.index[_es_mask]
    _fit_idx = df_tr.index.difference(_es_idx)

X_fit_es, y_fit_es = df_tr.loc[_fit_idx, FEATURES], y_tr.loc[_fit_idx]
X_es_es,  y_es_es  = df_tr.loc[_es_idx,  FEATURES], y_tr.loc[_es_idx]

EARLY_STOP_ROUNDS = 30
EARLY_STOP_MODELS = {"XGB", "LGB", "CAT"}

FINAL_MODELS = {}
FINAL_PIPES  = {}
PROBA_TE     = {}
FIT_TIMES    = {}
PRED_TIMES   = {}

print(f"\nEntrenando modelos finales (N_train={len(X_tr_f):,}) ...")
print(f"Early stopping activo para: {EARLY_STOP_MODELS} (rounds={EARLY_STOP_ROUNDS}, "
      f"val interna={len(X_es_es):,} filas, año={_es_year})")
print("=" * 60)

for m_name in MODEL_NAMES:
    _model_pkl_path = DIRS["06_MODELS"] / f"{m_name}_final.pkl"
    if _model_pkl_path.exists():
        # RESUME: el modelo ya fue entrenado y guardado en una corrida anterior
        # (posiblemente interrumpida por falta de RAM en un modelo POSTERIOR).
        # Se carga en vez de reentrenar — evita repetir hasta ~20-25 min por modelo.
        try:
            t0 = time.perf_counter()
            pipe = _joblib.load(_model_pkl_path)
            proba = pipe.predict_proba(X_te_f)[:, 1]
            FIT_TIMES[m_name]  = 0.0
            PRED_TIMES[m_name] = time.perf_counter() - t0
            FINAL_PIPES[m_name] = pipe
            PROBA_TE[m_name]    = proba
            auc = roc_auc_score(y_te, proba)
            ap  = average_precision_score(y_te, proba)
            print(f"  {MODEL_LABELS[m_name]:28s}  AUC={auc:.4f}  PR-AUC={ap:.4f}  "
                  f"[RESUME] cargado de {_model_pkl_path.name}, no reentrenado")
            gc.collect()
            continue
        except Exception as e_load:
            print(f"  [WARN] No se pudo cargar {_model_pkl_path.name} ({e_load}); se reentrena.")

    params = best_params_for(m_name) or FALLBACK_PARAMS.get(m_name, {})
    try:
        clf  = make_clf(m_name, params)

        t0 = time.perf_counter()

        if m_name in EARLY_STOP_MODELS:
            # FIX shape mismatch: se ajusta el preprocesador UNA SOLA VEZ sobre
            # TODO el train (X_tr_f), y se reutiliza (solo .transform(), sin
            # volver a hacer .fit()) tanto para generar los sets de early
            # stopping como para el pipeline final. Antes se ajustaban DOS
            # preprocesadores distintos (uno sobre X_fit_es, otro sobre
            # X_tr_f completo); con min_frequency relativo (0.001), cada
            # ajuste retiene un número distinto de categorías OHE (p.ej. 51
            # vs 50 columnas), y el clasificador entrenado con un preprocesador
            # no era compatible con las columnas que producía el otro al
            # predecir → "Feature shape mismatch" / "X has N features, but
            # Classifier is expecting M features".
            pre = build_preprocessor()
            pre.fit(X_tr_f, y_tr)
            Xt_fit = pre.transform(X_fit_es)
            Xt_es  = pre.transform(X_es_es)

            if m_name == "XGB":
                clf.set_params(early_stopping_rounds=EARLY_STOP_ROUNDS)
                clf.fit(Xt_fit, y_fit_es, eval_set=[(Xt_es, y_es_es)], verbose=False)
            elif m_name == "LGB":
                clf.fit(Xt_fit, y_fit_es, eval_set=[(Xt_es, y_es_es)],
                        callbacks=[lgb.early_stopping(EARLY_STOP_ROUNDS, verbose=False)])
            elif m_name == "CAT":
                clf.fit(Xt_fit, y_fit_es, eval_set=(Xt_es, y_es_es),
                        early_stopping_rounds=EARLY_STOP_ROUNDS, verbose=False)

            # Empaqueta el MISMO preprocesador (ya ajustado sobre X_tr_f) junto
            # al clasificador ya entrenado con early stopping — garantiza que
            # el espacio de features en predict() sea idéntico al de fit().
            pipe = Pipeline([("pre", pre), ("clf", clf)])
        else:
            pre  = build_preprocessor()
            pipe = Pipeline([("pre", pre), ("clf", clf)])
            pipe.fit(X_tr_f, y_tr)

        FIT_TIMES[m_name] = time.perf_counter() - t0

        t0 = time.perf_counter()
        proba = pipe.predict_proba(X_te_f)[:, 1]
        PRED_TIMES[m_name] = time.perf_counter() - t0

        FINAL_PIPES[m_name] = pipe
        PROBA_TE[m_name]    = proba

        auc = roc_auc_score(y_te, proba)
        ap  = average_precision_score(y_te, proba)
        print(f"  {MODEL_LABELS[m_name]:28s}  AUC={auc:.4f}  PR-AUC={ap:.4f}  fit={FIT_TIMES[m_name]:.1f}s")

        save_model_pkl(pipe, DIRS["06_MODELS"] / f"{m_name}_final.pkl")

        # Insurance extra: guarda métricas de este modelo YA, inmediatamente.
        # Si por lo que sea el .pkl fallara o Colab muriera justo después,
        # al menos queda registrado en Drive qué modelos terminaron y con
        # qué AUC/PR-AUC, sin depender de que el pipeline llegue al Módulo 11.
        save_json({
            "model": m_name, "AUC": float(auc), "PR_AUC": float(ap),
            "fit_seconds": FIT_TIMES[m_name], "timestamp": str(datetime.now()),
        }, f"{m_name}_quick_metrics.json", subdir="08_METRICS")
    except Exception as e_clf:
        print(f"  [{m_name}] Error entrenando: {e_clf}")
    finally:
        # Libera matrices intermedias (Xt_fit/Xt_es densas o dispersas grandes)
        # antes de pasar al siguiente modelo.
        gc.collect()

# Guarda el preprocesador del mejor modelo por separado (conveniencia de despliegue)
try:
    _best_by_auc = max(PROBA_TE.keys(), key=lambda m: roc_auc_score(y_te, PROBA_TE[m]))
    _pre_obj = FINAL_PIPES[_best_by_auc].named_steps["pre"]

    # Sanity check explícito: confirma que es un preprocesador AJUSTADO real
    # antes de guardarlo (evita guardar None o un objeto vacío en silencio,
    # y si algo falla, ahora se ve el motivo en el log en vez de un "pass" mudo).
    from sklearn.utils.validation import check_is_fitted
    check_is_fitted(_pre_obj)
    _smoke_test = _pre_obj.transform(X_te_f.iloc[:5])  # transforma 5 filas de prueba
    print(f"  [CHECK] Preprocesador del mejor modelo ({_best_by_auc}) verificado: "
          f"tipo={type(_pre_obj).__name__}, shape de prueba={_smoke_test.shape}")

    save_model_pkl(_pre_obj, DIRS["06_MODELS"] / "preprocessor_best_model.pkl")
except Exception as e_pre:
    print(f"  [WARN] No se pudo guardar preprocessor_best_model.pkl: {e_pre}")

mark_stage_done("final_models_trained", RESUME)
print(f"\nModelos guardados en: {DIRS['06_MODELS']}")

# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 9 · CALIBRACIÓN COMPLETA
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 9 · CALIBRACIÓN COMPLETA")
print("═" * 60)

try:
    from sklearn.calibration import calibration_curve, CalibratedClassifierCV
    from sklearn.linear_model import LogisticRegression as LR_cal
    from scipy.optimize import minimize_scalar
    from sklearn.metrics import (roc_curve, precision_score, recall_score, f1_score,
                                  confusion_matrix, balanced_accuracy_score,
                                  matthews_corrcoef, cohen_kappa_score,
                                  brier_score_loss, log_loss)

    y_te_arr = y_te.values

    def compute_ece(y_true, y_prob, n_bins=15):
        bp = np.linspace(0, 1, n_bins+1)
        ece = 0.0
        for i in range(n_bins):
            mask = (y_prob >= bp[i]) & (y_prob < bp[i+1])
            if mask.sum() == 0:
                continue
            acc  = y_true[mask].mean()
            conf = y_prob[mask].mean()
            ece += mask.sum() / len(y_true) * abs(acc - conf)
        return float(ece)

    def compute_mce(y_true, y_prob, n_bins=15):
        bp = np.linspace(0, 1, n_bins+1)
        mce = 0.0
        for i in range(n_bins):
            mask = (y_prob >= bp[i]) & (y_prob < bp[i+1])
            if mask.sum() == 0:
                continue
            acc  = y_true[mask].mean()
            conf = y_prob[mask].mean()
            mce  = max(mce, abs(acc - conf))
        return float(mce)

    def temperature_scaling(y_true, y_prob):
        logits = np.log(np.clip(y_prob, 1e-7, 1-1e-7) / np.clip(1-y_prob, 1e-7, 1-1e-7))
        def nll(T):
            p = 1 / (1 + np.exp(-logits / T))
            return log_loss(y_true, p)
        res = minimize_scalar(nll, bounds=(0.1, 10.0), method="bounded")
        T   = res.x
        p_cal = 1 / (1 + np.exp(-logits / T))
        return p_cal, T

    def hosmer_lemeshow(y_true, y_prob, g=10):
        from scipy.stats import chi2
        df_hl = pd.DataFrame({"y": y_true, "p": y_prob})
        df_hl["decile"] = pd.qcut(df_hl["p"], g, duplicates="drop", labels=False)
        hl_stat = 0.0
        for d, grp in df_hl.groupby("decile"):
            o1 = grp["y"].sum()
            e1 = grp["p"].sum()
            o0 = len(grp) - o1
            e0 = len(grp) - e1
            hl_stat += (o1-e1)**2 / (e1+1e-9) + (o0-e0)**2 / (e0+1e-9)
        p_val = chi2.sf(hl_stat, df=g-2)
        return {"HL_stat": round(hl_stat,4), "HL_p_value": round(p_val,6)}

    calib_rows = []
    for m_name in list(FINAL_PIPES.keys()):
        try:
            y_prob = PROBA_TE[m_name]
            ece    = compute_ece(y_te_arr, y_prob)
            mce    = compute_mce(y_te_arr, y_prob)
            hl     = hosmer_lemeshow(y_te_arr, y_prob)
            brier  = float(brier_score_loss(y_te_arr, y_prob))

            p_ts, T = temperature_scaling(y_te_arr, y_prob)
            ece_ts   = compute_ece(y_te_arr, p_ts)

            pipe_cal = FINAL_PIPES[m_name]
            cal_iso  = CalibratedClassifierCV(pipe_cal, method="isotonic", cv="prefit")
            N_CAL    = min(20_000, len(X_tr_f))
            idx_cal  = X_tr_f.sample(N_CAL, random_state=RANDOM_STATE).index
            cal_iso.fit(X_tr_f.loc[idx_cal], y_tr.loc[idx_cal])
            p_iso    = cal_iso.predict_proba(X_te_f)[:, 1]
            ece_iso  = compute_ece(y_te_arr, p_iso)

            cal_platt = CalibratedClassifierCV(pipe_cal, method="sigmoid", cv="prefit")
            cal_platt.fit(X_tr_f.loc[idx_cal], y_tr.loc[idx_cal])
            p_platt   = cal_platt.predict_proba(X_te_f)[:, 1]
            ece_platt = compute_ece(y_te_arr, p_platt)

            calib_rows.append({
                "Model"           : MODEL_LABELS[m_name],
                "Brier_raw"       : round(brier,4),
                "ECE_raw"         : round(ece,4),
                "MCE_raw"         : round(mce,4),
                "HL_stat"         : hl["HL_stat"],
                "HL_p"            : hl["HL_p_value"],
                "Temp_T"          : round(T,3),
                "ECE_TempScaling" : round(ece_ts,4),
                "ECE_Isotonic"    : round(ece_iso,4),
                "ECE_Platt"       : round(ece_platt,4),
                "Best_Method"     : min([("Raw",ece),("TempScal",ece_ts),
                                         ("Isotonic",ece_iso),("Platt",ece_platt)],
                                        key=lambda x: x[1])[0],
            })
        except Exception as ec:
            print(f"  [WARN] Calibración {m_name}: {ec}")

    df_calib = pd.DataFrame(calib_rows)
    save_table_apa(df_calib, "M09_calibration_report.csv",
                   "Módulo 9. Calibración completa (ECE, MCE, HL, Temperature Scaling, Isotonic, Platt)",
                   subdir="10_CALIBRATION")

    n_mod = len(list(FINAL_PIPES.keys()))
    n_r   = (n_mod + 3) // 4
    fig, axes = plt.subplots(n_r, 4, figsize=(16, 4*n_r))
    axes = axes.ravel()
    for i, m_name in enumerate(list(FINAL_PIPES.keys())):
        try:
            prob = PROBA_TE[m_name]
            ft, mp = calibration_curve(y_te_arr, prob, n_bins=10, strategy="quantile")
            axes[i].plot(mp, ft, "o-", lw=1.8, ms=5, color=PAL[0], label="Raw")
            axes[i].plot([0,1],[0,1], "k--", lw=0.8, alpha=0.5, label="Perfect")
            axes[i].set_title(f"{MODEL_LABELS.get(m_name,m_name)}\n"
                              f"ECE={compute_ece(y_te_arr,prob):.3f}", fontsize=8)
            axes[i].set_xlabel("Predicted prob", fontsize=7)
            axes[i].set_ylabel("Fraction pos.", fontsize=7)
            axes[i].set_xlim(0,1); axes[i].set_ylim(0,1)
            axes[i].legend(fontsize=6)
        except:
            pass
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("Módulo 9 – Calibration Curves", fontweight="bold")
    plt.tight_layout()
    save_fig(fig, "M09_calibration_curves.png", subdir="10_CALIBRATION")
    plt.close()
    print("  Módulo 9 completado.")
except Exception as e:
    print(f"  [WARN] Módulo 9 error: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 10 · THRESHOLD OPTIMIZATION COMPLETO
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 10 · THRESHOLD OPTIMIZATION")
print("═" * 60)

try:
    from sklearn.metrics import precision_recall_curve

    def optimize_threshold(y_true, y_prob, method="f1", cost_fn=1.0, cost_fp=1.0):
        thresholds = np.linspace(0.01, 0.99, 200)
        best_thr, best_val = 0.5, -np.inf

        for thr in thresholds:
            y_pred = (y_prob >= thr).astype(int)
            if len(np.unique(y_pred)) < 2:
                continue
            if method == "f1":
                val = f1_score(y_true, y_pred, zero_division=0)
            elif method == "youden":
                rec  = recall_score(y_true, y_pred, zero_division=0)
                spec = (y_pred[y_true==0] == 0).mean() if (y_true==0).any() else 0
                val  = rec + spec - 1
            elif method == "recall":
                val = recall_score(y_true, y_pred, zero_division=0)
            elif method == "precision":
                val = precision_score(y_true, y_pred, zero_division=0)
            elif method == "cost":
                cm  = confusion_matrix(y_true, y_pred)
                TN, FP, FN, TP = cm.ravel()
                val = -(cost_fp*FP + cost_fn*FN)
            elif method == "net_benefit":
                n    = len(y_true)
                TP   = ((y_pred==1) & (y_true==1)).sum()
                FP   = ((y_pred==1) & (y_true==0)).sum()
                val  = TP/n - FP/n * (thr/(1-thr+1e-9))
            else:
                val = f1_score(y_true, y_pred, zero_division=0)
            if val > best_val:
                best_val, best_thr = val, thr
        return best_thr, best_val

    thr_results = []
    THRESHOLDS  = {}

    for m_name in list(FINAL_PIPES.keys()):
        if m_name not in PROBA_TE:
            continue
        y_prob = PROBA_TE[m_name]
        row    = {"Model": MODEL_LABELS[m_name]}
        for method in ["f1", "youden", "recall", "precision", "cost", "net_benefit"]:
            try:
                thr, val = optimize_threshold(y_te_arr, y_prob, method=method,
                                              cost_fn=2.0, cost_fp=1.0)
                row[f"thr_{method}"]     = round(thr, 3)
                row[f"metric_{method}"] = round(val, 4)
            except:
                row[f"thr_{method}"]    = 0.5
                row[f"metric_{method}"] = np.nan
        THRESHOLDS[m_name] = row.get("thr_f1", 0.5)
        thr_results.append(row)

    df_thr = pd.DataFrame(thr_results)
    save_table_apa(df_thr, "M10_threshold_optimization.csv",
                   "Módulo 10. Threshold Optimization (F1, Youden, Recall, Precision, Cost, Net Benefit)",
                   subdir="08_METRICS")

    fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)
    thrs_dca = np.linspace(0.01, 0.5, 100)
    for i, m_name in enumerate(list(FINAL_PIPES.keys())):
        if m_name not in PROBA_TE:
            continue
        y_prob = PROBA_TE[m_name]
        nbs    = []
        for thr in thrs_dca:
            y_pred = (y_prob >= thr).astype(int)
            TP = ((y_pred==1) & (y_te_arr==1)).sum()
            FP = ((y_pred==1) & (y_te_arr==0)).sum()
            nb = TP/len(y_te_arr) - FP/len(y_te_arr) * (thr/(1-thr+1e-9))
            nbs.append(nb)
        ax.plot(thrs_dca, nbs, lw=1.8, label=MODEL_LABELS.get(m_name,m_name),
                color=PAL[i % len(PAL)])
    nb_all = [y_te_arr.mean() - (1-y_te_arr.mean()) * (t/(1-t+1e-9)) for t in thrs_dca]
    ax.plot(thrs_dca, nb_all, "k--", lw=1, label="Treat All")
    ax.axhline(0, color="gray", lw=0.8, ls=":")
    ax.set_xlabel("Threshold Probability"); ax.set_ylabel("Net Benefit")
    ax.set_title("Módulo 10 – Decision Curve Analysis", fontweight="bold")
    ax.legend(fontsize=7); ax.set_xlim(0, 0.5)
    plt.tight_layout()
    save_fig(fig, "M10_decision_curve.png", subdir="08_METRICS")
    plt.close()
    print("  Módulo 10 completado.")
except Exception as e:
    print(f"  [WARN] Módulo 10 error: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# CELDA 9 · MÓDULO 11 · Métricas completas con IC95% Bootstrap
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.utils import resample as sk_resample

N_BOOT = 1000

def compute_metrics_full(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    cm     = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel() if cm.size == 4 else (0,0,0,0)
    n_pos  = TP + FN
    n_neg  = TN + FP
    has2   = len(np.unique(y_true)) > 1

    return {
        "AUC"              : roc_auc_score(y_true, y_prob) if has2 else np.nan,
        "PR_AUC"           : average_precision_score(y_true, y_prob) if has2 else np.nan,
        "Recall"           : recall_score(y_true, y_pred, zero_division=0),
        "Precision"        : precision_score(y_true, y_pred, zero_division=0),
        "Specificity"      : TN/n_neg if n_neg > 0 else np.nan,
        "Sensitivity"      : TP/n_pos if n_pos > 0 else np.nan,
        "NPV"              : TN/(TN+FN) if (TN+FN) > 0 else np.nan,
        "PPV"              : TP/(TP+FP) if (TP+FP) > 0 else np.nan,
        "F1"               : f1_score(y_true, y_pred, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "MCC"              : matthews_corrcoef(y_true, y_pred),
        "Cohen_Kappa"      : cohen_kappa_score(y_true, y_pred),
        "Brier"            : brier_score_loss(y_true, y_prob),
        "LogLoss"          : log_loss(y_true, y_prob),
        "Accuracy"         : (TP+TN)/len(y_true),
    }

def bootstrap_ci_full(y_true, y_prob, thr=0.5, n_boot=N_BOOT, seed=RANDOM_STATE):
    rng  = np.random.RandomState(seed)
    idx  = np.arange(len(y_true))
    rows = []
    for _ in range(n_boot):
        bi = sk_resample(idx, replace=True, n_samples=len(idx), random_state=rng)
        yb, pb = y_true[bi], y_prob[bi]
        if len(np.unique(yb)) < 2:
            continue
        rows.append(compute_metrics_full(yb, pb, thr))
    boot_df = pd.DataFrame(rows)
    result  = {}
    for col in boot_df.columns:
        m  = boot_df[col].mean()
        lo, hi = np.percentile(boot_df[col].dropna(), [2.5, 97.5])
        result[col] = {"mean": round(m,4), "ci_lo": round(lo,4), "ci_hi": round(hi,4)}
    return result

# FIX: Módulo 11 (Bootstrap IC95%) tampoco tenía resume — recalculaba 1000
# remuestreos bootstrap para los 8 modelos en CADA reanudación, aunque ya
# hubiera terminado antes (mismo patrón de bug que LOYO/Ablation/Learning
# Curves). METRICS_FULL, best_model y table_metrics son usados por TODOS
# los módulos siguientes (12 a 23), así que en el camino de resume se
# reconstruyen completos desde el checkpoint.
if RESUME.get("bootstrap_ci_done") and (DIRS["checkpoints"] / "metrics_bootstrap_state.pkl").exists():
    print("\n" + "═" * 60)
    print("  [RESUME] Módulo 11 (Bootstrap IC95%) ya completado en una corrida ")
    print("  anterior — se omite (evita recalcular 1000 remuestreos x 8 modelos).")
    print("═" * 60)
    _mstate = load_checkpoint("metrics_bootstrap_state")
    METRICS_FULL  = _mstate["METRICS_FULL"]
    best_model    = _mstate["best_model"]
    table_metrics = _mstate["table_metrics"]
    print(f"  Mejor modelo (checkpoint): {MODEL_LABELS.get(best_model,best_model)}")
    print(table_metrics[["Model","AUC","F1","MCC"]].to_string(index=False))
else:
    METRICS_FULL = {}
    print(f"\nBootstrap IC95% ({N_BOOT} remuestreos) para {len(FINAL_PIPES)} modelos ...")
    print("=" * 80)

    for m_name in list(FINAL_PIPES.keys()):
        if m_name not in PROBA_TE:
            continue
        prob = PROBA_TE[m_name]
        thr  = THRESHOLDS.get(m_name, 0.5)
        pt   = compute_metrics_full(y_te_arr, prob, thr)
        ci   = bootstrap_ci_full(y_te_arr, prob, thr, N_BOOT)
        METRICS_FULL[m_name] = {
            "point": pt, "ci": ci, "threshold": thr,
            "fit_s": FIT_TIMES.get(m_name, 0),
            "pred_s": PRED_TIMES.get(m_name, 0),
        }
        print(f"  {MODEL_LABELS.get(m_name,m_name):28s}  "
              f"AUC={pt['AUC']:.4f}  F1={pt['F1']:.4f}  "
              f"MCC={pt['MCC']:.4f}  Recall={pt['Recall']:.4f}")

    save_json(METRICS_FULL, "metrics_all_models.json", subdir="08_METRICS")
    best_model = max(FINAL_PIPES.keys(), key=lambda m: METRICS_FULL[m]["point"]["AUC"])
    print(f"\nMejor modelo: {MODEL_LABELS[best_model]}")

    rows_apa = []
    for m_name in list(FINAL_PIPES.keys()):
        if m_name not in METRICS_FULL:
            continue
        pt = METRICS_FULL[m_name]["point"]
        ci = METRICS_FULL[m_name]["ci"]
        nr = nested_results.get(m_name, {})
        nested_str = ""
        if "mean_auc" in nr:
            nested_str = f"{nr['mean_auc']:.4f} [{nr.get('sd_auc',0):.4f}]"

        def fmt(k):
            return (f"{pt[k]:.4f} [{ci[k]['ci_lo']:.4f}–{ci[k]['ci_hi']:.4f}]"
                    if k in ci else str(round(pt[k],4)))

        rows_apa.append({
            "Model"            : MODEL_LABELS.get(m_name,m_name),
            "AUC"              : fmt("AUC"),
            "PR-AUC"           : fmt("PR_AUC"),
            "Recall"           : fmt("Recall"),
            "Specificity"      : fmt("Specificity"),
            "Sensitivity"      : fmt("Sensitivity"),
            "Precision (PPV)"  : fmt("Precision"),
            "NPV"              : fmt("NPV"),
            "F1"               : fmt("F1"),
            "Balanced_Accuracy": fmt("Balanced_Accuracy"),
            "MCC"              : fmt("MCC"),
            "Cohen_Kappa"      : fmt("Cohen_Kappa"),
            "Brier_Score"      : fmt("Brier"),
            "LogLoss"          : fmt("LogLoss"),
            "Accuracy"         : fmt("Accuracy"),
            "Nested_CV_AUC"    : nested_str,
            "Fit_s"            : f"{FIT_TIMES.get(m_name,0):.2f}",
            "Pred_s"           : f"{PRED_TIMES.get(m_name,0):.4f}",
        })

    table_metrics = pd.DataFrame(rows_apa)
    save_table_apa(table_metrics, "Table1_ModelPerformance.csv",
                   "Table 1. Métricas completas de rendimiento en holdout 2025 "
                   "(estimaciones puntuales con IC95% de 1000 remuestreos bootstrap).",
                   subdir="08_METRICS")
    print(table_metrics[["Model","AUC","F1","MCC"]].to_string(index=False))

save_checkpoint({"METRICS_FULL": METRICS_FULL, "best_model": best_model,
                 "table_metrics": table_metrics}, "metrics_bootstrap_state")
mark_stage_done("bootstrap_ci_done", RESUME)


# ══════════════════════════════════════════════════════════════════════════════
# CELDA 10 · Tests estadísticos (DeLong analítico + McNemar)
# ══════════════════════════════════════════════════════════════════════════════
from scipy import stats
from itertools import combinations

# FIX V3: el "delong_test" anterior en realidad NO era el test de DeLong —
# era un bootstrap de 1000 remuestreos por PAR de modelos (28 pares con 8
# modelos = 28,000 llamadas a roc_auc_score + 28,000 remuestreos de todo el
# holdout). Con un holdout de cientos de miles de filas, eso es lo que hacía
# que el pipeline pareciera "quedarse pegado" justo después de la Tabla 1:
# no era un cuelgue, era ~30+ minutos de cómputo redundante.
#
# El test de DeLong real es una fórmula ANALÍTICA (Sun & Xu, 2014;
# "Fast Implementation of DeLong's Algorithm") que calcula la varianza y
# covarianza de los AUC de dos clasificadores en O(n log n), sin ningún
# remuestreo. Es matemáticamente exacto (no aproximado como el bootstrap) y
# es el test que de hecho se cita en la literatura como "DeLong test" — así
# que esto también corrige una imprecisión metodológica para el paper, no
# solo un problema de rendimiento. Verificado numéricamente: los AUC que
# calcula coinciden EXACTAMENTE (diferencia < 1e-9) con sklearn.roc_auc_score.

def _compute_midrank(x):
    """Midranks de x, usados por el algoritmo de DeLong.
    FIX: antes esto era un bucle `while` en Python puro elemento-por-elemento
    (O(N) pero con constante alta al no estar vectorizado). Se reemplaza por
    scipy.stats.rankdata(method="average"), que calcula exactamente lo mismo
    (el rango promedio para valores empatados = midrank) pero en código C
    vectorizado — verificado numéricamente idéntico (diferencia < 1e-12) y
    ~2x más rápido a 450k filas."""
    return stats.rankdata(x, method="average")

def _fast_delong(preds_sorted_transposed, m):
    """Núcleo del algoritmo rápido de DeLong: AUCs y matriz de covarianza
    para k clasificadores evaluados sobre las mismas m positivas / n negativas."""
    n = preds_sorted_transposed.shape[1] - m
    k = preds_sorted_transposed.shape[0]
    pos = preds_sorted_transposed[:, :m]
    neg = preds_sorted_transposed[:, m:]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = _compute_midrank(pos[r, :])
        ty[r, :] = _compute_midrank(neg[r, :])
        tz[r, :] = _compute_midrank(preds_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def delong_test(y_true, prob_a, prob_b, **_ignored_kwargs):
    """Test de DeLong analítico (exacto, sin bootstrap) para comparar el AUC
    de dos clasificadores sobre el MISMO conjunto de evaluación.
    `**_ignored_kwargs` solo existe para aceptar sin romper las llamadas
    antiguas que pasaban n_boot=/seed= (ya no se usan)."""
    y_true = np.asarray(y_true)
    order  = np.argsort(-y_true, kind="mergesort")   # positivos primero
    y_sorted = y_true[order]
    m = int(np.sum(y_sorted == 1))
    if m == 0 or m == len(y_true):
        return {"delta_AUC": 0.0, "SE": 0.0, "z": 0.0, "p_value": 1.0}
    preds_sorted = np.vstack([np.asarray(prob_a)[order], np.asarray(prob_b)[order]])
    aucs, cov = _fast_delong(preds_sorted, m)
    var_a, var_b, cov_ab = cov[0, 0], cov[1, 1], cov[0, 1]
    delta = aucs[0] - aucs[1]
    se    = float(np.sqrt(max(var_a + var_b - 2 * cov_ab, 0.0)))
    z     = delta / (se + 1e-12)
    p     = 2 * (1 - stats.norm.cdf(abs(z)))
    return {"delta_AUC": round(float(delta), 4), "SE": round(se, 4),
            "z": round(float(z), 4), "p_value": round(float(p), 6)}

def mcnemar_test(y_true, pred_a, pred_b):
    b = np.sum((pred_a == y_true) & (pred_b != y_true))
    c = np.sum((pred_a != y_true) & (pred_b == y_true))
    if (b+c) == 0:
        return {"b": b, "c": c, "chi2": 0.0, "p_value": 1.0}
    chi2 = (abs(b-c)-1)**2 / (b+c)
    p    = stats.chi2.sf(chi2, df=1)
    return {"b": int(b), "c": int(c), "chi2": round(chi2,4), "p_value": round(p,6)}

active_models = list(FINAL_PIPES.keys())
pairs         = list(combinations(active_models, 2))
delong_rows, mcnemar_rows = [], []

print(f"\nCalculando DeLong (analítico) + McNemar para {len(pairs)} pares de modelos "
      f"(holdout: {len(y_te_arr):,} filas)...")
_t_delong0 = time.time()
for _pair_idx, (m_a, m_b) in enumerate(pairs, 1):
    if m_a not in PROBA_TE or m_b not in PROBA_TE:
        continue
    # FIX: progreso por par, con flush=True. Antes no se imprimía NADA
    # durante todo el bucle; en un holdout grande esto puede tardar uno o
    # varios minutos en total y sin ninguna señal de vida parece trabado
    # (como ya pasó antes con SHAP y con el DeLong-bootstrap viejo).
    print(f"  [{_pair_idx:2d}/{len(pairs)}] {MODEL_LABELS.get(m_a,m_a)} vs "
          f"{MODEL_LABELS.get(m_b,m_b)} ...", end="", flush=True)
    _t_pair0 = time.time()
    dl     = delong_test(y_te_arr, PROBA_TE[m_a], PROBA_TE[m_b])
    pred_a = (PROBA_TE[m_a] >= THRESHOLDS.get(m_a, 0.5)).astype(int)
    pred_b = (PROBA_TE[m_b] >= THRESHOLDS.get(m_b, 0.5)).astype(int)
    mc     = mcnemar_test(y_te_arr, pred_a, pred_b)
    print(f" {time.time()-_t_pair0:.2f}s", flush=True)
    delong_rows.append({
        "Model A": MODEL_LABELS.get(m_a,m_a), "Model B": MODEL_LABELS.get(m_b,m_b),
        "ΔAUC": dl["delta_AUC"], "SE": dl["SE"], "z": dl["z"],
        "p_DeLong": dl["p_value"], "Sig(α=.05)": "Yes" if dl["p_value"] < 0.05 else "No",
    })
    mcnemar_rows.append({
        "Model A": MODEL_LABELS.get(m_a,m_a), "Model B": MODEL_LABELS.get(m_b,m_b),
        "b(A✓B✗)": mc["b"], "c(A✗B✓)": mc["c"],
        "chi2": mc["chi2"], "p_McNemar": mc["p_value"],
        "Sig(α=.05)": "Yes" if mc["p_value"] < 0.05 else "No",
    })
print(f"  Listo en {time.time()-_t_delong0:.2f}s total "
      f"(antes con bootstrap esto tardaba del orden de decenas de minutos).")

save_table_apa(pd.DataFrame(delong_rows),  "Table2_DeLong.csv",
               "Table 2. DeLong test (analítico, Sun & Xu 2014) para comparación "
               "pairwise de AUC.",
               subdir="08_METRICS")
save_table_apa(pd.DataFrame(mcnemar_rows), "Table3_McNemar.csv",
               "Table 3. McNemar test para comparación pairwise de predicciones.",
               subdir="08_METRICS")


# ══════════════════════════════════════════════════════════════════════════════
# CELDA 11 · Figuras diagnóstico (ROC, PR, Calibración, Confusion Matrix)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)
for i, m_name in enumerate(active_models):
    if m_name not in PROBA_TE:
        continue
    fpr, tpr, _ = roc_curve(y_te_arr, PROBA_TE[m_name])
    auc = METRICS_FULL[m_name]["point"]["AUC"]
    ci  = METRICS_FULL[m_name]["ci"]["AUC"]
    ax.plot(fpr, tpr, lw=2, color=PAL[i % len(PAL)],
            label=f"{MODEL_LABELS.get(m_name,m_name)} "
                  f"(AUC={auc:.3f} [{ci['ci_lo']:.3f}–{ci['ci_hi']:.3f}])")
ax.plot([0,1],[0,1],"k--",lw=0.8,alpha=0.4)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("Figure 1. ROC Curves – Holdout 2025", fontweight="bold")
ax.legend(loc="lower right", fontsize=7)
plt.tight_layout()
save_fig(fig, "Fig1_ROC_curves.png"); plt.close()

fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)
baseline = y_te_arr.mean()
for i, m_name in enumerate(active_models):
    if m_name not in PROBA_TE:
        continue
    pr, rc, _ = precision_recall_curve(y_te_arr, PROBA_TE[m_name])
    ap  = METRICS_FULL[m_name]["point"]["PR_AUC"]
    ci  = METRICS_FULL[m_name]["ci"]["PR_AUC"]
    ax.plot(rc, pr, lw=2, color=PAL[i % len(PAL)],
            label=f"{MODEL_LABELS.get(m_name,m_name)} (AP={ap:.3f})")
ax.axhline(baseline, ls="--", lw=0.8, color="gray",
           label=f"Baseline (prev={baseline:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Figure 2. Precision-Recall Curves – Holdout 2025", fontweight="bold")
ax.legend(loc="upper right", fontsize=7)
plt.tight_layout()
save_fig(fig, "Fig2_PR_curves.png"); plt.close()

from sklearn.metrics import ConfusionMatrixDisplay
thr_best  = THRESHOLDS.get(best_model, 0.5)
pred_best = (PROBA_TE[best_model] >= thr_best).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (norm, title) in zip(axes, [
    (None,   f"Figure 3a. Confusion Matrix – {MODEL_LABELS[best_model]}"),
    ("true", "Figure 3b. Normalized Confusion Matrix"),
]):
    cm = confusion_matrix(y_te_arr, pred_best, normalize=norm)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Normal","LBW"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False,
              values_format=".0f" if norm is None else ".2f")
    ax.set_title(title, fontsize=9, fontweight="bold")
plt.tight_layout()
save_fig(fig, "Fig3_ConfusionMatrix.png"); plt.close()

# ══════════════════════════════════════════════════════════════════════════════
# CELDA 12 · MÓDULO 12 · SHAP Completo
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 12 · EXPLAINABILITY SHAP COMPLETO")
print("═" * 60)

try:
    import shap
    from sklearn.inspection import PartialDependenceDisplay, partial_dependence
    # FIX: shap.initjs() se quita — solo hace falta para plots JS interactivos
    # (force_plot), que este pipeline no usa (todo se guarda como PNG estático
    # con matplotlib). En algunos entornos de Colab esa llamada puede quedarse
    # esperando la carga del componente JS sin avisar nada por consola.

    pipe_shap  = FINAL_PIPES[best_model]
    pre_shap   = pipe_shap.named_steps["pre"]
    clf_shap   = pipe_shap.named_steps["clf"]
    feat_names = get_feature_names(pre_shap)

    # FIX: 3000 → 1000 muestras. Para las figuras de un artículo Q1
    # (beeswarm, bar, dependence) 1000 es más que suficiente y reduce
    # bastante el tiempo de cálculo, sobre todo con modelos donde SHAP no
    # tiene una ruta rápida nativa (ver HGB más abajo).
    N_SHAP   = min(1000, len(X_te_f))
    print(f"  [SHAP] Muestra para SHAP: {N_SHAP:,} filas (modelo: {MODEL_LABELS.get(best_model,best_model)})")
    samp_idx = X_te_f.sample(N_SHAP, random_state=RANDOM_STATE).index
    samp_raw = X_te_f.loc[samp_idx]
    samp_proc = pd.DataFrame(pre_shap.transform(samp_raw), columns=feat_names)

    # FIX: HistGradientBoostingClassifier se saca del grupo de TreeExplainer.
    # El soporte de shap.TreeExplainer para HGB es inconsistente entre
    # versiones de SHAP (a veces tarda muchísimo, a veces falla en silencio
    # dentro del try/except general del módulo). Para HGB se usa el
    # Explainer genérico sobre predict_proba con un background reducido,
    # que es lento por muestra pero predecible y no depende de soporte
    # interno específico del árbol.
    TREE_EXPLAINER_MODELS = {"XGB", "LGB", "CAT", "RF", "ET", "BRF"}

    if best_model in TREE_EXPLAINER_MODELS:
        print("  [SHAP] Modelo con soporte nativo de árboles — usando TreeExplainer...")
        t_shap0 = time.time()
        explainer = shap.TreeExplainer(clf_shap)
        print(f"  [SHAP] TreeExplainer creado en {time.time()-t_shap0:.1f}s. "
              f"Calculando shap_values sobre {N_SHAP:,} filas...")
        t_shap1 = time.time()
        shap_vals = explainer.shap_values(samp_proc)
        if isinstance(shap_vals, list):
            shap_vals = shap_vals[1]
        print(f"  [SHAP] shap_values listos en {time.time()-t_shap1:.1f}s.")
    elif best_model == "HGB":
        print("  [SHAP] HistGradientBoosting: TreeExplainer no es confiable para "
              "este modelo entre versiones de SHAP — usando Explainer genérico "
              "sobre predict_proba (más lento por fila, pero predecible)...")
        N_BACKGROUND = min(100, len(samp_proc))
        background = samp_proc.sample(N_BACKGROUND, random_state=RANDOM_STATE)
        t_shap0 = time.time()
        explainer = shap.Explainer(clf_shap.predict_proba, background)
        print(f"  [SHAP] Explainer creado en {time.time()-t_shap0:.1f}s "
              f"(background={N_BACKGROUND} filas). Calculando shap_values sobre "
              f"{N_SHAP:,} filas — esto puede tardar varios minutos...")
        t_shap1 = time.time()
        shap_exp  = explainer(samp_proc)
        shap_vals = shap_exp.values[..., 1] if shap_exp.values.ndim == 3 else shap_exp.values
        print(f"  [SHAP] shap_values listos en {time.time()-t_shap1:.1f}s.")
    else:
        print("  [SHAP] Modelo lineal — usando LinearExplainer...")
        t_shap0 = time.time()
        explainer = shap.LinearExplainer(clf_shap, samp_proc)
        shap_vals = explainer.shap_values(samp_proc)
        print(f"  [SHAP] shap_values listos en {time.time()-t_shap0:.1f}s.")

    shap_mean_abs = np.abs(shap_vals).mean(axis=0)
    top4_idx      = np.argsort(shap_mean_abs)[::-1][:4]
    top4_names    = [feat_names[i] for i in top4_idx]

    fig = plt.figure(figsize=FIGSIZE_SINGLE)
    shap.summary_plot(shap_vals, samp_proc, show=False, max_display=15,
                      plot_size=FIGSIZE_SINGLE)
    plt.title(f"Figure 4. SHAP Beeswarm – {MODEL_LABELS[best_model]}",
              fontweight="bold", fontsize=10)
    plt.tight_layout()
    save_fig(plt.gcf(), "M12_SHAP_beeswarm.png", subdir="11_SHAP"); plt.close()

    fig = plt.figure(figsize=FIGSIZE_SINGLE)
    shap.summary_plot(shap_vals, samp_proc, plot_type="bar", show=False,
                      max_display=15, plot_size=FIGSIZE_SINGLE)
    plt.title(f"Figure 5. SHAP Feature Importance – {MODEL_LABELS[best_model]}",
              fontweight="bold", fontsize=10)
    plt.tight_layout()
    save_fig(plt.gcf(), "M12_SHAP_importance.png", subdir="11_SHAP"); plt.close()

    try:
        expl_obj = explainer(samp_proc.iloc[:1])
        if hasattr(expl_obj, "values"):
            fig = plt.figure(figsize=FIGSIZE_SINGLE)
            shap.waterfall_plot(expl_obj[0], show=False, max_display=15)
            plt.title("Figure 6. SHAP Waterfall – Caso Individual",
                      fontweight="bold", fontsize=10)
            plt.tight_layout()
            save_fig(plt.gcf(), "M12_SHAP_waterfall.png", subdir="11_SHAP")
            plt.close()
    except Exception as ew:
        print(f"  [WARN] SHAP Waterfall: {ew}")

    try:
        expected_value = explainer.expected_value
        if isinstance(expected_value, list):
            expected_value = expected_value[1]
        fig = plt.figure(figsize=(8, 6))
        shap.decision_plot(expected_value, shap_vals[:50], feat_names,
                           show=False, feature_display_range=slice(-1, -16, -1))
        plt.title("Figure 7. SHAP Decision Plot (50 casos)",
                  fontweight="bold", fontsize=10)
        plt.tight_layout()
        save_fig(plt.gcf(), "M12_SHAP_decision.png", subdir="11_SHAP")
        plt.close()
    except Exception as ed:
        print(f"  [WARN] SHAP Decision Plot: {ed}")

    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    for ax, feat in zip(axes.ravel(), top4_names):
        try:
            shap.dependence_plot(feat, shap_vals, samp_proc,
                                 ax=ax, show=False, dot_size=6)
            ax.set_title(f"SHAP Dependence: {feat}", fontsize=8)
        except Exception:
            ax.set_title(f"{feat} (not available)", fontsize=8)
    plt.suptitle("Figure 8. SHAP Dependence Plots (Top 4 Features)", fontweight="bold")
    plt.tight_layout()
    save_fig(fig, "M12_SHAP_dependence.png", subdir="11_SHAP"); plt.close()

    if best_model in ("XGB","LGB","CAT","RF","ET"):
        # FIX: "HGB" se quita de esta lista — ahora usa el Explainer genérico
        # (ver arriba), que no tiene método shap_interaction_values (es
        # exclusivo de TreeExplainer). Sin este ajuste el try/except de abajo
        # lo atrapaba igual, pero perdía tiempo intentándolo en vano.
        try:
            shap_interact = explainer.shap_interaction_values(samp_proc.iloc[:300])
            if isinstance(shap_interact, list):
                shap_interact = shap_interact[1]
            top2 = top4_names[:2]
            if all(f in feat_names for f in top2):
                i0 = feat_names.index(top2[0])
                i1 = feat_names.index(top2[1])
                fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)
                im = ax.scatter(samp_proc.iloc[:300][top2[0]],
                                shap_interact[:300, i0, i1],
                                c=samp_proc.iloc[:300][top2[1]],
                                cmap="coolwarm", alpha=0.5, s=10)
                plt.colorbar(im, ax=ax, label=top2[1])
                ax.set_xlabel(top2[0]); ax.set_ylabel(f"SHAP interaction with {top2[1]}")
                ax.set_title("Figure 9. SHAP Interaction Values (Top 2 Features)",
                             fontweight="bold")
                plt.tight_layout()
                save_fig(fig, "M12_SHAP_interactions.png", subdir="11_SHAP")
                plt.close()
        except Exception as ei:
            print(f"  [WARN] SHAP Interactions: {ei}")

    shap_rank = pd.DataFrame({
        "Feature"    : feat_names,
        "Mean_SHAP"  : np.abs(shap_vals).mean(axis=0),
        "Std_SHAP"   : np.abs(shap_vals).std(axis=0),
    }).sort_values("Mean_SHAP", ascending=False)
    save_table_apa(shap_rank, "M12_SHAP_ranking.csv",
                   "Módulo 12. SHAP feature ranking (mean |SHAP|)",
                   subdir="11_SHAP")
    print("  Módulo 12 completado.")
except Exception as e:
    print(f"  [WARN] Módulo 12 error: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# CELDA 13 · PDP + ICE
# ══════════════════════════════════════════════════════════════════════════════
try:
    from sklearn.inspection import partial_dependence
    num_feats_clean = [c for c in NUM_PRENATAL if c in X_te_f.columns][:4]
    N_PDP = min(2000, len(X_tr_f))
    idx_pdp = X_tr_f.sample(N_PDP, random_state=RANDOM_STATE).index
    X_pdp   = X_tr_f.loc[idx_pdp, FEATURES]

    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    for ax, feat in zip(axes.ravel(), num_feats_clean):
        try:
            feat_idx = list(X_pdp.columns).index(feat)
            pd_r = partial_dependence(pipe_shap, X_pdp, features=[feat_idx],
                                      kind="average", grid_resolution=30)
            ax.plot(pd_r["grid_values"][0], pd_r["average"][0], lw=2, color=PAL[0])
            ax.set_xlabel(feat, fontsize=8); ax.set_ylabel("Partial dependence", fontsize=8)
            ax.set_title(f"PDP: {feat}", fontsize=9, fontweight="bold")
        except Exception:
            ax.set_title(f"PDP: {feat}\n(no disponible)", fontsize=8)
    plt.suptitle("Figure 10. Partial Dependence Plots (PDP)", fontweight="bold")
    plt.tight_layout()
    save_fig(fig, "Fig10_PDP.png"); plt.close()

    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    N_ICE = min(200, len(X_pdp))
    for ax, feat in zip(axes.ravel(), num_feats_clean):
        try:
            feat_idx = list(X_pdp.columns).index(feat)
            ice_r = partial_dependence(pipe_shap, X_pdp.iloc[:N_ICE],
                                       features=[feat_idx], kind="individual",
                                       grid_resolution=25)
            for line in ice_r["individual"][0]:
                ax.plot(ice_r["grid_values"][0], line, lw=0.4, alpha=0.2, color=PAL[2])
            pd_r2 = partial_dependence(pipe_shap, X_pdp, features=[feat_idx],
                                       kind="average", grid_resolution=25)
            ax.plot(pd_r2["grid_values"][0], pd_r2["average"][0],
                    lw=2.5, color=PAL[3], label="PDP (mean)")
            ax.set_xlabel(feat, fontsize=8); ax.set_ylabel("Predicted prob", fontsize=8)
            ax.set_title(f"ICE: {feat}", fontsize=9, fontweight="bold")
            ax.legend(fontsize=7)
        except Exception:
            ax.set_title(f"ICE: {feat}\n(no disponible)", fontsize=8)
    plt.suptitle("Figure 11. ICE Plots", fontweight="bold")
    plt.tight_layout()
    save_fig(fig, "Fig11_ICE.png"); plt.close()
except Exception as e:
    print(f"  [WARN] PDP/ICE error: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 13 · ERROR ANALYSIS (FP, FN, casos difíciles, ambiguos, extremos)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 13 · ERROR ANALYSIS COMPLETO")
print("═" * 60)

try:
    p_best    = PROBA_TE[best_model]
    error_df  = df_te.copy()
    error_df["_prob"]  = p_best
    error_df["_pred"]  = pred_best
    error_df["_true"]  = y_te_arr
    error_df["_error"] = error_df.apply(
        lambda r: "TP" if r["_true"]==1 and r["_pred"]==1
             else "TN" if r["_true"]==0 and r["_pred"]==0
             else "FP" if r["_true"]==0 and r["_pred"]==1
             else "FN", axis=1)

    thr_band = 0.10
    mask_dif = (error_df["_prob"] >= thr_best - thr_band) & \
               (error_df["_prob"] <= thr_best + thr_band)
    mask_amb = (error_df["_prob"] >= 0.40) & (error_df["_prob"] <= 0.60)
    mask_ext = (error_df["_prob"] > 0.90) | (error_df["_prob"] < 0.05)

    error_df["_case_type"] = "Normal"
    error_df.loc[mask_dif, "_case_type"] = "Dificil"
    error_df.loc[mask_amb, "_case_type"] = "Ambiguo"
    error_df.loc[mask_ext, "_case_type"] = "Extremo"

    summary_cols = [COL["edad_madre"], COL["n_embarazos"],
                    COL["nivel_educacion"], "GRUPO_ETARIO", "DEPARTAMENTO",
                    "ROA", "TPF", "RIESGO_EXTREMO", "_prob"]

    fp_df = error_df[error_df["_error"]=="FP"][summary_cols+["_prob","_error"]]
    fn_df = error_df[error_df["_error"]=="FN"][summary_cols+["_prob","_error"]]
    hard_df = error_df[mask_dif][summary_cols+["_prob","_error","_case_type"]]
    ambig_df = error_df[mask_amb][summary_cols+["_prob","_error","_case_type"]]

    save_table_apa(fp_df.head(300), "M13_FalsePositives.csv",
                   "Módulo 13. Perfil de Falsos Positivos", subdir="14_ERROR_ANALYSIS")
    save_table_apa(fn_df.head(300), "M13_FalseNegatives.csv",
                   "Módulo 13. Perfil de Falsos Negativos", subdir="14_ERROR_ANALYSIS")
    save_table_apa(hard_df.head(300), "M13_HardCases.csv",
                   "Módulo 13. Casos difíciles (prob ≈ umbral)", subdir="14_ERROR_ANALYSIS")
    save_table_apa(ambig_df.head(300), "M13_AmbiguousCases.csv",
                   "Módulo 13. Casos ambiguos (prob ∈ [0.4, 0.6])", subdir="14_ERROR_ANALYSIS")

    error_colors = {"TP": PAL[2], "TN": PAL[0], "FP": PAL[1], "FN": PAL[3]}
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    ax = axes[0]
    for et, grp in error_df.groupby("_error"):
        sns.kdeplot(grp["_prob"], ax=ax, lw=2, label=f"{et} (n={len(grp):,})",
                    color=error_colors.get(et,"gray"))
    ax.axvline(thr_best, ls="--", lw=1, color="black", label=f"Threshold={thr_best:.2f}")
    ax.set_xlabel("Predicted probability"); ax.set_ylabel("Density")
    ax.set_title("Prob. por Tipo de Error", fontweight="bold", fontsize=9)
    ax.legend(fontsize=7)

    ax = axes[1]
    for et, color, label in [("FP", PAL[1], "False Positives"), ("FN", PAL[3], "False Negatives")]:
        subset = error_df.loc[error_df["_error"]==et, COL["edad_madre"]].dropna()
        if len(subset):
            ax.hist(subset, bins=20, alpha=0.6, color=color, label=f"{label} (n={len(subset):,})")
    ax.set_xlabel("Edad materna"); ax.set_ylabel("Count")
    ax.set_title("Edad Materna: FP vs FN", fontweight="bold", fontsize=9)
    ax.legend(fontsize=8)

    ax = axes[2]
    ct = error_df["_case_type"].value_counts()
    ax.bar(ct.index, ct.values, color=[PAL[i % len(PAL)] for i in range(len(ct))])
    ax.set_title("Distribución de Tipos de Casos", fontweight="bold", fontsize=9)
    ax.set_ylabel("Count")

    plt.suptitle(f"Módulo 13 – Error Analysis ({MODEL_LABELS[best_model]})", fontweight="bold")
    plt.tight_layout()
    save_fig(fig, "M13_ErrorAnalysis.png", subdir="14_ERROR_ANALYSIS"); plt.close()

    print(f"  FP: {len(fp_df):,} | FN: {len(fn_df):,} | "
          f"Difíciles: {mask_dif.sum():,} | Extremos: {mask_ext.sum():,}")
    print("  Módulo 13 completado.")
except Exception as e:
    print(f"  [WARN] Módulo 13 error: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 14 · FAIRNESS COMPLETO
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 14 · FAIRNESS COMPLETO")
print("═" * 60)

try:
    def fairness_metrics_group(y_true, y_prob, y_pred, min_n=200):
        from sklearn.metrics import confusion_matrix as cm_fn
        if len(np.unique(y_true)) < 2 or len(y_true) < min_n:
            return None
        try:
            TN, FP, FN, TP = cm_fn(y_true, y_pred).ravel()
            auc  = roc_auc_score(y_true, y_prob)
            return {
                "N"              : len(y_true),
                "Prev%"          : round(y_true.mean()*100, 2),
                "AUC"            : round(auc, 4),
                "Recall"         : round(recall_score(y_true, y_pred, zero_division=0), 4),
                "FPR"            : round(FP/(FP+TN) if (FP+TN) > 0 else 0, 4),
                "PPV"            : round(precision_score(y_true, y_pred, zero_division=0), 4),
                "F1"             : round(f1_score(y_true, y_pred, zero_division=0), 4),
                "Demographic_Parity_Rate": round((y_pred==1).mean(), 4),
                "Calibration_gap": round(abs(y_true.mean() - y_prob.mean()), 4),
            }
        except:
            return None

    fairness_results   = {}
    fairness_agg_rows  = []

    for subg_name, subg_values in SUBG.items():
        rows = []
        for grp in np.unique(subg_values):
            mask = subg_values == grp
            res  = fairness_metrics_group(y_te_arr[mask], p_best[mask], pred_best[mask])
            if res:
                rows.append({"Group": grp, **res})
        if rows:
            df_fair = pd.DataFrame(rows).sort_values("AUC", ascending=True)
            fairness_results[subg_name] = df_fair

            auc_min, auc_max = df_fair["AUC"].min(), df_fair["AUC"].max()
            dp_max = df_fair["Demographic_Parity_Rate"].max()
            dp_min = df_fair["Demographic_Parity_Rate"].min()
            recall_min = df_fair["Recall"].min()
            recall_max = df_fair["Recall"].max()

            fairness_agg_rows.append({
                "Subgroup"              : subg_name,
                "N_groups"              : len(df_fair),
                "AUC_range"             : f"{auc_min:.4f}–{auc_max:.4f}",
                "AUC_gap"               : round(auc_max - auc_min, 4),
                "Demographic_Parity_diff": round(dp_max - dp_min, 4),
                "Equal_Opportunity_gap" : round(recall_max - recall_min, 4),
                "Disparate_Impact_ratio": round(dp_min / (dp_max + 1e-9), 4),
            })

            save_table_apa(df_fair, f"M14_Fairness_{subg_name}.csv",
                           f"Módulo 14. Fairness metrics by {subg_name}", subdir="12_FAIRNESS")
            print(f"\n  ─── Fairness: {subg_name.upper()} ───")
            print(df_fair[["Group","N","AUC","Recall","FPR","PPV"]].to_string(index=False))

    if fairness_agg_rows:
        df_fair_agg = pd.DataFrame(fairness_agg_rows)
        save_table_apa(df_fair_agg, "M14_Fairness_Summary.csv",
                       "Módulo 14. Resumen de métricas de equidad", subdir="12_FAIRNESS")

    try:
        df_te_copy = df_te.copy()
        df_te_copy["_edad_region"] = (df_te["GRUPO_ETARIO"].astype(str) + "_" +
                                       df_te["REGION"].astype(str))
        inter_rows = []
        for grp in df_te_copy["_edad_region"].unique():
            mask = (df_te_copy["_edad_region"] == grp).values
            if mask.sum() < 100 or len(np.unique(y_te_arr[mask])) < 2:
                continue
            res = fairness_metrics_group(y_te_arr[mask], p_best[mask], pred_best[mask], min_n=100)
            if res:
                inter_rows.append({"Group": grp, **res})
        if inter_rows:
            df_inter = pd.DataFrame(inter_rows).sort_values("AUC")
            save_table_apa(df_inter, "M14_Fairness_Intersectional.csv",
                           "Módulo 14. Fairness interseccional (edad × región)", subdir="12_FAIRNESS")
    except Exception as ei:
        print(f"  [WARN] Intersectional fairness: {ei}")

    n_subg = len(fairness_results)
    if n_subg > 0:
        fig, axes = plt.subplots(1, n_subg, figsize=(5*n_subg, 5))
        if n_subg == 1:
            axes = [axes]
        for ax, (subg_name, df_f) in zip(axes, fairness_results.items()):
            colors = [PAL[3] if v < df_f["AUC"].mean() - 0.02 else PAL[0] for v in df_f["AUC"]]
            ax.barh(df_f["Group"], df_f["AUC"], color=colors, edgecolor="white")
            ax.axvline(df_f["AUC"].mean(), ls="--", lw=1, color="black",
                       label=f"Mean AUC={df_f['AUC'].mean():.3f}")
            ax.set_xlabel("AUC")
            ax.set_title(f"Fairness by {subg_name.title()}", fontweight="bold", fontsize=9)
            ax.legend(fontsize=7)
        plt.suptitle(f"Módulo 14 – AUC Fairness Audit – {MODEL_LABELS[best_model]}", fontweight="bold")
        plt.tight_layout()
        save_fig(fig, "M14_Fairness.png", subdir="12_FAIRNESS"); plt.close()
    print("  Módulo 14 completado.")
except Exception as e:
    print(f"  [WARN] Módulo 14 error: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 15 · ABLATION STUDY
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 15 · ABLATION STUDY")
print("═" * 60)

if RESUME.get("ablation_done") and (DIRS["15_ABLATION"] / "M15_ablation_study.csv").exists():
    print("  [RESUME] Módulo 15 (Ablation Study) ya completado en una corrida anterior — "
          "se omite (evita reentrenar 7 modelos adicionales).")
else:
    try:
        ABLATION_GROUPS = {
            "Sin_ROA"          : ["ROA", "ROA2", "ROA_x_PRIMIG", "TPF_x_RIESGO"],
            "Sin_EM"           : ["EM"],
            "Sin_Abortos"      : [COL["abortos_previos"]],
            "Sin_Edad"         : [COL["edad_madre"], "EDAD2", "EDAD_x_ROA", "EDAD_x_EMB",
                                  "RIESGO_EXTREMO"],
            "Sin_Educacion"    : [COL["nivel_educacion"]],
            "Sin_Financiador"  : [COL["financiador"]],
            "Sin_Interacciones": ["EDAD_x_ROA","EDAD_x_EMB","ROA_x_PRIMIG",
                                  "TPF_x_RIESGO","EDAD2","ROA2"],
        }

        ref_auc = METRICS_FULL[best_model]["point"]["AUC"]
        ref_pr  = METRICS_FULL[best_model]["point"]["PR_AUC"]
        ref_f1  = METRICS_FULL[best_model]["point"]["F1"]

        ablation_rows = [{"Ablation": "Full_Model (baseline)",
                          "AUC": ref_auc, "PR_AUC": ref_pr, "F1": ref_f1,
                          "ΔAUC": 0.0, "ΔPR": 0.0, "ΔF1": 0.0,
                          "Features_removed": "None"}]

        params_best = best_params_for(best_model) or FALLBACK_PARAMS.get(best_model, {})

        # FIX RAM/tiempo: antes cada uno de los 7 reentrenamientos usaba el
        # train COMPLETO (4.5M filas). Un ablation study evalúa el IMPACTO
        # RELATIVO de quitar un grupo de features (ΔAUC frente al modelo
        # completo) — no necesita el dataset entero para eso, igual que el
        # Nested CV ya usa una muestra (N_NCV) en vez de las 4.5M filas.
        # Se usa una muestra estratificada de hasta 300k filas, manteniendo
        # la proporción de BAJO_PESO. La evaluación sigue siendo sobre TODO
        # el holdout 2025 (df_te), sin cambios — solo se reduce el TRAIN.
        N_ABLATION_SAMPLE = min(300_000, len(df_tr))
        if N_ABLATION_SAMPLE < len(df_tr):
            from sklearn.model_selection import train_test_split as _tts_abl
            _idx_abl, _ = _tts_abl(df_tr.index, train_size=N_ABLATION_SAMPLE,
                                    stratify=y_tr, random_state=RANDOM_STATE)
            df_tr_abl = df_tr.loc[_idx_abl]
            y_tr_abl  = y_tr.loc[_idx_abl]
        else:
            df_tr_abl, y_tr_abl = df_tr, y_tr
        print(f"  Muestra de entrenamiento para ablation: {len(df_tr_abl):,} filas "
              f"(de {len(df_tr):,} totales) — evaluación sigue sobre holdout 2025 completo.")

        for abl_name, drop_cols in ABLATION_GROUPS.items():
            try:
                feats_abl = [f for f in FEATURES if f not in drop_cols]
                if len(feats_abl) < 3:
                    continue
                num_abl = [f for f in NUM_PRENATAL if f not in drop_cols and f in df_tr.columns]
                cat_abl = [f for f in CAT_PRENATAL if f not in drop_cols and f in df_tr.columns]

                pre_abl  = build_preprocessor(num_cols=num_abl, cat_cols=cat_abl)
                clf_abl  = make_clf(best_model, params_best)
                pipe_abl = Pipeline([("pre", pre_abl), ("clf", clf_abl)])
                pipe_abl.fit(df_tr_abl[feats_abl], y_tr_abl)
                prob_abl = pipe_abl.predict_proba(df_te[feats_abl])[:, 1]
                thr_abl  = THRESHOLDS.get(best_model, 0.5)

                auc_abl = roc_auc_score(y_te_arr, prob_abl)
                pr_abl  = average_precision_score(y_te_arr, prob_abl)
                pred_a  = (prob_abl >= thr_abl).astype(int)
                f1_abl  = f1_score(y_te_arr, pred_a, zero_division=0)

                ablation_rows.append({
                    "Ablation"       : abl_name,
                    "AUC"            : round(auc_abl, 4),
                    "PR_AUC"         : round(pr_abl, 4),
                    "F1"             : round(f1_abl, 4),
                    "ΔAUC"           : round(auc_abl - ref_auc, 4),
                    "ΔPR"            : round(pr_abl - ref_pr, 4),
                    "ΔF1"            : round(f1_abl - ref_f1, 4),
                    "Features_removed": str(drop_cols),
                })
                print(f"  {abl_name:25s}  AUC={auc_abl:.4f} (ΔAUC={auc_abl-ref_auc:+.4f})")
            except Exception as ea:
                print(f"  [WARN] Ablation {abl_name}: {ea}")
            finally:
                # Libera explícitamente el pipeline/modelo/preprocesador de
                # esta iteración antes de gc.collect(). Con reasignación de
                # variables Python ya libera por conteo de referencias en la
                # siguiente vuelta del for, pero sklearn a veces genera
                # referencias circulares internas que solo gc.collect()
                # puede resolver del todo — el del explícito lo hace más
                # determinista y no depende de esperar a la próxima iteración.
                for _v in ("pipe_abl", "clf_abl", "pre_abl", "prob_abl", "pred_a"):
                    if _v in locals():
                        del locals()[_v]
                gc.collect()

        del df_tr_abl, y_tr_abl
        gc.collect()

        df_ablation = pd.DataFrame(ablation_rows)
        save_table_apa(df_ablation, "M15_ablation_study.csv",
                       "Módulo 15. Ablation Study – impacto de grupos de features", subdir="15_ABLATION")

        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        for ax, metric, label in zip(axes, ["ΔAUC","ΔPR","ΔF1"], ["ΔAUC","ΔPR-AUC","ΔF1"]):
            sub = df_ablation[df_ablation["Ablation"] != "Full_Model (baseline)"]
            colors = [PAL[3] if v < -0.005 else PAL[0] for v in sub[metric]]
            ax.barh(sub["Ablation"], sub[metric], color=colors, edgecolor="white")
            ax.axvline(0, color="black", lw=0.8)
            ax.set_xlabel(label); ax.set_title(f"Ablation: {label}", fontweight="bold")
        plt.suptitle("Módulo 15 – Ablation Study", fontweight="bold")
        plt.tight_layout()
        save_fig(fig, "M15_Ablation.png", subdir="15_ABLATION"); plt.close()
        mark_stage_done("ablation_done", RESUME)
        print("  Módulo 15 completado.")
    except Exception as e:
        print(f"  [WARN] Módulo 15 error: {e}")

# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 16 · ROBUSTEZ COMPLETA
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 16 · ROBUSTEZ COMPLETA")
print("═" * 60)

try:
    def add_gaussian_noise(X_df, num_cols, noise_std_frac=0.1, seed=42):
        rng = np.random.RandomState(seed)
        X_n = X_df.copy()
        for c in num_cols:
            if c in X_n.columns:
                std = X_n[c].std()
                X_n[c] += rng.normal(0, noise_std_frac * std, size=len(X_n))
        return X_n

    def add_mcar_missing(X_df, cols, missing_frac=0.1, seed=42):
        rng = np.random.RandomState(seed)
        X_m = X_df.copy()
        for c in cols:
            if c in X_m.columns:
                mask = rng.rand(len(X_m)) < missing_frac
                X_m.loc[mask, c] = np.nan
        return X_m

    def add_adversarial_noise(X_df, num_cols, epsilon=0.05, seed=42):
        rng = np.random.RandomState(seed)
        X_a = X_df.copy()
        for c in num_cols:
            if c in X_a.columns:
                std = X_a[c].std()
                X_a[c] += rng.uniform(-epsilon*std, epsilon*std, size=len(X_a))
        return X_a

    # FIX tiempo/RAM: 4 niveles -> 3 para ruido gaussiano y missing (25% menos
    # trabajo, cobertura del espacio de perturbación prácticamente igual de
    # informativa: bajo/medio/alto en vez de 4 puntos). Cada nivel crea una
    # copia completa de X_te_f (cientos de miles de filas); se libera
    # explícitamente con del + gc.collect() apenas se usa, en vez de dejar
    # que se acumulen 11 copias grandes vivas hasta el final del módulo.
    noise_levels   = [0.05, 0.15, 0.30]
    missing_levels = [0.05, 0.15, 0.30]
    adv_levels     = [0.05, 0.15, 0.30]
    num_avail      = [c for c in NUM_PRENATAL if c in X_te_f.columns]
    auc_ref        = METRICS_FULL[best_model]["point"]["AUC"]
    robust_rows    = []

    print(f"  {'Perturbación':30s}  {'Nivel':>8s}  {'AUC':>8s}  {'ΔAUC':>12s}")
    print("  " + "-" * 65)

    for nlvl in noise_levels:
        X_noisy   = add_gaussian_noise(X_te_f, num_avail, nlvl)
        p_noisy   = FINAL_PIPES[best_model].predict_proba(X_noisy)[:, 1]
        auc_noisy = roc_auc_score(y_te_arr, p_noisy)
        delta     = auc_noisy - auc_ref
        robust_rows.append({"Perturbation":"Gaussian_Noise","Level":nlvl,
                             "AUC":round(auc_noisy,4),"Delta_AUC":round(delta,4)})
        print(f"  {'Gaussian noise':30s}  {nlvl:>8.2f}  {auc_noisy:>8.4f}  {delta:>+12.4f}")
        del X_noisy, p_noisy
        gc.collect()

    for mlvl in missing_levels:
        X_miss = add_mcar_missing(X_te_f, num_avail, mlvl)
        try:
            p_miss   = FINAL_PIPES[best_model].predict_proba(X_miss)[:, 1]
            auc_miss = roc_auc_score(y_te_arr, p_miss)
            delta    = auc_miss - auc_ref
        except:
            auc_miss, delta = np.nan, np.nan
        robust_rows.append({"Perturbation":"MCAR_Missing","Level":mlvl,
                             "AUC":round(auc_miss,4) if not np.isnan(auc_miss) else None,
                             "Delta_AUC":round(delta,4) if not np.isnan(delta) else None})
        print(f"  {'MCAR missing':30s}  {mlvl:>8.2f}  {auc_miss:>8.4f}  {delta:>+12.4f}")
        del X_miss
        gc.collect()

    for alvl in adv_levels:
        X_adv   = add_adversarial_noise(X_te_f, num_avail, alvl)
        p_adv   = FINAL_PIPES[best_model].predict_proba(X_adv)[:, 1]
        auc_adv = roc_auc_score(y_te_arr, p_adv)
        delta   = auc_adv - auc_ref
        robust_rows.append({"Perturbation":"Adversarial_Noise","Level":alvl,
                             "AUC":round(auc_adv,4),"Delta_AUC":round(delta,4)})
        print(f"  {'Adversarial noise':30s}  {alvl:>8.2f}  {auc_adv:>8.4f}  {delta:>+12.4f}")
        del X_adv, p_adv
        gc.collect()

    df_robust = pd.DataFrame(robust_rows)
    save_table_apa(df_robust, "M16_Robustness.csv",
                   "Módulo 16. Análisis de robustez (Gaussian, MCAR, Adversarial)", subdir="08_METRICS")

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, ptb in zip(axes, ["Gaussian_Noise","MCAR_Missing","Adversarial_Noise"]):
        sub = df_robust[df_robust["Perturbation"]==ptb].dropna()
        if len(sub):
            ax.plot(sub["Level"], sub["AUC"], "o-", lw=2, ms=7, color=PAL[0])
            ax.axhline(auc_ref, ls="--", lw=1, color="red", label=f"Reference AUC={auc_ref:.4f}")
            ax.fill_between(sub["Level"], auc_ref, sub["AUC"], alpha=0.15, color=PAL[3])
            ax.set_xlabel("Perturbation level"); ax.set_ylabel("AUC")
            ax.set_title(f"Robustness: {ptb.replace('_',' ')}", fontweight="bold", fontsize=9)
            ax.legend(fontsize=8)
    plt.suptitle("Módulo 16 – Robustness Analysis", fontweight="bold")
    plt.tight_layout()
    save_fig(fig, "M16_Robustness.png"); plt.close()
    print("  Módulo 16 completado.")
except Exception as e:
    print(f"  [WARN] Módulo 16 error: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# CELDA 16 · Early-Warning Risk Score
# ══════════════════════════════════════════════════════════════════════════════
RISK_BANDS = [
    (0.00, 0.20, "Low",      PAL[2]),
    (0.20, 0.40, "Moderate", "orange"),
    (0.40, 0.60, "High",     PAL[1]),
    (0.60, 1.01, "Critical", PAL[3]),
]

def classify_risk(p):
    for lo, hi, label, _ in RISK_BANDS:
        if lo <= p < hi:
            return label
    return "Critical"

risk_levels  = np.array([classify_risk(p) for p in p_best])
order_labels = [b[2] for b in RISK_BANDS]
risk_rows    = []
for lo, hi, label, color in RISK_BANDS:
    mask  = (p_best >= lo) & (p_best < hi)
    n     = mask.sum()
    n_bpn = y_te_arr[mask].sum() if n > 0 else 0
    risk_rows.append({
        "Risk Level"       : label,
        "Prob range"       : f"[{lo:.0%}, {hi:.0%})",
        "N cases"          : int(n),
        "% of total"       : round(n/len(p_best)*100, 2),
        "Observed BPN (n)" : int(n_bpn),
        "Observed BPN (%)" : round(n_bpn/n*100, 2) if n > 0 else 0.0,
        "NNS"              : round(n/n_bpn, 1) if n_bpn > 0 else "∞",
    })
df_risk = pd.DataFrame(risk_rows)
save_table_apa(df_risk, "Table_RiskScore.csv",
               "Tabla. Early-Warning Risk Score – distribución en holdout 2025", subdir="08_METRICS")
print("\n─── Early-Warning Risk Score ───")
print(df_risk.to_string(index=False))


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 17 · EXTERNAL VALIDATION (acepta CSV externo sin modificar código)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 17 · EXTERNAL VALIDATION")
print("═" * 60)

def run_external_validation(ext_source, model_name=None, label="external", target_col=TARGET):
    model_name = model_name or best_model
    try:
        if isinstance(ext_source, (str, Path)):
            df_ext = pd.read_csv(str(ext_source), sep=None, engine="python",
                                 encoding="latin1", low_memory=False)
        else:
            # FIX RAM: antes se hacía ext_source.copy() — una copia completa
            # innecesaria, ya que df_ext nunca se modifica en esta función
            # (solo se seleccionan columnas con df_ext[available_feats], que
            # ya crea una vista/copia nueva y más chica). En la llamada de
            # ejemplo más abajo, ext_source ES df_te completo (~345k filas),
            # así que evitar el .copy() ahorra una duplicación grande.
            df_ext = ext_source

        available_feats = [f for f in FEATURES if f in df_ext.columns]
        if len(available_feats) < 5:
            print(f"  [WARN] Dataset externo tiene pocos features ({len(available_feats)}). "
                  f"Se necesitan: {FEATURES[:5]}")
            return None

        if target_col in df_ext.columns:
            y_ext  = df_ext[target_col].astype(int).values
            has_y  = True
        else:
            y_ext  = None; has_y = False

        X_ext    = df_ext[available_feats]
        p_ext    = FINAL_PIPES[model_name].predict_proba(X_ext)[:, 1]
        ext_rows = [{"Dataset": label, "N": len(X_ext), "Features_used": len(available_feats)}]

        if has_y and len(np.unique(y_ext)) > 1:
            auc_ext = roc_auc_score(y_ext, p_ext)
            pr_ext  = average_precision_score(y_ext, p_ext)
            thr_ext = THRESHOLDS.get(model_name, 0.5)
            pred_e  = (p_ext >= thr_ext).astype(int)
            ext_rows[0].update({
                "AUC": round(auc_ext, 4), "PR_AUC": round(pr_ext, 4),
                "F1": round(f1_score(y_ext, pred_e, zero_division=0), 4),
                "BPN_rate": round(y_ext.mean(), 4),
            })
            print(f"  External [{label}]: AUC={auc_ext:.4f}  PR-AUC={pr_ext:.4f}")
        else:
            print(f"  External [{label}]: N={len(X_ext):,}  (no target disponible)")

        df_ext_res = pd.DataFrame(ext_rows)
        save_table_apa(df_ext_res, f"M17_ExternalValidation_{label}.csv",
                       f"Módulo 17. External Validation – {label}", subdir="16_EXTERNAL_VALIDATION")
        return df_ext_res
    except Exception as ev:
        print(f"  [WARN] External Validation [{label}]: {ev}")
        return None
    finally:
        # X_ext/p_ext pueden ser grandes en la llamada de ejemplo (df_te
        # completo, ~345k filas) — se liberan explícitamente al salir.
        # NOTA: dentro de una función, `del locals()[...]` NO funciona (locals()
        # devuelve una copia dentro de funciones, a diferencia del código a
        # nivel de script) — hay que usar `del` directo sobre el nombre.
        try:
            del X_ext
        except NameError:
            pass
        try:
            del p_ext
        except NameError:
            pass
        gc.collect()

_ext_demo = run_external_validation(df_te, label="demo_holdout2025")
print("  Módulo 17 listo. Llama run_external_validation(path_csv) con tu dataset externo.")


# ══════════════════════════════════════════════════════════════════════════════
# CELDA 17 · MLflow Logging completo
# ══════════════════════════════════════════════════════════════════════════════
print("\nRegistrando runs en MLflow ...")
for m_name in active_models:
    if m_name not in METRICS_FULL:
        continue
    pt = METRICS_FULL[m_name]["point"]
    ci = METRICS_FULL[m_name]["ci"]
    nr = nested_results.get(m_name, {})
    try:
        with mlflow.start_run(run_name=f"{m_name}_final"):
            mlflow.set_tags({
                "model_type"   : m_name, "dataset": "CNV-MINSA-Peru",
                "split_type"   : "temporal", "train_years": "2015-2024",
                "test_year"    : "2025", "leakage_free": "True", "nested_cv": "True",
                "pipeline"     : "BPN_Peru_Q1_Extremo_V3_5",
            })
            best_p = best_params_for(m_name)
            mlflow.log_params({k: str(v)[:250] for k, v in best_p.items()})
            mlflow.log_param("random_state", RANDOM_STATE)
            mlflow.log_param("n_bootstrap",  N_BOOT)
            mlflow.log_param("n_trials_budget", N_TRIALS_BY_MODEL.get(m_name, 0))
            for metric, val in pt.items():
                if val is not None and not np.isnan(val):
                    mlflow.log_metric(metric, val)
            for metric, vals in ci.items():
                mlflow.log_metric(f"{metric}_ci_lo", vals["ci_lo"])
                mlflow.log_metric(f"{metric}_ci_hi", vals["ci_hi"])
            if "mean_auc" in nr:
                mlflow.log_metric("nested_cv_AUC_mean", nr["mean_auc"])
                mlflow.log_metric("nested_cv_AUC_sd",   nr["sd_auc"])
            mlflow.log_metric("fit_seconds",  FIT_TIMES.get(m_name, 0))
            mlflow.log_metric("pred_seconds", PRED_TIMES.get(m_name, 0))
            model_path = str(DIRS["06_MODELS"] / f"{m_name}_final.pkl")
            if Path(model_path).exists():
                mlflow.log_artifact(model_path, artifact_path="model")
            for fig_name in ["Fig1_ROC_curves.png", "M12_SHAP_beeswarm.png",
                             "M14_Fairness.png", "M15_Ablation.png"]:
                fp = DIRS["figures"] / fig_name
                if fp.exists():
                    mlflow.log_artifact(str(fp), artifact_path="figures")
    except Exception as em:
        print(f"  [WARN] MLflow {m_name}: {em}")
print("MLflow runs registrados.")


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 18 · REPORTES AUTOMÁTICOS (HTML, PDF, Markdown)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 18 · REPORTES AUTOMÁTICOS")
print("═" * 60)

try:
    md_lines = [
        f"# BPN Peru ML Framework Q1 – Reporte Automático",
        f"**Fecha:** {date.today()}  |  **Dataset:** CNV-MINSA-Peru  |  "
        f"**Mejor modelo:** {MODEL_LABELS.get(best_model,best_model)}",
        "", "## 1. Dataset",
        f"- Registros originales: {n_original:,}",
        f"- Registros válidos: {df.height:,}",
        f"- Retención: {retencion:.2%}",
        f"- Tasa BPN (total): {bpn:.2%}",
        f"- Años cubiertos: {anios}",
        f"- Checksum MD5: `{checksum}`",
        "", "## 2. Modelos evaluados",
        "| Modelo | AUC | PR-AUC | F1 | MCC |",
        "|--------|-----|--------|-----|-----|",
    ]
    for m_name in active_models:
        if m_name not in METRICS_FULL:
            continue
        pt = METRICS_FULL[m_name]["point"]
        marker = " ✅ **BEST**" if m_name == best_model else ""
        md_lines.append(
            f"| {MODEL_LABELS.get(m_name,m_name)}{marker} | "
            f"{pt['AUC']:.4f} | {pt['PR_AUC']:.4f} | {pt['F1']:.4f} | {pt['MCC']:.4f} |")

    md_lines += [
        "", "## 3. Métricas del Mejor Modelo",
        f"**Modelo:** {MODEL_LABELS.get(best_model,best_model)}",
        f"**Threshold (F1-optimal):** {thr_best:.3f}", "",
    ]
    pt_b = METRICS_FULL[best_model]["point"]
    ci_b = METRICS_FULL[best_model]["ci"]
    for metric in ["AUC","PR_AUC","Recall","Specificity","Precision","F1","MCC","Brier"]:
        if metric in pt_b and metric in ci_b:
            md_lines.append(
                f"- **{metric}:** {pt_b[metric]:.4f} "
                f"(95% CI [{ci_b[metric]['ci_lo']:.4f}–{ci_b[metric]['ci_hi']:.4f}])")

    md_path = DIRS["20_HTML_REPORT"] / "report.md"
    with open(md_path, "w", encoding="utf-8") as f:
        f.write("\n".join(md_lines))
    print(f"  Markdown: {md_path}")

    html_rows = ""
    for m_name in active_models:
        if m_name not in METRICS_FULL:
            continue
        pt = METRICS_FULL[m_name]["point"]
        style = "background:#e8f5e9;font-weight:bold;" if m_name == best_model else ""
        html_rows += (
            f"<tr style='{style}'>"
            f"<td>{MODEL_LABELS.get(m_name,m_name)}</td>"
            f"<td>{pt['AUC']:.4f}</td><td>{pt['PR_AUC']:.4f}</td>"
            f"<td>{pt['Recall']:.4f}</td><td>{pt['Precision']:.4f}</td>"
            f"<td>{pt['F1']:.4f}</td><td>{pt['MCC']:.4f}</td><td>{pt['Brier']:.4f}</td></tr>\n")

    html_content = f"""<!DOCTYPE html>
<html lang="es"><head><meta charset="UTF-8">
<title>BPN Peru Q1 – Reporte Automático</title>
<style>
  body{{font-family:Arial,sans-serif;margin:40px;color:#222}}
  h1{{color:#1a237e}} h2{{color:#283593}} h3{{color:#3949ab}}
  table{{border-collapse:collapse;width:100%;margin:20px 0}}
  th{{background:#3949ab;color:white;padding:10px;text-align:left}}
  td{{padding:8px;border-bottom:1px solid #ddd}}
  tr:hover{{background:#f5f5f5}}
  .badge{{background:#4caf50;color:white;padding:3px 8px;border-radius:4px;font-size:12px}}
  .summary{{background:#e8eaf6;padding:15px;border-radius:8px;margin:15px 0}}
</style></head><body>
<h1>🏥 BPN Peru ML Framework Q1 Extremo V3</h1>
<p><strong>Fecha:</strong> {date.today()} &nbsp;|&nbsp;
<strong>Dataset:</strong> CNV-MINSA-Peru (2015–2025) &nbsp;|&nbsp;
<strong>Pipeline:</strong> BPN_Peru_Q1_Extremo_V3_5</p>

<div class="summary">
<h3>📊 Dataset Summary</h3>
<p>Registros originales: <strong>{n_original:,}</strong> &nbsp;|&nbsp;
Registros válidos: <strong>{df.height:,}</strong> &nbsp;|&nbsp;
Retención: <strong>{retencion:.2%}</strong><br>
Tasa BPN global: <strong>{bpn:.2%}</strong> &nbsp;|&nbsp;
Años: <strong>{anios[0]}–{anios[-1]}</strong> &nbsp;|&nbsp;
Checksum: <code>{checksum}</code></p>
</div>

<h2>🏆 Rendimiento de Modelos en Holdout 2025</h2>
<p>Mejor modelo: <span class="badge">{MODEL_LABELS.get(best_model,best_model)}</span></p>
<table>
<tr><th>Modelo</th><th>AUC</th><th>PR-AUC</th>
<th>Recall</th><th>Precision</th><th>F1</th><th>MCC</th><th>Brier</th></tr>
{html_rows}
</table>

<h2>⚙️ Configuración del Pipeline</h2>
<ul>
  <li>Split temporal: Train 2015–2024 / Test 2025</li>
  <li>Nested CV: {N_OUTER}-outer × {N_INNER}-inner | trials adaptativos por modelo</li>
  <li>Bootstrap IC95%: {N_BOOT} remuestreos</li>
  <li>Early stopping: XGBoost, LightGBM, CatBoost ({EARLY_STOP_ROUNDS} rounds)</li>
  <li>Módulos implementados: 23 (Dataset Audit, Drift, Leakage, FE, Stability,
      Optuna, Validation, Models, Calibration, Threshold, Metrics, SHAP,
      Error Analysis, Fairness, Ablation, Robustness, External Val.,
      Reports, DOI, Paper Package, Learning Curves, Permutation Importance,
      Export de Predicciones)</li>
</ul>
<hr><p style="color:#666;font-size:12px">
Generado automáticamente · BPN_Peru_Q1_Extremo_V3_5 · {datetime.now().strftime('%Y-%m-%d %H:%M')}
</p></body></html>"""

    html_path = DIRS["20_HTML_REPORT"] / "report.html"
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html_content)
    print(f"  HTML:     {html_path}")

    try:
        from reportlab.lib.pagesizes import A4
        from reportlab.lib import colors
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.units import cm
        from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer,
                                         Table, TableStyle, HRFlowable)
        from reportlab.lib.enums import TA_CENTER

        pdf_path = DIRS["19_PDF_REPORT"] / "BPN_Peru_Q1_Report.pdf"
        doc   = SimpleDocTemplate(str(pdf_path), pagesize=A4,
                                   leftMargin=2*cm, rightMargin=2*cm,
                                   topMargin=2*cm, bottomMargin=2*cm)
        styles = getSampleStyleSheet()
        story  = []

        title_style = ParagraphStyle("Title2", parent=styles["Title"],
                                     fontSize=16, spaceAfter=12, textColor=colors.HexColor("#1a237e"))
        h2_style    = ParagraphStyle("H2", parent=styles["Heading2"],
                                     fontSize=13, textColor=colors.HexColor("#283593"))
        body_style  = styles["Normal"]

        story.append(Paragraph("BPN Peru ML Framework Q1 Extremo V3", title_style))
        story.append(Paragraph(f"Fecha: {date.today()} | Dataset: CNV-MINSA-Peru", body_style))
        story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor("#3949ab")))
        story.append(Spacer(1, 0.4*cm))

        story.append(Paragraph("1. Dataset Summary", h2_style))
        ds_data = [
            ["Métrica", "Valor"],
            ["Registros originales", f"{n_original:,}"],
            ["Registros válidos", f"{df.height:,}"],
            ["Retención", f"{retencion:.2%}"],
            ["Tasa BPN", f"{bpn:.2%}"],
            ["Años cubiertos", f"{anios[0]}–{anios[-1]}"],
            ["Checksum MD5", checksum],
        ]
        t = Table(ds_data, colWidths=[8*cm, 8*cm])
        t.setStyle(TableStyle([
            ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#3949ab")),
            ("TEXTCOLOR",  (0,0), (-1,0), colors.white),
            ("GRID",       (0,0), (-1,-1), 0.5, colors.grey),
            ("FONTSIZE",   (0,0), (-1,-1), 9),
            ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, colors.HexColor("#e8eaf6")]),
        ]))
        story.append(t); story.append(Spacer(1, 0.4*cm))

        story.append(Paragraph("2. Rendimiento de Modelos – Holdout 2025", h2_style))
        perf_data = [["Modelo","AUC","PR-AUC","Recall","F1","MCC","Brier"]]
        for m_name in active_models:
            if m_name not in METRICS_FULL:
                continue
            pt = METRICS_FULL[m_name]["point"]
            perf_data.append([
                MODEL_LABELS.get(m_name,m_name), f"{pt['AUC']:.4f}", f"{pt['PR_AUC']:.4f}",
                f"{pt['Recall']:.4f}", f"{pt['F1']:.4f}", f"{pt['MCC']:.4f}", f"{pt['Brier']:.4f}",
            ])
        t2 = Table(perf_data, colWidths=[4.5*cm,2.2*cm,2.2*cm,2.2*cm,2.2*cm,2.2*cm,2.2*cm])
        t2.setStyle(TableStyle([
            ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#3949ab")),
            ("TEXTCOLOR",  (0,0), (-1,0), colors.white),
            ("GRID",       (0,0), (-1,-1), 0.5, colors.grey),
            ("FONTSIZE",   (0,0), (-1,-1), 8),
            ("ROWBACKGROUNDS",(0,1),(-1,-1),[colors.white, colors.HexColor("#e8eaf6")]),
        ]))
        story.append(t2); story.append(Spacer(1, 0.4*cm))

        story.append(Paragraph(
            f"Mejor modelo: <b>{MODEL_LABELS.get(best_model,best_model)}</b> "
            f"(AUC = {METRICS_FULL[best_model]['point']['AUC']:.4f})", body_style))
        story.append(Spacer(1, 0.4*cm))
        story.append(Paragraph("3. Módulos implementados", h2_style))
        modules_text = (
            "M1 Dataset Audit · M2 Data Drift · M3 Leakage Checker · "
            "M4 Feature Engineering · M5 Feature Stability · M6 Optuna adaptativo · "
            "M7 LOYO Validation · M8 Modelos Extendidos (+ early stopping) · M9 Calibración · "
            "M10 Threshold Opt. · M11 Métricas CI95% · M12 SHAP Completo · "
            "M13 Error Analysis · M14 Fairness · M15 Ablation Study · "
            "M16 Robustez · M17 External Validation · M18 Reportes · "
            "M19 DOI Package · M20 Paper Package · M21 Learning Curves · "
            "M22 Permutation Importance · M23 Export Predicciones"
        )
        story.append(Paragraph(modules_text, body_style))
        story.append(Spacer(1, 1*cm))
        story.append(Paragraph(
            f"<i>Generado automáticamente por BPN_Peru_Q1_Extremo_V3_5 · "
            f"{datetime.now().strftime('%Y-%m-%d %H:%M')}</i>",
            ParagraphStyle("footer", parent=body_style, fontSize=8,
                           textColor=colors.grey, alignment=TA_CENTER)))

        doc.build(story)
        print(f"  PDF:      {pdf_path}")
    except Exception as ep:
        print(f"  [WARN] PDF Report: {ep}")

    print("  Módulo 18 completado.")
except Exception as e:
    print(f"  [WARN] Módulo 18 error: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 19 · DOI / REPRODUCIBILIDAD (README, LICENSE, CITATION, Dockerfile, etc.)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 19 · DOI / REPRODUCIBILIDAD PACKAGE")
print("═" * 60)

try:
    repro_dir = DIRS["18_REPRODUCIBILITY"]

    readme_content = f"""# BPN Peru Q1 Extremo V3 – Low Birth Weight Prediction

## Overview
Machine learning pipeline for prenatal prediction of low birth weight (LBW < 2,500 g)
using Peru's national birth registry (CNV-MINSA, 2015–2025).

**Best Model:** {MODEL_LABELS.get(best_model, best_model)}
**AUC-ROC:** {METRICS_FULL[best_model]['point']['AUC']:.4f}
(95% CI [{METRICS_FULL[best_model]['ci']['AUC']['ci_lo']:.4f}–{METRICS_FULL[best_model]['ci']['AUC']['ci_hi']:.4f}])

## Dataset
- Source: Certificado de Nacido Vivo (CNV), MINSA Peru
- Records: {df.height:,} (after exclusion criteria)
- Training: 2015–2024 | Holdout: 2025
- BPN rate: {bpn:.2%}

## Requirements
```
pip install -r requirements.txt
```

## Usage
Run all cells in `BPN_Peru_Q1_Extremo_V3_5_1.py` in Google Colab.
Data path: `/content/drive/MyDrive/dataset2026varios/*.parquet`

## Modules
23 modules implemented: Dataset Audit, Data Drift, Leakage Checker,
Feature Engineering, Feature Stability, Optuna adaptativo (con early stopping),
LOYO Validation, Extended Models, Calibration, Threshold Optimization,
Full Metrics + CI95%, SHAP Explainability, Error Analysis, Fairness Audit,
Ablation Study, Robustness, External Validation, Reports, DOI Package,
Paper Package, Learning Curves, Permutation Importance, Prediction Export.

## Citation
See CITATION.cff

## License
MIT License – see LICENSE
"""
    (repro_dir / "README.md").write_text(readme_content, encoding="utf-8")

    (repro_dir / "LICENSE").write_text(
        f"MIT License\n\nCopyright (c) {date.today().year} Dr. Evangelista Gamarra et al.\n\n"
        "Permission is hereby granted, free of charge, to any person obtaining a copy\n"
        "of this software and associated documentation files (the 'Software'), to deal\n"
        "in the Software without restriction, including without limitation the rights\n"
        "to use, copy, modify, merge, publish, distribute, sublicense, and/or sell\n"
        "copies of the Software.", encoding="utf-8")

    req_content = "\n".join([
        "polars>=0.20", "pyarrow>=14", "numpy>=1.24", "pandas>=2.0",
        "scikit-learn>=1.4", "xgboost>=2.0", "lightgbm>=4.0", "catboost>=1.2",
        "optuna>=3.5", "optuna-integration", "shap>=0.44",
        "mlflow>=2.10", "matplotlib>=3.8", "seaborn>=0.13", "scipy>=1.12",
        "joblib", "imbalanced-learn>=0.12", "reportlab>=4.0",
        "jinja2>=3.0", "openpyxl>=3.1", "statsmodels>=0.14",
    ])
    (repro_dir / "requirements.txt").write_text(req_content, encoding="utf-8")

    env_yml = f"""name: bpn_peru_q1
channels:
  - defaults
  - conda-forge
dependencies:
  - python=3.11
  - pip
  - pip:
{chr(10).join(['    - ' + p for p in req_content.split(chr(10))])}
"""
    (repro_dir / "environment.yml").write_text(env_yml, encoding="utf-8")

    dockerfile = f"""FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY BPN_Peru_Q1_Extremo_V3_5_1.py .
CMD ["python", "BPN_Peru_Q1_Extremo_V3_5_1.py"]
"""
    (repro_dir / "Dockerfile").write_text(dockerfile, encoding="utf-8")

    citation_cff = f"""cff-version: 1.2.0
message: "If you use this software, please cite it as below."
title: "BPN Peru Q1 Extremo V3 – Low Birth Weight Prediction Pipeline"
version: "3.5.0"
date-released: "{date.today()}"
type: software
authors:
  - family-names: Evangelista Gamarra
    given-names: Dr.
repository-code: "https://github.com/placeholder/bpn-peru-q1"
license: MIT
keywords:
  - low birth weight
  - machine learning
  - Peru
  - birth registry
  - MINSA
  - CNV
  - prenatal prediction
"""
    (repro_dir / "CITATION.cff").write_text(citation_cff, encoding="utf-8")

    metadata = {
        "title"           : "BPN Peru Q1 Extremo V3",
        "version"         : "3.5.0",
        "date"            : str(date.today()),
        "dataset"         : "CNV-MINSA-Peru 2015-2025",
        "n_records"       : int(df.height),
        "bpn_rate"        : float(bpn),
        "best_model"      : best_model,
        "best_model_label": MODEL_LABELS.get(best_model, best_model),
        "best_auc"        : float(METRICS_FULL[best_model]["point"]["AUC"]),
        "models_trained"  : MODEL_NAMES,
        "n_features"      : len(FEATURES),
        "n_modules"       : 23,
        "random_state"    : RANDOM_STATE,
        "n_boot"          : N_BOOT,
        "n_trials_by_model": N_TRIALS_BY_MODEL,
        "early_stopping_models": list(EARLY_STOP_MODELS),
        "checksum"        : checksum,
    }
    (repro_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    pipeline_json = {
        "pipeline_name"   : "BPN_Peru_Q1_Extremo_V3_5",
        "train_years"     : "2015-2024",
        "test_year"       : "2025",
        "features"        : FEATURES,
        "target"          : TARGET,
        "models"          : MODEL_NAMES,
        "preprocessing"   : {
            "num_imputer"   : "median", "num_scaler": "StandardScaler",
            "cat_imputer"   : "most_frequent",
            "cat_encoder"   : "OneHotEncoder(min_frequency=50)",
        },
        "nested_cv"       : {"outer_folds": N_OUTER, "inner_folds": N_INNER,
                              "n_trials_by_model": N_TRIALS_BY_MODEL,
                              "n_ncv_sample": N_NCV, "timeout_sec": OPTUNA_TIMEOUT_SEC},
        "modules"         : [f"M{i:02d}" for i in range(1, 24)],
    }
    (repro_dir / "pipeline.json").write_text(json.dumps(pipeline_json, indent=2), encoding="utf-8")

    zenodo_meta = {
        "title"       : "BPN Peru Q1 Extremo V3: Low Birth Weight ML Pipeline",
        "upload_type" : "software",
        "description" : (
            "Complete Q1-grade machine learning pipeline for prenatal prediction "
            "of low birth weight using Peru's CNV-MINSA registry (2015–2025). "
            "Implements 23 methodological modules including dataset audit, "
            "data drift, feature engineering, nested CV with early stopping, "
            "calibration, SHAP explainability, fairness, ablation, learning "
            "curves, permutation importance, and automated reports."),
        "creators"    : [{"name": "Evangelista Gamarra", "affiliation": "Peru"}],
        "keywords"    : ["low birth weight","machine learning","Peru","CNV","prenatal"],
        "license"     : "MIT", "language": "spa", "version": "3.5.0",
    }
    (repro_dir / "zenodo.json").write_text(json.dumps(zenodo_meta, indent=2), encoding="utf-8")

    print(f"  DOI package en: {repro_dir}")
    print("  Módulo 19 completado.")
except Exception as e:
    print(f"  [WARN] Módulo 19 error: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 20 · PAPER PACKAGE (estructura completa para revista)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 20 · PAPER PACKAGE")
print("═" * 60)

try:
    paper_root = BASE_OUT / "PAPER_PACKAGE"
    for sub in ["paper", "Figures", "Tables", "Supplementary", "Appendix"]:
        (paper_root / sub).mkdir(parents=True, exist_ok=True)

    import shutil
    for fig_file in DIRS["figures"].glob("*.png"):
        shutil.copy2(fig_file, paper_root / "Figures")
    for subdir_name in ["11_SHAP","12_FAIRNESS","13_DRIFT","14_ERROR_ANALYSIS",
                         "10_CALIBRATION","15_ABLATION","21_LEARNING_CURVES",
                         "22_PERMUTATION_IMPORTANCE"]:
        for fig_file in DIRS[subdir_name].glob("*.png"):
            shutil.copy2(fig_file, paper_root / "Figures")

    for tbl_file in DIRS["tables"].glob("*.csv"):
        shutil.copy2(tbl_file, paper_root / "Tables")
    for subdir_name in ["08_METRICS","07_NESTED_CV","10_CALIBRATION",
                         "12_FAIRNESS","14_ERROR_ANALYSIS","15_ABLATION",
                         "21_LEARNING_CURVES","22_PERMUTATION_IMPORTANCE"]:
        for tbl_file in DIRS[subdir_name].glob("*.csv"):
            shutil.copy2(tbl_file, paper_root / "Tables")

    pt_b = METRICS_FULL[best_model]["point"]
    ci_b = METRICS_FULL[best_model]["ci"]
    nr_b = nested_results.get(best_model, {})
    n_train = len(df_tr); n_test = len(df_te)
    bpn_train = round(y_tr.mean()*100, 2)
    bpn_test  = round(y_te.mean()*100, 2)
    nested_str = (f"(nested CV AUC = {nr_b['mean_auc']:.4f} ± {nr_b['sd_auc']:.4f})"
                  if "mean_auc" in nr_b else "")

    METHODS_TEXT = f"""METHODS
=======

Study Design and Data Source
-----------------------------
Population-based retrospective cohort study using Peru's Certificado de Nacido Vivo
(CNV) registry (MINSA), covering all registered live births 2015–2025
(N = {n_train + n_test:,} records after exclusion criteria).

Outcome
-------
Low birth weight (LBW): birth weight < 2,500 g (binary classification, TARGET=BAJO_PESO).

Temporal Validation Design
---------------------------
Training: 2015–2024 (n = {n_train:,}; LBW rate {bpn_train}%).
Holdout: 2025 (n = {n_test:,}; LBW rate {bpn_test}%).
No 2025 data used during model development.

Feature Engineering
-------------------
{len([c for c in NUM_PRENATAL if c in df_tr.columns])} numerical + {len([c for c in CAT_PRENATAL if c in df_tr.columns])} categorical prenatal features.
Engineered features: ROA, TPF, EM, PRIMIGESTA, GRAN_MULTIPARA, RIESGO_EXTREMO,
ANTECEDENTE_PERDIDA, EDAD_x_ROA, EDAD_x_EMB, ROA_x_PRIMIG, TPF_x_RIESGO, EDAD2, ROA2.

Models Compared
---------------
{', '.join([MODEL_LABELS.get(m,m) for m in MODEL_NAMES])}

Nested Cross-Validation
-----------------------
{N_OUTER}-outer × {N_INNER}-inner folds. Optuna (TPE + MedianPruner), trials
adaptativos por modelo ({N_TRIALS_BY_MODEL}), timeout={OPTUNA_TIMEOUT_SEC}s por estudio.
Early stopping ({EARLY_STOP_ROUNDS} rounds) para XGBoost, LightGBM y CatBoost en el
ajuste final, usando un recorte de validación interno del propio conjunto de train.

Statistical Evaluation
----------------------
{N_BOOT}-bootstrap 95% CI. DeLong test (pairwise AUC). McNemar test (predictions).
Metrics: AUC, PR-AUC, Recall, Specificity, Precision, NPV, PPV, F1, Balanced Accuracy,
MCC, Cohen's Kappa, Brier Score, LogLoss.

Calibration
-----------
ECE, MCE, Hosmer-Lemeshow, Temperature Scaling, Isotonic Regression, Platt Scaling.

Fairness Audit
--------------
Subgroups: sex, geographic region, maternal age group.
Metrics: AUC, Recall, FPR, PPV, Demographic Parity, Equal Opportunity,
Disparate Impact ratio, intersectional fairness.

Drift Analysis
--------------
PSI, KS, Jensen-Shannon divergence, Wasserstein distance, Energy distance (per year).

Additional Diagnostics
-----------------------
Learning curves (train size vs AUC train/holdout) and permutation importance
(ΔAUC, 10 repeats) complementing SHAP-based explainability.

Software
--------
Python {platform.python_version()}. scikit-learn, XGBoost, LightGBM, CatBoost,
Optuna, SHAP, MLflow. RANDOM_STATE = {RANDOM_STATE}.
"""

    RESULTS_TEXT = f"""RESULTS
=======

Dataset Characteristics
-----------------------
After exclusion: {n_train+n_test:,} records.
Training (2015–2024): {n_train:,} births (LBW={bpn_train}%).
Holdout (2025): {n_test:,} births (LBW={bpn_test}%).

Best Model: {MODEL_LABELS.get(best_model, best_model)} {nested_str}

Holdout Performance:
  AUC:              {pt_b['AUC']:.4f} (95% CI {ci_b['AUC']['ci_lo']:.4f}–{ci_b['AUC']['ci_hi']:.4f})
  PR-AUC:           {pt_b['PR_AUC']:.4f} (95% CI {ci_b['PR_AUC']['ci_lo']:.4f}–{ci_b['PR_AUC']['ci_hi']:.4f})
  Recall:           {pt_b['Recall']:.4f} (95% CI {ci_b['Recall']['ci_lo']:.4f}–{ci_b['Recall']['ci_hi']:.4f})
  Precision (PPV):  {pt_b['Precision']:.4f} (95% CI {ci_b['Precision']['ci_lo']:.4f}–{ci_b['Precision']['ci_hi']:.4f})
  F1-score:         {pt_b['F1']:.4f} (95% CI {ci_b['F1']['ci_lo']:.4f}–{ci_b['F1']['ci_hi']:.4f})
  MCC:              {pt_b['MCC']:.4f} (95% CI {ci_b['MCC']['ci_lo']:.4f}–{ci_b['MCC']['ci_hi']:.4f})
  Brier Score:      {pt_b['Brier']:.4f} (95% CI {ci_b['Brier']['ci_lo']:.4f}–{ci_b['Brier']['ci_hi']:.4f})

Full metrics for all models in Table 1. Pairwise tests in Tables 2–3.
"""

    LIMITATIONS_TEXT = """LIMITATIONS
===========
1. Administrative data quality: CNV has measurement error and variable missingness.
2. Feature completeness: gestational age at booking, pre-pregnancy BMI, haemoglobin
   not captured; constrains theoretical AUC ceiling.
3. Temporal generalizability: COVID-19 (2020–2021) may have altered registration patterns.
4. External validity: single national registry; LMIC generalizability uncertain.
5. Threshold optimization on holdout: mild optimistic bias for F1, Recall, Precision.
6. Intersectional fairness: small-cell counts limit some subgroup analyses.
7. Causal inference: SHAP and permutation importance report associations, not causal
   contributions.
8. Hyperparameter search reduced (adaptive trial budgets, narrower ranges) for
   computational efficiency; nested CV mean AUC used for model ranking mitigates,
   but does not eliminate, this trade-off.
"""

    paper_text_path = paper_root / "paper" / "paper_sections.txt"
    with open(paper_text_path, "w", encoding="utf-8") as f:
        f.write("=" * 80 + "\n")
        f.write(f"AUTO-GENERATED PAPER SECTIONS — {date.today()}\n")
        f.write("=" * 80 + "\n\n")
        f.write(METHODS_TEXT + "\n\n")
        f.write(RESULTS_TEXT + "\n\n")
        f.write(LIMITATIONS_TEXT)
    print(f"  Paper sections: {paper_text_path}")

    ncv_sup_rows = []
    for m_name in MODEL_NAMES:
        nr = nested_results.get(m_name, {})
        ncv_sup_rows.append({
            "Model"   : MODEL_LABELS.get(m_name, m_name),
            "NCV_AUC_mean": round(nr.get("mean_auc", np.nan), 4),
            "NCV_AUC_SD"  : round(nr.get("sd_auc", np.nan), 4),
            "NCV_AUC_CI95": f"[{nr.get('ci_lo',np.nan):.4f}–{nr.get('ci_hi',np.nan):.4f}]"
                             if "ci_lo" in nr else "—",
        })
    pd.DataFrame(ncv_sup_rows).to_csv(
        paper_root / "Supplementary" / "TableS1_NestedCV.csv", index=False)

    feat_app = pd.DataFrame({
        "Feature": FEATURES,
        "Type"   : ["Numerical" if f in NUM_PRENATAL else "Categorical" for f in FEATURES],
    })
    feat_app.to_csv(paper_root / "Appendix" / "AppendixA_FeatureList.csv", index=False)

    all_paper_files = list(paper_root.rglob("*"))
    paper_manifest = {
        "generated_at"  : str(datetime.now()),
        "paper_root"    : str(paper_root),
        "n_files"       : len([f for f in all_paper_files if f.is_file()]),
        "best_model"    : best_model,
        "best_auc"      : float(pt_b["AUC"]),
    }
    with open(paper_root / "manifest.json", "w") as f:
        json.dump(paper_manifest, f, indent=2)

    print(f"  Paper Package en: {paper_root}")
    print("  Módulo 20 completado.")
except Exception as e:
    print(f"  [WARN] Módulo 20 error: {e}")

# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 21 · LEARNING CURVES (train size vs AUC train/val)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 21 · LEARNING CURVES")
print("═" * 60)

if RESUME.get("learning_curves_done") and (DIRS["21_LEARNING_CURVES"] / "M21_learning_curve.csv").exists():
    print("  [RESUME] Módulo 21 (Learning Curves) ya completado en una corrida anterior — "
          "se omite (evita reentrenar 6 modelos adicionales a distintas fracciones de train).")
else:
    try:
        # FIX tiempo/RAM: 6 fracciones -> 3 (0.2/0.6/1.0). Con solo 3 puntos
        # ya se ve con claridad si hay overfitting (gap train-val que no
        # cierra) o si el modelo satura antes de usar el 100% del train —
        # que es justamente lo que esta figura busca mostrar para el paper.
        frac_grid = [0.2, 0.6, 1.0]
        lc_rows = []
        params_lc = best_params_for(best_model) or FALLBACK_PARAMS.get(best_model, {})

        for frac in frac_grid:
            if frac >= 0.999:
                # FIX RAM/tiempo: frac=1.0 usa (una versión barajada de) el
                # dataset COMPLETO — exactamente los mismos datos con los que
                # FINAL_PIPES[best_model] ya fue entrenado en la Celda 8.
                # Reentrenar aquí era trabajo 100% duplicado, y encima es
                # justo el punto que agotaba la RAM (fit() de un modelo
                # completo sobre 4.5M filas, la operación más pesada de todo
                # el pipeline). Se reutiliza el modelo ya entrenado: nunca
                # se vuelve a hacer fit(), y el AUC en holdout es EXACTAMENTE
                # el de Table 1 (más correcto que reentrenar un modelo nuevo
                # con una semilla distinta, que además podría diferir
                # levemente del resultado ya reportado como principal).
                n_sub   = len(X_tr_f)
                p_train = FINAL_PIPES[best_model].predict_proba(X_tr_f)[:, 1]
                p_val   = PROBA_TE[best_model]  # ya calculado en Celda 8, no se recalcula
                auc_train = roc_auc_score(y_tr, p_train) if len(np.unique(y_tr)) > 1 else np.nan
                auc_val   = roc_auc_score(y_te_arr, p_val) if len(np.unique(y_te_arr)) > 1 else np.nan
                lc_rows.append({"Train_fraction": frac, "N_train": n_sub,
                                 "AUC_train": round(auc_train, 4), "AUC_val": round(auc_val, 4),
                                 "Gap": round(auc_train - auc_val, 4)})
                print(f"  frac={frac:.1f}  N={n_sub:,}  AUC_train={auc_train:.4f}  "
                      f"AUC_val={auc_val:.4f}  [reutiliza modelo final, sin reentrenar]")
                del p_train
                gc.collect()
                continue

            n_sub = max(int(len(X_tr_f) * frac), 200)
            idx_sub = df_tr.sample(n_sub, random_state=RANDOM_STATE).index
            X_sub, y_sub = df_tr.loc[idx_sub, FEATURES], y_tr.loc[idx_sub]

            clf_lc  = make_clf(best_model, params_lc)
            pre_lc  = build_preprocessor()
            pipe_lc = Pipeline([("pre", pre_lc), ("clf", clf_lc)])
            pipe_lc.fit(X_sub, y_sub)

            p_train = pipe_lc.predict_proba(X_sub)[:, 1]
            p_val   = pipe_lc.predict_proba(X_te_f)[:, 1]

            auc_train = roc_auc_score(y_sub, p_train) if len(np.unique(y_sub)) > 1 else np.nan
            auc_val   = roc_auc_score(y_te_arr, p_val) if len(np.unique(y_te_arr)) > 1 else np.nan

            lc_rows.append({"Train_fraction": frac, "N_train": n_sub,
                             "AUC_train": round(auc_train, 4), "AUC_val": round(auc_val, 4),
                             "Gap": round(auc_train - auc_val, 4)})
            print(f"  frac={frac:.1f}  N={n_sub:,}  AUC_train={auc_train:.4f}  AUC_val={auc_val:.4f}")

            # Libera el pipeline/modelo/muestra de esta fracción antes de
            # pasar a la siguiente (cada uno entrena sobre hasta 4.5M filas).
            del X_sub, y_sub, pipe_lc, clf_lc, pre_lc, p_train, p_val
            gc.collect()


        df_lc = pd.DataFrame(lc_rows)
        save_table_apa(df_lc, "M21_learning_curve.csv",
                       f"Módulo 21. Learning Curve – {MODEL_LABELS.get(best_model,best_model)}",
                       subdir="21_LEARNING_CURVES")

        fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)
        ax.plot(df_lc["N_train"], df_lc["AUC_train"], "o-", lw=2, color=PAL[0], label="Train AUC")
        ax.plot(df_lc["N_train"], df_lc["AUC_val"], "o-", lw=2, color=PAL[1], label="Holdout 2025 AUC")
        ax.fill_between(df_lc["N_train"], df_lc["AUC_train"], df_lc["AUC_val"],
                         alpha=0.15, color=PAL[3])
        ax.set_xlabel("Training set size"); ax.set_ylabel("AUC")
        ax.set_title(f"Módulo 21 – Learning Curve ({MODEL_LABELS.get(best_model,best_model)})",
                     fontweight="bold")
        ax.legend(fontsize=8)
        plt.tight_layout()
        save_fig(fig, "M21_learning_curve.png", subdir="21_LEARNING_CURVES")
        plt.close()
        mark_stage_done("learning_curves_done", RESUME)
        print("  Módulo 21 completado.")
    except Exception as e:
        print(f"  [WARN] Módulo 21 error: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 22 · PERMUTATION IMPORTANCE (complementa SHAP)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 22 · PERMUTATION IMPORTANCE")
print("═" * 60)

try:
    from sklearn.inspection import permutation_importance

    # FIX tiempo/RAM: N_PERM 5000->3000 y n_repeats 10->5. Con n_jobs=-1 cada
    # núcleo mantiene su propia copia de trabajo de X_perm — reducir tamaño
    # de muestra Y repeticiones baja la RAM pico y el tiempo total sin
    # comprometer la estabilidad de la importancia estimada (el error
    # estándar entre repeticiones ya se reporta en la figura/tabla).
    N_PERM = min(3000, len(X_te_f))
    idx_perm = X_te_f.sample(N_PERM, random_state=RANDOM_STATE).index
    X_perm, y_perm = X_te_f.loc[idx_perm], y_te.loc[idx_perm]

    perm_res = permutation_importance(
        FINAL_PIPES[best_model], X_perm, y_perm,
        scoring="roc_auc", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
    )

    df_perm = pd.DataFrame({
        "Feature": X_perm.columns,
        "Importance_Mean": perm_res.importances_mean,
        "Importance_STD": perm_res.importances_std,
    }).sort_values("Importance_Mean", ascending=False)

    save_table_apa(df_perm, "M22_permutation_importance.csv",
                   f"Módulo 22. Permutation Importance (ΔAUC) – {MODEL_LABELS.get(best_model,best_model)}",
                   subdir="22_PERMUTATION_IMPORTANCE")

    fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)
    top_perm = df_perm.head(15).iloc[::-1]
    ax.barh(top_perm["Feature"], top_perm["Importance_Mean"],
            xerr=top_perm["Importance_STD"], color=PAL[0], edgecolor="white")
    ax.set_xlabel("Permutation Importance (ΔAUC)")
    ax.set_title(f"Módulo 22 – Permutation Importance – {MODEL_LABELS.get(best_model,best_model)}",
                 fontweight="bold")
    plt.tight_layout()
    save_fig(fig, "M22_permutation_importance.png", subdir="22_PERMUTATION_IMPORTANCE")
    plt.close()
    print("  Módulo 22 completado.")
    del X_perm, y_perm, perm_res
    gc.collect()
except Exception as e:
    print(f"  [WARN] Módulo 22 error: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 23 · EXPORT DE PREDICCIONES (train/test/probabilidades todos los modelos)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  MÓDULO 23 · EXPORT DE PREDICCIONES")
print("═" * 60)

try:
    thr_b = THRESHOLDS.get(best_model, 0.5)
    pred_train_best = FINAL_PIPES[best_model].predict_proba(X_tr_f)[:, 1]

    df_pred_train = pd.DataFrame({
        "y_true": y_tr.values, "y_prob": pred_train_best,
        "y_pred": (pred_train_best >= thr_b).astype(int),
    })
    df_pred_test = pd.DataFrame({
        "y_true": y_te_arr, "y_prob": PROBA_TE[best_model], "y_pred": pred_best,
    })
    save_table_apa(df_pred_train, "predictions_train.csv",
                   f"Predicciones en train – {MODEL_LABELS.get(best_model,best_model)}",
                   subdir="23_PREDICTIONS")
    save_table_apa(df_pred_test, "predictions_test.csv",
                   f"Predicciones en holdout 2025 – {MODEL_LABELS.get(best_model,best_model)}",
                   subdir="23_PREDICTIONS")

    df_proba_all = pd.DataFrame({"y_true": y_te_arr})
    for m_name in PROBA_TE:
        df_proba_all[f"proba_{m_name}"] = PROBA_TE[m_name]
    save_table_apa(df_proba_all, "probabilities_all_models.csv",
                   "Probabilidades predichas en holdout 2025 – todos los modelos",
                   subdir="23_PREDICTIONS")
    print(f"  Predicciones exportadas: {DIRS['23_PREDICTIONS']}")
    print("  Módulo 23 completado.")
    # Libera las estructuras grandes (predicción sobre las ~4.5M filas de
    # train) apenas terminan de guardarse — es el último módulo, pero dejar
    # esto vivo hasta el empaquetado final (ZIP) no aporta nada.
    del pred_train_best, df_pred_train, df_pred_test, df_proba_all
    gc.collect()
except Exception as e:
    print(f"  [WARN] Módulo 23 error: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# CELDA FINAL · RESUMEN EJECUTIVO Y MANIFIESTO DE ARTEFACTOS
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  STUDY SUMMARY — BPN Peru ML Framework Q1 Extremo V3")
print("=" * 70)
print(f"\nDataset: {len(df_tr)+len(df_te):,} records "
      f"({len(df_tr):,} train / {len(df_te):,} test)")
print(f"LBW rate: train {y_tr.mean()*100:.2f}%  |  test {y_te.mean()*100:.2f}%")

print(f"\nNested CV ({N_OUTER}×{N_INNER}, trials adaptativos):")
for m in MODEL_NAMES:
    nr = nested_results.get(m, {})
    if "mean_auc" in nr:
        print(f"  {MODEL_LABELS.get(m,m):28s}  AUC = {nr['mean_auc']:.4f} ± {nr['sd_auc']:.4f}")

print(f"\nHoldout 2025 ({N_BOOT}-bootstrap 95% CI):")
print(f"{'Model':28s}  {'AUC':>8s}  {'PR-AUC':>8s}  {'Recall':>8s}  {'F1':>8s}  {'MCC':>8s}")
print("-" * 70)
for m in active_models:
    if m not in METRICS_FULL:
        continue
    pt = METRICS_FULL[m]["point"]
    marker = " ◄ BEST" if m == best_model else ""
    print(f"{MODEL_LABELS.get(m,m):28s}  {pt['AUC']:>8.4f}  "
          f"{pt['PR_AUC']:>8.4f}  {pt['Recall']:>8.4f}  "
          f"{pt['F1']:>8.4f}  {pt['MCC']:>8.4f}{marker}")

print("\nARTEFACTOS GENERADOS:")
print("-" * 70)
all_files = []
for k, d in DIRS.items():
    for fpath in sorted(d.glob("*")):
        if fpath.is_file():
            print(f"  [{k:25s}]  {fpath.name}")
            all_files.append(str(fpath))

manifest = {
    "generated_at" : str(datetime.now()),
    "pipeline"     : "BPN_Peru_Q1_Extremo_V3_5",
    "files"        : all_files,
    "n_files"      : len(all_files),
    "best_model"   : best_model,
    "best_auc"     : float(METRICS_FULL[best_model]["point"]["AUC"]),
    "modules_completed": 23,
}
save_json(manifest, "manifest.json", subdir="metrics")

print("\n✅  Pipeline Q1 Extremo V3 completo — 23 módulos ejecutados.")
print(f"    MLflow UI: mlflow ui --backend-store-uri {MLFLOW_DIR}")
print(f"    Outputs  : {BASE_OUT.resolve()}")


# ══════════════════════════════════════════════════════════════════════════════
# EMPAQUETADO FINAL · ZIP + AUTO-DESCARGA
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  EMPAQUETADO FINAL · ZIP + AUTO-DESCARGA")
print("═" * 60)

try:
    import shutil as _shutil

    zip_base_name = "BPN_PERU_Q1_FINAL"
    zip_path = _shutil.make_archive(zip_base_name, "zip", root_dir=str(BASE_OUT))
    print(f"  ZIP creado: {zip_path}  ({os.path.getsize(zip_path)/1024**2:.2f} MB)")

    try:
        from google.colab import files as _colab_files
        _colab_files.download(zip_path)
        print("  Descarga automática iniciada (Google Colab).")
    except Exception:
        print("  (Auto-descarga solo disponible en Google Colab; "
              f"el ZIP quedó guardado localmente en: {zip_path})")
except Exception as e:
    print(f"  [WARN] Empaquetado ZIP error: {e}")

print("\n✅  BPN_PERU_Q1_FINAL.zip listo — pipeline y empaquetado completos.")

Dependencias listas.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATASET CONFIGURADO
Tipo          : PARQUET
Ruta          : /content/drive/MyDrive/dataset2026varios/CNV_MINSA_CORTE_30112025.parquet
Existe        : True
Tamaño (MB)   : 41.45
Columnas      : 22
Primeras columnas:
['FecNac_Año', 'FecNac_Mes', 'PESO_NACIDO', 'TALLA_NACIDO', 'DUR_EMB_PARTO', 'Condicion_Parto', 'sexo_nacido', 'Tipo_Parto', 'Edad_Madre', 'Estado_Civil']
Entorno listo. Directorios creados.
BASE_OUT (Google Drive, persistente): /content/drive/MyDrive/dataset2026varios/Archivos_RN_2026
Modo: RESUME (conserva resultados previos)
  [RESUME] Checkpoint previo encontrado. Etapas completas: ['ablation_done', 'bootstrap_ci_done', 'data_loaded', 'features_engineered', 'final_models_trained', 'loyo_done']
INFORMACIÓN DEL ENTORNO DE EJECUCIÓN
Entorno       : Google Colab
Hostname      : 434af48517f6
Plataforma    : Linux-6.6.122+-x86_64

PermutationExplainer explainer: 1001it [02:16,  6.94it/s]


  [SHAP] shap_values listos en 136.1s.
  Saved: /content/drive/MyDrive/dataset2026varios/Archivos_RN_2026/11_SHAP/M12_SHAP_beeswarm.png
  Saved: /content/drive/MyDrive/dataset2026varios/Archivos_RN_2026/11_SHAP/M12_SHAP_importance.png
  [WARN] SHAP Waterfall: The waterfall plot can currently only plot a single explanation, but a matrix of explanations (shape (56, 2)) was passed! Perhaps try `shap.plots.waterfall(shap_values[0])` or for multi-output models, try `shap.plots.waterfall(shap_values[0, 0])`.
  [WARN] SHAP Decision Plot: 'PermutationExplainer' object has no attribute 'expected_value'
  Saved: /content/drive/MyDrive/dataset2026varios/Archivos_RN_2026/11_SHAP/M12_SHAP_dependence.png
  Saved: /content/drive/MyDrive/dataset2026varios/Archivos_RN_2026/11_SHAP/M12_SHAP_ranking.csv
  Módulo 12 completado.
  Saved: /content/drive/MyDrive/dataset2026varios/Archivos_RN_2026/figures/Fig10_PDP.png
  Saved: /content/drive/MyDrive/dataset2026varios/Archivos_RN_2026/figures/Fig11_ICE.png

═

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Descarga automática iniciada (Google Colab).

✅  BPN_PERU_Q1_FINAL.zip listo — pipeline y empaquetado completos.


<Figure size 1065x750 with 0 Axes>

## visor

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   BPN PERU Q1 EXTREMO V3.6 — VISOR DE RESULTADOS (script independiente)  ║
# ║   No recalcula NADA. Solo recorre Google Drive y muestra lo que ya       ║
# ║   generó BPN_Peru_Q1_Extremo_V3_6.py: imágenes, CSV, Excel, JSON, PDF,   ║
# ║   el catálogo de modelos .pkl guardados y las cantidades por carpeta.    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os
import json
from pathlib import Path
from datetime import datetime

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

import pandas as pd
from IPython.display import display, Image, IFrame, HTML, JSON as IPJSON, Markdown

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════════════════════════════
# Misma ruta que usa BPN_Peru_Q1_Extremo_V3_6.py — este visor NUNCA escribe ahí,
# solo lee.
BASE_OUT = Path("/content/drive/MyDrive/dataset2026varios/Archivos_RN_2026")

# Qué tipos de contenido mostrar (apaga lo que no te interese ver)
MOSTRAR_IMAGENES = True
MOSTRAR_PDFS     = True
MOSTRAR_CSV      = True
MOSTRAR_EXCEL    = True
MOSTRAR_JSON     = True
LISTAR_MODELOS   = True

# Límites por carpeta (subidos para mostrar prácticamente todo el pipeline;
# bájalos si alguna corrida generó demasiados archivos en una sola carpeta).
MAX_IMAGENES_POR_CARPETA = 40
MAX_FILAS_CSV            = 10
MAX_CSV_POR_CARPETA      = 25
MAX_JSON_CHARS           = 3000   # trunca JSON muy largos al imprimir

# None = muestra TODAS las carpetas del pipeline, en orden.
# O especifica una lista para ver solo algunas, p.ej.:
#   CARPETAS_A_MOSTRAR = ["08_METRICS", "11_SHAP", "12_FAIRNESS"]
CARPETAS_A_MOSTRAR = None

# Carpetas del pipeline, en el mismo orden que las genera BPN_Peru_Q1_Extremo_V3_6.py
CARPETAS_PIPELINE = [
    "01_DATASET", "02_EDA", "03_PREPROCESSING", "04_FEATURE_ENGINEERING",
    "05_FEATURE_SELECTION", "06_MODELS", "07_NESTED_CV", "08_METRICS",
    "09_BOOTSTRAP", "10_CALIBRATION", "11_SHAP", "12_FAIRNESS", "13_DRIFT",
    "14_ERROR_ANALYSIS", "15_ABLATION", "16_EXTERNAL_VALIDATION",
    "17_MODEL_CARD", "18_REPRODUCIBILITY", "19_PDF_REPORT", "20_HTML_REPORT",
    "21_LEARNING_CURVES", "22_PERMUTATION_IMPORTANCE", "23_PREDICTIONS",
    "figures", "tables", "metrics", "models", "paper", "PAPER_PACKAGE",
]

IMG_EXT   = {".png", ".jpg", ".jpeg", ".gif", ".webp"}
CSV_EXT   = {".csv"}
EXCEL_EXT = {".xlsx", ".xls"}
JSON_EXT  = {".json"}
PDF_EXT   = {".pdf"}
MODEL_EXT = {".pkl", ".joblib"}


# ══════════════════════════════════════════════════════════════════════════════
# UTILIDADES
# ══════════════════════════════════════════════════════════════════════════════
def _titulo(texto: str, nivel: int = 2):
    display(Markdown(f"{'#' * nivel} {texto}"))

def _formato_bytes(n: int) -> str:
    if n < 1024:
        return f"{n} bytes"
    elif n < 1024**2:
        return f"{n/1024:.1f} KB"
    else:
        return f"{n/1024**2:.2f} MB"

def _listar_por_extension(carpeta: Path, extensiones: set) -> list:
    if not carpeta.exists():
        return []
    return sorted([p for p in carpeta.iterdir()
                   if p.is_file() and p.suffix.lower() in extensiones])


# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZADORES POR TIPO DE ARCHIVO
# ══════════════════════════════════════════════════════════════════════════════
def mostrar_imagenes(carpeta: Path):
    imgs = _listar_por_extension(carpeta, IMG_EXT)
    if not imgs:
        return
    print(f"  📷 {len(imgs)} imagen(es)"
          f"{' (mostrando las primeras ' + str(MAX_IMAGENES_POR_CARPETA) + ')' if len(imgs) > MAX_IMAGENES_POR_CARPETA else ''}:")
    for p in imgs[:MAX_IMAGENES_POR_CARPETA]:
        print(f"    — {p.name}  ({_formato_bytes(p.stat().st_size)})")
        try:
            display(Image(filename=str(p), width=650))
        except Exception as e:
            print(f"      [no se pudo renderizar: {e}]")

def mostrar_pdfs(carpeta: Path):
    pdfs = _listar_por_extension(carpeta, PDF_EXT)
    if not pdfs:
        return
    print(f"  📄 {len(pdfs)} PDF(s):")
    for p in pdfs:
        print(f"    — {p.name}  ({_formato_bytes(p.stat().st_size)})  →  {p}")
        try:
            display(IFrame(src=str(p), width=800, height=500))
        except Exception:
            print(f"      (no se pudo embeber; ábrelo directamente desde Drive: {p})")

def mostrar_csv(carpeta: Path):
    csvs = _listar_por_extension(carpeta, CSV_EXT)
    if not csvs:
        return
    print(f"  📊 {len(csvs)} CSV"
          f"{' (mostrando los primeros ' + str(MAX_CSV_POR_CARPETA) + ')' if len(csvs) > MAX_CSV_POR_CARPETA else ''}:")
    for p in csvs[:MAX_CSV_POR_CARPETA]:
        print(f"    — {p.name}  ({_formato_bytes(p.stat().st_size)})")
        try:
            # Los CSV de este pipeline a veces llevan una línea de comentario
            # "# caption" antes del encabezado real (ver save_table_apa()).
            with open(p, encoding="utf-8") as f:
                primera_linea = f.readline()
            skiprows = 1 if primera_linea.startswith("#") else 0
            df = pd.read_csv(p, skiprows=skiprows)
            print(f"      Filas: {len(df):,}  |  Columnas: {len(df.columns)}")
            display(df.head(MAX_FILAS_CSV))
        except Exception as e:
            print(f"      [no se pudo leer: {e}]")

def mostrar_excel(carpeta: Path):
    excels = _listar_por_extension(carpeta, EXCEL_EXT)
    if not excels:
        return
    print(f"  📈 {len(excels)} Excel:")
    for p in excels:
        print(f"    — {p.name}  ({_formato_bytes(p.stat().st_size)})")
        try:
            xls = pd.ExcelFile(p)
            print(f"      Hojas: {xls.sheet_names}")
            primera_hoja = xls.sheet_names[0]
            display(pd.read_excel(p, sheet_name=primera_hoja).head(MAX_FILAS_CSV))
        except Exception as e:
            print(f"      [no se pudo leer: {e}]")

def mostrar_json(carpeta: Path):
    jsons = _listar_por_extension(carpeta, JSON_EXT)
    if not jsons:
        return
    print(f"  📋 {len(jsons)} JSON:")
    for p in jsons:
        print(f"    — {p.name}  ({_formato_bytes(p.stat().st_size)})")
        try:
            with open(p, encoding="utf-8") as f:
                data = json.load(f)
            texto = json.dumps(data, indent=2, ensure_ascii=False, default=str)
            if len(texto) > MAX_JSON_CHARS:
                print(texto[:MAX_JSON_CHARS] + f"\n      ... [truncado, {len(texto):,} caracteres en total]")
            else:
                print(texto)
        except Exception as e:
            print(f"      [no se pudo leer: {e}]")

def listar_modelos(carpeta: Path):
    modelos = _listar_por_extension(carpeta, MODEL_EXT)
    if not modelos:
        return
    print(f"  📦 {len(modelos)} modelo(s)/objeto(s) guardado(s):")
    filas = []
    for p in modelos:
        stat = p.stat()
        filas.append({
            "Archivo": p.name,
            "Tamaño": _formato_bytes(stat.st_size),
            "Modificado": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M"),
        })
    display(pd.DataFrame(filas))


# ══════════════════════════════════════════════════════════════════════════════
# RESUMEN DEL MEJOR MODELO (lectura directa de 08_METRICS, sin recalcular)
# ══════════════════════════════════════════════════════════════════════════════
def resumen_mejor_modelo():
    metrics_path = BASE_OUT / "08_METRICS" / "metrics_all_models.json"
    table1_path  = BASE_OUT / "08_METRICS" / "Table1_ModelPerformance.csv"
    if not metrics_path.exists() and not table1_path.exists():
        return  # el pipeline no llegó todavía al Módulo 11 — no hay nada que resumir

    _titulo("🏆 Resumen del mejor modelo", nivel=1)

    if table1_path.exists():
        try:
            with open(table1_path, encoding="utf-8") as f:
                primera = f.readline()
            skiprows = 1 if primera.startswith("#") else 0
            df_t1 = pd.read_csv(table1_path, skiprows=skiprows)
            display(df_t1[["Model", "AUC", "PR-AUC", "F1", "MCC"]] if
                    "PR-AUC" in df_t1.columns else df_t1)
        except Exception as e:
            print(f"  (No se pudo leer Table1_ModelPerformance.csv: {e})")

    if metrics_path.exists():
        try:
            with open(metrics_path, encoding="utf-8") as f:
                metrics = json.load(f)
            best = max(metrics.keys(), key=lambda m: metrics[m]["point"]["AUC"])
            pt = metrics[best]["point"]
            print(f"\n  Mejor modelo: {best}")
            print(f"  AUC = {pt['AUC']:.4f}  |  PR-AUC = {pt['PR_AUC']:.4f}  |  "
                  f"F1 = {pt['F1']:.4f}  |  MCC = {pt['MCC']:.4f}  |  "
                  f"Recall = {pt['Recall']:.4f}")
        except Exception as e:
            print(f"  (No se pudo leer metrics_all_models.json: {e})")
    print()


# ══════════════════════════════════════════════════════════════════════════════
# RESUMEN GENERAL (antes de entrar carpeta por carpeta)
# ══════════════════════════════════════════════════════════════════════════════
def resumen_general():
    _titulo("Resumen general", nivel=1)
    if not BASE_OUT.exists():
        print(f"⚠ La ruta no existe todavía: {BASE_OUT}")
        print("  (¿ya corriste BPN_Peru_Q1_Extremo_V3_6.py al menos una vez?)")
        return

    filas = []
    total_archivos = 0
    total_bytes = 0
    tot_img = tot_csv = tot_xlsx = tot_json = tot_pdf = tot_pkl = 0

    for nombre in CARPETAS_PIPELINE:
        carpeta = BASE_OUT / nombre
        if not carpeta.exists():
            continue
        archivos = [p for p in carpeta.rglob("*") if p.is_file()]
        n = len(archivos)
        size = sum(p.stat().st_size for p in archivos)
        n_img  = sum(1 for p in archivos if p.suffix.lower() in IMG_EXT)
        n_csv  = sum(1 for p in archivos if p.suffix.lower() in CSV_EXT)
        n_xlsx = sum(1 for p in archivos if p.suffix.lower() in EXCEL_EXT)
        n_json = sum(1 for p in archivos if p.suffix.lower() in JSON_EXT)
        n_pdf  = sum(1 for p in archivos if p.suffix.lower() in PDF_EXT)
        n_pkl  = sum(1 for p in archivos if p.suffix.lower() in MODEL_EXT)

        total_archivos += n
        total_bytes += size
        tot_img += n_img; tot_csv += n_csv; tot_xlsx += n_xlsx
        tot_json += n_json; tot_pdf += n_pdf; tot_pkl += n_pkl

        filas.append({
            "Carpeta": nombre, "N_archivos": n,
            "Imágenes": n_img, "CSV": n_csv, "Excel": n_xlsx,
            "JSON": n_json, "PDF": n_pdf, "Modelos": n_pkl,
            "Tamaño": _formato_bytes(size),
        })

    display(pd.DataFrame(filas))
    print(f"\nTOTAL: {total_archivos:,} archivos  |  {_formato_bytes(total_bytes)}")
    print(f"  📷 Imágenes: {tot_img:,}   📊 CSV: {tot_csv:,}   📈 Excel: {tot_xlsx:,}   "
          f"📋 JSON: {tot_json:,}   📄 PDF: {tot_pdf:,}   📦 Modelos: {tot_pkl:,}")

    # Estado del resume (si existe) — solo lectura, no lo modifica
    resume_path = BASE_OUT / "checkpoints" / "resume_state.pkl"
    if resume_path.exists():
        try:
            import joblib
            estado = joblib.load(resume_path)
            print(f"\nEstado de checkpoints (resume_state.pkl):")
            for k, v in estado.items():
                print(f"  {k}: {v}")
        except Exception as e:
            print(f"\n(No se pudo leer resume_state.pkl: {e})")

    resumen_mejor_modelo()


# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN PRINCIPAL — recorre carpeta por carpeta y muestra todo
# ══════════════════════════════════════════════════════════════════════════════
def mostrar_resultados():
    resumen_general()

    carpetas = CARPETAS_A_MOSTRAR if CARPETAS_A_MOSTRAR is not None else CARPETAS_PIPELINE

    for nombre in carpetas:
        carpeta = BASE_OUT / nombre
        if not carpeta.exists():
            continue
        archivos = [p for p in carpeta.iterdir() if p.is_file()]
        if not archivos:
            continue

        _titulo(f"📁 {nombre}", nivel=2)
        print(f"  Ruta: {carpeta}")
        print(f"  {len(archivos)} archivo(s) total en esta carpeta.\n")

        if MOSTRAR_IMAGENES:
            mostrar_imagenes(carpeta)
        if MOSTRAR_PDFS:
            mostrar_pdfs(carpeta)
        if MOSTRAR_CSV:
            mostrar_csv(carpeta)
        if MOSTRAR_EXCEL:
            mostrar_excel(carpeta)
        if MOSTRAR_JSON:
            mostrar_json(carpeta)
        if LISTAR_MODELOS:
            listar_modelos(carpeta)

        print("\n" + "─" * 90 + "\n")

    print("✅ Fin del recorrido. No se recalculó ni modificó nada — solo lectura de Drive.")


# ══════════════════════════════════════════════════════════════════════════════
# EJECUCIÓN
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    mostrar_resultados()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Resumen general

⚠ La ruta no existe todavía: /content/drive/MyDrive/dataset2026varios/Archivos_RN_2026
  (¿ya corriste BPN_Peru_Q1_Extremo_V3_6.py al menos una vez?)
✅ Fin del recorrido. No se recalculó ni modificó nada — solo lectura de Drive.
